# Harm in AI-Driven Societies: analysis notebook

Exploratory notebook for the Chirper.ai toxicity-adoption audit.

**This notebook requires the raw corpus, which cannot be redistributed.**
To reproduce the published results from the released derived tables, use the
scripts in `analysis/` instead (see `README.md`). This notebook is provided for
transparency about how the derived tables were produced.


# Exposure/Adoption setting: 
## 1. If a chirper comments a post, that counts as an EXPOSURE
## 2. If a chirpers provide a toxic comment to a post, that counts as an ADOPTION 

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import os
raw_dir = "/Users/ericacoppolillo/Desktop/PhD/WORKS/ChirperAI/BACKUP_DATA/processed/"
root_dir = "/Users/ericacoppolillo/Desktop/PhD/WORKS/ChirperAI/"
figs_dir = os.path.join("/Users/ericacoppolillo/Desktop/PhD/WORKS/ChirperAI/FIG")

In [ ]:
import seaborn as sns

In [ ]:
import json
with open(os.path.join(root_dir, "processed/toxic_chirper_description_gemini_classification.json")) as f:
    gemini_prompt_classification = json.load(f)

In [ ]:
with open(os.path.join(root_dir, "processed/remaining_chirper_description_gemini_classification.json")) as f:
    remained_prompt_classification = json.load(f)

In [ ]:
all_prompt_classification = remained_prompt_classification | gemini_prompt_classification

In [ ]:
prompted_to_toxicity = list(all_prompt_classification.values())

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
valid_indices = []
valid_results = []
yes_motivations = []

for i, p in enumerate(prompted_to_toxicity):
    if p == "no" or " no " in p or " no" in p or "no " in p or "no." in p or "no;" in p or "**no**" in p or "no," in p or "no - " in p:
        valid_indices.append(i)
        valid_results.append(0)
    elif p == "yes" or " yes " in p or " yes" in p or "yes " in p or "yes." in p or "yes;" or "**yes**" in p or "yes," in p or "yes - " in p:
        valid_indices.append(i)
        valid_results.append(1)
        yes_motivations.append((i, p.replace("yes, ", "").strip()))

In [ ]:
np_valid_results = np.array(valid_results)
len(np_valid_results[np_valid_results == 1])/len(np_valid_results)

In [ ]:
all_prompt_keys = list(all_prompt_classification.keys())

reasons = ["hate", "suicidal", "conspiracy", "sex", "violence"]

yes_reasons = []
other_reasons = {}

agents_with_yes_reasons = {}

for agent_index, m in yes_motivations:
    yes_reason = None
    for r in reasons:
        if m == "nsfw":
            m = "sexual"
        
        if "gore" in m or "sadistic" in m or "violent" in m:
            m = "violence"
        if r in m:
            yes_reason = r
    if yes_reason is None:
        yes_reasons.append((all_prompt_keys[agent_index], "other"))
        other_reasons[all_prompt_keys[agent_index]] = m
    else:
        if yes_reason == "suicidal":
            yes_reason = "suicide"
        yes_reasons.append((all_prompt_keys[agent_index], yes_reason))

reas = [x[1] for x in yes_reasons]

In [ ]:
df_yes_reasons = pd.DataFrame(yes_reasons, columns=["agent", "reason"])

In [ ]:
counts = pd.Series(reas).value_counts()

In [ ]:
norm = counts/sum(counts)

In [ ]:
sns.set_style("whitegrid")
fig, ax = plt.subplots()
bar_container = ax.bar(norm.index, norm.values, width=0.5)
plt.xticks(rotation=45, ha="right")

# Add labels to the bars
# 'fmt' parameter formats the display string. '{:,.0f}' formats as an integer with commas.
ax.bar_label(bar_container, fmt='{:,.2f}', fontsize=14)

plt.ylabel("Proportion")
plt.grid(linestyle='--')
plt.tight_layout()

plt.savefig(os.path.join(figs_dir, "yes_reasons.pdf"), dpi=200, bbox_inches="tight")

plt.show()

In [ ]:
remaining_agent_description_path = os.path.join(root_dir, 'processed', 'remaining_chirper_description_dict.json')
with open(remaining_agent_description_path, 'r') as f:
    remaining_agent_description = json.load(f)

In [ ]:
sample_for_gemini = df_yes_reasons.groupby("reason").sample(n=30, random_state=42)

In [ ]:
from operator import itemgetter

agents_positive_sample_for_gemini = sample_for_gemini["agent"]

In [ ]:
not_prompted = [k for k, v in all_prompt_classification.items() if "no" in v]
np.random.seed(42)
agents_negative_sample_for_gemini = np.random.choice(not_prompted, size=30, replace=False) 

In [ ]:
agents_sample_for_gemini = list(agents_positive_sample_for_gemini) + list(agents_positive_sample_for_gemini)

In [ ]:
negative_sample_for_gemini = pd.DataFrame(zip(agents_negative_sample_for_gemini, ["no"]*len(agents_negative_sample_for_gemini)), columns=sample_for_gemini.columns)

In [ ]:
total_sample_for_gemini = pd.concat([sample_for_gemini, negative_sample_for_gemini], ignore_index=True)


In [ ]:
chirpers_df = pd.read_csv(os.path.join(root_dir, "chirpers_202405081904.csv"))

In [ ]:
agent_system_prompt_dict = dict(zip(chirpers_df["id"], chirpers_df["description"]))

In [ ]:
total_sample_for_gemini["system_prompt"] = list(itemgetter(*total_sample_for_gemini["agent"].values)(agent_system_prompt_dict))

In [ ]:
total_sample_for_gemini = total_sample_for_gemini.sample(frac=1).reset_index(drop=True) # shuffle


In [ ]:
total_sample_for_gemini["system_prompt"].to_csv("total_sample_for_gemini.csv")

In [ ]:
annot_total_sample_for_gemini = pd.read_csv("annot_total_sample_for_gemini.csv")

In [ ]:
total_sample_for_gemini["annot_reason"] = annot_total_sample_for_gemini["category"]

In [ ]:
valid_total_sample_for_gemini = total_sample_for_gemini[total_sample_for_gemini["annot_reason"] != "invalid"]

In [ ]:
def check_merge(x):
    options = x["annot_reason"].split("/")
    reason = x["reason"]
    for o in options:
        if o == reason:
            return reason
    return options[0]

valid_total_sample_for_gemini["polish_annot_reason"] = valid_total_sample_for_gemini.apply(check_merge, axis=1)

In [ ]:
gemini_y_true, gemini_y_pred = valid_total_sample_for_gemini["reason"], valid_total_sample_for_gemini["polish_annot_reason"] 

In [ ]:
labels = list(np.unique(gemini_y_true)) + list(np.unique(gemini_y_pred))
labels = list(set(labels))

In [ ]:
labels_class_dict = dict(zip(labels, range(len(labels))))

In [ ]:
valid_total_sample_for_gemini["reason_class"] = valid_total_sample_for_gemini["reason"].map(labels_class_dict)
valid_total_sample_for_gemini["annot_class"] = valid_total_sample_for_gemini["polish_annot_reason"].map(labels_class_dict)

In [ ]:
gemini_y_true_class, gemini_y_pred_class = valid_total_sample_for_gemini["reason_class"].values, valid_total_sample_for_gemini["annot_class"].values

In [ ]:
classes = range(len(labels))

In [ ]:
from sklearn.metrics import f1_score, classification_report

In [ ]:
#print("AUC", roc_auc_score(gemini_y_true_class, gemini_y_pred_class, labels=classes, multi_class="ovo"))
print(recall_score(gemini_y_true_class, gemini_y_pred_class, labels=classes, average="macro"))
print(precision_score(gemini_y_true_class, gemini_y_pred_class, labels=classes, average="macro"))
print(f1_score(gemini_y_true_class, gemini_y_pred_class, labels=classes, average="macro"))

In [ ]:
print(classification_report(gemini_y_true_class, gemini_y_pred_class))

In [ ]:
print(f1_score(gemini_y_true_class, gemini_y_pred_class, labels=[0, 2, 3, 5, 6], average="macro"))

In [ ]:
grouped_yes_reason = df_yes_reasons.groupby("reason")

for reason, group in grouped_yes_reason:
    print(reason)
    x = group.iloc[:10]["agent"].values
    print(itemgetter(*x)(remaining_agent_description))
    if reason == "other":
        print(itemgetter(*x)(other_reasons))
    print("***")

In [ ]:
indices_agents_not_prompted_to_be_toxic = np.argwhere(np.array(valid_results) == 0).flatten()
indices_agents_prompted_to_be_toxic = np.argwhere(np.array(valid_results) == 1).flatten()

In [ ]:
agents_not_prompted_to_be_toxic = list(np.array(list(all_prompt_classification.keys()))[indices_agents_not_prompted_to_be_toxic])
agents_prompted_to_be_toxic = list(np.array(list(all_prompt_classification.keys()))[indices_agents_prompted_to_be_toxic])

In [ ]:
agents_prompted_to_be_toxic_non_sex = [x for x in agents_prompted_to_be_toxic if "sex" not in all_prompt_classification[x]]

In [ ]:
content_toxicity_df = pd.read_csv(os.path.join(raw_dir, "english_content_with_toxicity.csv"))

In [ ]:
positive_sample_for_detoxify = content_toxicity_df[content_toxicity_df["toxicity"] >= 0.5].sample(n=100, random_state=42)
negative_sample_for_detoxify = content_toxicity_df[content_toxicity_df["toxicity"] < 0.5].sample(n=100, random_state=42)

In [ ]:
sample_for_detoxify = pd.concat([positive_sample_for_detoxify, negative_sample_for_detoxify])
sample_for_detoxify = sample_for_detoxify.sample(frac=1).reset_index(drop=True) # shuffle


In [ ]:
sample_for_detoxify = sample_for_detoxify[["content", "toxicity"]]

In [ ]:
sample_for_detoxify["content"].to_csv("sample_for_detoxify.csv")

In [ ]:
sample_for_detoxify["is_toxic"] = sample_for_detoxify.apply(lambda x: 1 if x["toxicity"] >= 0.5 else 0, axis=1)
sample_for_detoxify["is_highly_toxic"] = sample_for_detoxify.apply(lambda x: 1 if x["toxicity"] >= 0.75 else 0, axis=1)

In [ ]:
annotated_sample_for_detoxify = pd.read_csv("annotated_sample_for_detoxify.csv")

In [ ]:
y_highly_true = annotated_sample_for_detoxify["is_highly_toxic"].values.tolist()
y_highly_pred = sample_for_detoxify["is_highly_toxic"].values.tolist()

y_true = annotated_sample_for_detoxify["is_toxic"].values.tolist()
y_pred = sample_for_detoxify["is_toxic"].values.tolist()

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score

In [ ]:
np.unique(y_true), np.unique(y_pred)

In [ ]:
print("AUC", roc_auc_score(y_highly_true, y_highly_pred))
print("Recall", recall_score(y_highly_true, y_highly_pred))
print("Precision", precision_score(y_highly_true, y_highly_pred))

In [ ]:
print("AUC", roc_auc_score(y_true, y_pred))
print("Recall", recall_score(y_true, y_pred))
print("Precision", precision_score(y_true, y_pred))

In [ ]:
sample_for_detoxify["is_toxic_annot"] = y_true
sample_for_detoxify["is_highly_toxic_annot"] = y_highly_true

In [ ]:
sample_for_detoxify[sample_for_detoxify["is_toxic"] != sample_for_detoxify["is_toxic_annot"]]

In [ ]:
content_toxicity_df["chirper"].nunique()/75000

In [ ]:
unique_agents = content_toxicity_df["chirper"].unique()

len(set(unique_agents).intersection(set(all_prompt_classification.keys())))/len(unique_agents)

In [ ]:
content_toxicity_df.iloc[0]["content"]

In [ ]:
# EXPOSED -> content that chirpers commented
comments_df = content_toxicity_df[content_toxicity_df["parent"].notna()]


In [ ]:
comments_df = comments_df[["id", "createdAt", "parent", "chirper", "toxicity"]]

In [ ]:
comments_df.rename(columns={"parent": "chirp"}, inplace=True)

In [ ]:
comments_df.dropna(inplace=True)

In [ ]:
comments_df["createdAt"] = pd.to_datetime(comments_df['createdAt'], format="%Y-%m-%d %H:%M:%S.%f %z")
comments_df['createdAt'] = comments_df['createdAt'].astype(int) // 1_000_000

In [ ]:
comments_to_post = pd.merge(comments_df, content_toxicity_df, left_on="chirp", right_on="id", how="inner")


In [ ]:
comments_to_post["createdAt_y"] = pd.to_datetime(comments_to_post['createdAt_y'], format="%Y-%m-%d %H:%M:%S.%f %z")
comments_to_post['createdAt_y'] = comments_to_post['createdAt_y'].astype(int) // 1_000_000

In [ ]:
exposure_df = comments_to_post[["createdAt_x", "chirp", "chirper_x", "toxicity_x", "toxicity_y"]]

In [ ]:
exposure_df.rename(columns={"createdAt_x": "exposure_date", "chirper_x": "chirper", "toxicity_x":"comment_toxicity", "toxicity_y": "post_toxicity"}, inplace=True)

In [ ]:
chirper_average_exposures = exposure_df.groupby("chirper")["chirp"].nunique()

In [ ]:
ev = comments_to_post[[
    "id_x", "createdAt_x", "chirp", "chirper_x", "chirper_y",
    "toxicity_x", "toxicity_y",
]].copy()

In [ ]:
toxicity_threshold = 0.5

In [ ]:

ev.columns = ["comment_id", "t_ms", "post_id", "agent", "source",
              "resp_tox_score", "stim_tox_score"]

ev = ev.dropna(subset=["t_ms", "agent", "source"])
ev["t_ms"] = ev["t_ms"].astype("int64")
ev["t"] = pd.to_datetime(ev["t_ms"], unit="ms", utc=True)
ev["stim_toxic"] = (ev["stim_tox_score"] >= toxicity_threshold).astype(int)
ev["resp_toxic"] = (ev["resp_tox_score"] >= toxicity_threshold).astype(int)
ev = ev.sort_values(["agent", "t_ms"]).reset_index(drop=True)

In [ ]:
high_toxicity_threshold = 0.75

In [ ]:
high_t_c = content_toxicity_df[content_toxicity_df["toxicity"] >= high_toxicity_threshold]

In [ ]:
high_t_c["chirper"].nunique()/content_toxicity_df["chirper"].nunique()

In [ ]:
ev[ev.stim_toxic == 1]["agent"].nunique()/content_toxicity_df["chirper"].nunique()

In [ ]:
# Numbers needed for the reproducibility table. Run against the notebook state.
import numpy as np, pandas as pd

print("=== CORPUS ===")
print("records collected (raw):        10,420,000   [from paper]")
print("chirpers (raw):                 ~75,000      [from paper]")
print("human creators (raw):           >30,000      [from paper]")
# Split the 8M figure into posts vs comments -- the reviewer's explicit question
print("posts after language filter:   ", len(content_toxicity_df) - len(comments_df))
print("comments after language filter:", len(comments_df))
print("agents after language filter:  ", content_toxicity_df["chirper"].nunique())

print("\n=== INTERACTION SET ===")
print("stimulus-response pairs:       ", len(comments_to_post))
print("  after dropping missing:      ", len(ev))
print("agents in interaction set:     ", ev.agent.nunique())
print("distinct posts as stimuli:     ", ev.post_id.nunique())
print("distinct source agents:        ", ev.source.nunique())

print("\n=== PREVALENCE ===")
print(f"toxic stimuli:   {len(ev[ev.stim_toxic == 1])} {100*len(ev[ev.stim_toxic == 1])/len(ev):.2f}%")
print(f"toxic responses: {len(ev[ev.resp_toxic == 1])} {100*len(ev[ev.resp_toxic == 1])/len(ev):.2f}%")

print(f"high toxic content: {len(high_t_c)} {100*len(high_t_c)/len(content_toxicity_df):.2f}%")
print(f"high toxic stimuli:   {len(ev[ev.stim_tox_score >= high_toxicity_threshold])} {100*len(ev[ev.stim_tox_score >= high_toxicity_threshold])/len(ev):.2f}%")
print(f"high toxic responses: {len(ev[ev.resp_tox_score >= high_toxicity_threshold])} {100*len(ev[ev.resp_tox_score >= high_toxicity_threshold])/len(ev):.2f}%")
for lo, hi in [(0,.1),(.1,.35),(.35,.5),(.5,.65),(.65,.9),(.9,1.01)]:
    m = (ev.resp_tox_score >= lo) & (ev.resp_tox_score < hi)
    print(f"  score [{lo:.2f},{hi:.2f}): {100*m.mean():6.3f}%")

print("\n=== ACTIVITY DISTRIBUTION ===")
n = ev.groupby("agent").size()
print(n.describe(percentiles=[.25,.5,.75,.9,.99]).round(1).to_string())
print(f"Gini: {(2*np.sum((np.arange(1,len(n)+1))*np.sort(n))/(len(n)*n.sum()) - (len(n)+1)/len(n)):.3f}")

print("\n=== MISSING RECORDS ===")
for c in ["createdAt_x", "toxicity_x", "toxicity_y", "chirper_x", "chirper_y", "chirp"]:
    if c in comments_to_post.columns:
        print(f"  {c:14s} missing: {comments_to_post[c].isna().sum():,}")

print("\n=== ANALYTIC SUBSETS ===")
print("RQ1  informative strata:            9,700")
print("RQ2  event-level model agents:     15,531   events: 3,925,541")
print("RQ2  survival: agent-events:    2,079,710   adoptions: 15,619")
print("RQ3  agents with both metrics:      9,716")
print("prediction: at-risk agents:        20,259")

print("\n=== TEMPORAL STABILITY (for the platform-change question) ===")
m = ev.set_index("t").resample("ME").agg(
    events=("resp_toxic","size"), tox_resp=("resp_toxic","mean"),
    tox_stim=("stim_toxic","mean"), agents=("agent","nunique"))
print(m.round(4).to_string())

In [ ]:
last = ev.groupby(ev.t.dt.floor("D")).size().tail(30)
print(last.to_string())

In [ ]:
mar = ev[(ev.t >= "2024-03-01") & (ev.t < "2024-04-01")]
top = mar.groupby("agent")["resp_toxic"].agg(["size", "sum"]).nlargest(20, "sum")
print(top, "\ntop-20 share of March toxic responses:",
      round(100 * top["sum"].sum() / mar.resp_toxic.sum(), 1), "%")

In [ ]:
9328 / 45072

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
sns.set_style('whitegrid')

In [ ]:
plt.figure()
plt.boxplot(chirper_average_exposures, showfliers=False, showmeans=True)
plt.xticks([])
plt.ylabel("Average Exposures", fontsize=16)
plt.yticks(fontsize=14)
plt.grid(linestyle='--')
plt.show()

In [ ]:
toxicity_values = content_toxicity_df["toxicity"].values

# toxicity_threshold = np.percentile(toxicity_values, 90)
toxicity_threshold = 0.5

In [ ]:
toxic_df = content_toxicity_df[content_toxicity_df["toxicity"] >= toxicity_threshold]

In [ ]:
toxic_content = toxic_df["id"].unique()

In [ ]:
toxic_df["createdAt"] = pd.to_datetime(toxic_df['createdAt'], format="%Y-%m-%d %H:%M:%S.%f %z")
toxic_df['createdAt'] = toxic_df['createdAt'].astype(int) // 1_000_000

In [ ]:
toxic_df[["id", "toxicity"]].head()

In [ ]:
from collections import defaultdict
from tqdm import tqdm

# Anecdotal Evidence

In [ ]:
toxic_to_toxic_examples = comments_to_post[(comments_to_post["toxicity_x"] >= toxicity_threshold) & (comments_to_post["toxicity_y"] >= toxicity_threshold)]
toxic_to_non_toxic_examples = comments_to_post[(comments_to_post["toxicity_x"] >= toxicity_threshold) & (comments_to_post["toxicity_y"] < toxicity_threshold)]
non_toxic_to_toxic_examples = comments_to_post[(comments_to_post["toxicity_x"] < toxicity_threshold) & (comments_to_post["toxicity_y"] >= toxicity_threshold)]
non_toxic_to_non_toxic_examples = comments_to_post[(comments_to_post["toxicity_x"] < toxicity_threshold) & (comments_to_post["toxicity_y"] < toxicity_threshold)]

In [ ]:
print(toxic_to_toxic_examples.iloc[0][["id_x", "id_y"]]) # comment, post
print(toxic_to_non_toxic_examples.iloc[0][["id_x", "id_y"]])
print(non_toxic_to_toxic_examples.iloc[0][["id_x", "id_y"]])
print(non_toxic_to_non_toxic_examples.iloc[0][["id_x", "id_y"]])

In [ ]:
content_toxicity_df[content_toxicity_df["id"].isin(toxic_to_toxic_examples.iloc[3][["id_x", "id_y"]].values)]["content"].values

In [ ]:
content_toxicity_df[content_toxicity_df["id"].isin(toxic_to_non_toxic_examples.iloc[7][["id_x", "id_y"]].values)]["content"].values

In [ ]:
content_toxicity_df[content_toxicity_df["id"].isin(non_toxic_to_toxic_examples.iloc[1][["id_x", "id_y"]].values)]["content"].values

In [ ]:
content_toxicity_df[content_toxicity_df["id"].isin(non_toxic_to_non_toxic_examples.iloc[1][["id_x", "id_y"]].values)]["content"].values

In [ ]:
grouped_exposure_df = exposure_df.groupby('chirper')

In [ ]:
for chirper, group in grouped_exposure_df:
    if len(group) < 2:
        continue
    print(chirper)
    print(group)
    break

In [ ]:
exposures_per_follower = dict(grouped_exposure_df["chirp"].nunique())

In [ ]:
adoptions_per_follower = dict(grouped_exposure_df["chirp"].size())

In [ ]:
toxic_exposure_df = exposure_df[exposure_df["post_toxicity"] >= toxicity_threshold]
non_toxic_exposure_df = exposure_df[exposure_df["post_toxicity"] < toxicity_threshold]

In [ ]:
toxic_exposures_per_follower = dict(toxic_exposure_df.groupby("chirper")["chirp"].nunique())
non_toxic_exposures_per_follower = dict(non_toxic_exposure_df.groupby("chirper")["chirp"].nunique())

In [ ]:
toxic_adoption_df = exposure_df[exposure_df["comment_toxicity"] >= toxicity_threshold]
non_toxic_adoption_df = exposure_df[exposure_df["comment_toxicity"] < toxicity_threshold]

In [ ]:
toxic_adoptions_per_follower = dict(toxic_adoption_df.groupby("chirper").size())
non_toxic_adoptions_per_follower = dict(non_toxic_adoption_df.groupby("chirper").size())

In [ ]:
toxic_adoptions = list(toxic_adoptions_per_follower.values())
non_toxic_adoptions = list(non_toxic_adoptions_per_follower.values())

In [ ]:
followers = exposure_df["chirper"].unique()

In [ ]:
toxic_adoptions_prompted = [v for k, v in toxic_adoptions_per_follower.items() if k in agents_prompted_to_be_toxic]
toxic_adoptions_non_prompted = [v for k, v in toxic_adoptions_per_follower.items() if k in agents_not_prompted_to_be_toxic]

In [ ]:
# toxic adoptions distribution of agents prompted/non-prompted to be toxic
plt.figure()

toxic_adoptions_prompted_non_prompted = pd.DataFrame({'n_toxic_adoptions': toxic_adoptions_prompted + toxic_adoptions_non_prompted,
                                                      'type': ['prompted'] * len(toxic_adoptions_prompted) + ['non-prompted'] * len(toxic_adoptions_non_prompted)})

In [ ]:
agents_with_toxic_adoptions = [k for k, v in toxic_adoptions_per_follower.items() if v > 0]


In [ ]:
plt.figure()

sns.boxplot(data=toxic_adoptions_prompted_non_prompted, y="n_toxic_adoptions", x="type", showfliers=False)
plt.show()

In [ ]:
from scipy import stats
import seaborn as sns
from operator import itemgetter

In [ ]:
import matplotlib as mtl
sns.reset_orig()

mtl.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.sans-serif": "Computer Modern Serif",
    'font.size' : 20,
})
import matplotlib.pyplot as plt


In [ ]:
exposures_for_toxic_adoptions = list(itemgetter(*toxic_adoptions_per_follower)(exposures_per_follower))

In [ ]:
exposures_for_toxic_adoptions_not_prompted = list(itemgetter(*toxic_adoptions_per_follower)(exposures_per_follower))

In [ ]:
exposures_for_non_toxic_adoptions = list(itemgetter(*non_toxic_adoptions_per_follower)(exposures_per_follower))

In [ ]:
from collections import Counter, defaultdict

In [ ]:
users_by_exposure = defaultdict(list)
for user, n_e in exposures_per_follower.items():
    users_by_exposure[n_e].append(user)

# 2. Compute P(A | n_e = N)
prob_toxic_adoption_given_exposure = {}
prob_non_toxic_adoption_given_exposure = {}

minimum_toxic_adoption =0# np.percentile(toxic_adoptions, 25) # 0 (?)
minimum_non_toxic_adoption =0# np.percentile(non_toxic_adoptions, 25) # 0 (?)

for N, users in users_by_exposure.items():
    # users with exposure N who adopted
    toxic_adopted = sum(1 if u in toxic_adoptions_per_follower and toxic_adoptions_per_follower[u] > minimum_toxic_adoption else 0 for u in users)
    non_toxic_adopted = sum(1 if u in non_toxic_adoptions_per_follower and non_toxic_adoptions_per_follower[u] > minimum_non_toxic_adoption else 0 for u in users)
    total = len(users)

    prob_toxic_adoption_given_exposure[N] = toxic_adopted / total if total > 0 else 0.0
    prob_non_toxic_adoption_given_exposure[N] = non_toxic_adopted / total if total > 0 else 0.0

In [ ]:
toxic_N_vals = np.array(list(prob_toxic_adoption_given_exposure.keys()))
toxic_Pr_vals = np.array(list(prob_toxic_adoption_given_exposure.values()))

In [ ]:
non_toxic_N_vals = np.array(list(prob_non_toxic_adoption_given_exposure.keys()))
non_toxic_Pr_vals = np.array(list(prob_non_toxic_adoption_given_exposure.values()))

In [ ]:
from scipy.optimize import curve_fit

def log_func(x, a, b):
    return a * np.log(x) + b

# Avoid log(0) → only fit points where N > 0
toxic_fit_mask = toxic_N_vals > 0

toxic_popt, _ = curve_fit(log_func, toxic_N_vals[toxic_fit_mask], toxic_Pr_vals[toxic_fit_mask])
toxic_a, toxic_b = toxic_popt

# smooth curve for plotting
toxic_x_fit = np.linspace(1, toxic_N_vals.max(), 500)
toxic_y_fit = log_func(toxic_x_fit, toxic_a, toxic_b)

In [ ]:
# Avoid log(0) → only fit points where N > 0
non_toxic_fit_mask = non_toxic_N_vals > 0

non_toxic_popt, _ = curve_fit(log_func, non_toxic_N_vals[non_toxic_fit_mask], non_toxic_Pr_vals[non_toxic_fit_mask])
non_toxic_a, non_toxic_b = non_toxic_popt

# smooth curve for plotting
non_toxic_x_fit = np.linspace(1, non_toxic_N_vals.max(), 500)
non_toxic_y_fit = log_func(non_toxic_x_fit, non_toxic_a, non_toxic_b)

In [ ]:
plt.figure(figsize=(5,4))
sns.set_style("whitegrid")

# scatter
plt.scatter(toxic_N_vals, toxic_Pr_vals, alpha=0.7, label="Data")

# regression curve
plt.plot(toxic_x_fit, toxic_y_fit, color=sns.color_palette()[3], linewidth=2.5, label="Log fit")

plt.scatter(non_toxic_N_vals, non_toxic_Pr_vals, alpha=0.7, label="Data")
plt.plot(non_toxic_x_fit, non_toxic_y_fit, color=sns.color_palette()[3], linewidth=2.5, label="Log fit")

# labels
plt.xlabel("Number of Toxic Exposures $n_e$", fontsize=12)
plt.ylabel("Probability of Toxic Adoption $Pr(A|n_e)$", fontsize=12)

#plt.xlim(0, N_vals.max() + 10)
#plt.ylim(0, max(Pr_vals) * 1.1)
plt.grid(linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
threshold = 150
toxic_restricted_N_vals = toxic_N_vals[toxic_N_vals <= threshold]
toxic_restricted_Pr_vals = toxic_Pr_vals[toxic_N_vals <= threshold]

In [ ]:
non_toxic_restricted_N_vals = non_toxic_N_vals[non_toxic_N_vals <= threshold]
non_toxic_restricted_Pr_vals = non_toxic_Pr_vals[non_toxic_N_vals <= threshold]

In [ ]:
sns.color_palette()

In [ ]:

# Avoid log(0) → only fit points where N > 0
toxic_fit_mask = toxic_restricted_N_vals > 0

toxic_popt, _ = curve_fit(log_func, toxic_restricted_N_vals[toxic_fit_mask], toxic_restricted_Pr_vals[toxic_fit_mask])
toxic_a, toxic_b = toxic_popt

# smooth curve for plotting
toxic_x_fit = np.linspace(1, toxic_restricted_N_vals.max(), 50)
toxic_y_fit = np.clip(log_func(toxic_x_fit, toxic_a, toxic_b), 0, 1)

In [ ]:
# Avoid log(0) → only fit points where N > 0
non_toxic_fit_mask = non_toxic_restricted_N_vals > 0

non_toxic_popt, _ = curve_fit(log_func, non_toxic_restricted_N_vals[non_toxic_fit_mask], non_toxic_restricted_Pr_vals[non_toxic_fit_mask])
non_toxic_a, non_toxic_b = non_toxic_popt

# smooth curve for plotting
non_toxic_x_fit = np.linspace(1, non_toxic_restricted_N_vals.max(), 50)
non_toxic_y_fit = log_func(non_toxic_x_fit, non_toxic_a, non_toxic_b)

In [ ]:
import pymannkendall as mk

In [ ]:
plt.figure(figsize=(5,4))
sns.set_style("whitegrid")
# scatter
plt.scatter(toxic_restricted_N_vals, toxic_restricted_Pr_vals, alpha=0.7, label="Data", color="gray")

result = mk.original_test(toxic_restricted_Pr_vals)
print(result)
# regression curve
plt.plot(toxic_x_fit, toxic_y_fit, linewidth=2.5, label="Log fit", color="black") #color=sns.color_palette()[3], 


#plt.scatter(non_toxic_restricted_N_vals, non_toxic_restricted_Pr_vals, alpha=0.7, label="Data")
#plt.plot(non_toxic_x_fit, non_toxic_y_fit, linewidth=2.5, label="Log fit") # color=sns.color_palette()[3], 


# labels
plt.xlabel("$n_{\mathcal{S}}$", fontsize=16)
plt.ylabel(r"$\textrm{P}(\mathcal{R}_T|n_{\mathcal{S}})$", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)


#plt.xlim(0, N_vals.max() + 10)
#plt.ylim(0, max(Pr_vals) * 1.1)

#plt.xlim(0, 51)
plt.ylim(bottom=-0.1, top=1.1)

plt.grid(linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(figs_dir, 'probability_of_toxic_adoption.pdf'), dpi=200, bbox_inches="tight")
plt.show()

# Grouping by post (toxic/non-toxic)

In [ ]:
from tqdm import tqdm
import json

In [ ]:
CREATE = False

paths = ["count_chirper_exposition_non_toxic_content_dict.json",
     "count_chirper_exposition_toxic_content_dict.json",
     "count_chirper_toxic_adoption_non_toxic_content_dict.json",
     "count_chirper_toxic_adoption_toxic_content_dict.json",
     "count_chirper_non_toxic_adoption_non_toxic_content_dict.json",
     "count_chirper_non_toxic_adoption_toxic_content_dict.json"]

if CREATE:
    count_chirper_exposition_non_toxic_content = defaultdict()
    count_chirper_exposition_toxic_content = defaultdict()
    
    count_chirper_toxic_adoption_non_toxic_content = defaultdict()
    count_chirper_toxic_adoption_toxic_content = defaultdict()
    
    count_chirper_non_toxic_adoption_non_toxic_content = defaultdict()
    count_chirper_non_toxic_adoption_toxic_content = defaultdict()
    
    for chirper, group in tqdm(grouped_exposure_df, total=len(grouped_exposure_df)):
    
        chirper_exposed_non_toxic_content = group[group["post_toxicity"] < toxicity_threshold]["chirp"].unique()
        chirper_exposed_toxic_content = group[group["post_toxicity"] >= toxicity_threshold]["chirp"].unique()
    
        # Toxic adoption
        chirper_toxic_adoption_df = toxic_adoption_df[toxic_adoption_df["chirper"] == chirper]
        chirper_toxic_adopted_non_toxic_content = chirper_toxic_adoption_df[chirper_toxic_adoption_df["chirp"].isin(chirper_exposed_non_toxic_content)]["chirp"].values.tolist()
        chirper_toxic_adopted_toxic_content = chirper_toxic_adoption_df[chirper_toxic_adoption_df["chirp"].isin(chirper_exposed_toxic_content)]["chirp"].values.tolist()
    
        # Non-toxic adoption
        chirper_non_toxic_adoption_df = non_toxic_adoption_df[non_toxic_adoption_df["chirper"] == chirper]
        chirper_non_toxic_adopted_non_toxic_content = chirper_non_toxic_adoption_df[chirper_non_toxic_adoption_df["chirp"].isin(chirper_exposed_non_toxic_content)]["chirp"].values.tolist()
        chirper_non_toxic_adopted_toxic_content = chirper_non_toxic_adoption_df[chirper_non_toxic_adoption_df["chirp"].isin(chirper_exposed_toxic_content)]["chirp"].values.tolist()
    
        count_chirper_exposition_non_toxic_content[chirper] = len(chirper_exposed_non_toxic_content)
        count_chirper_exposition_toxic_content[chirper] = len(chirper_exposed_toxic_content)
    
        count_chirper_toxic_adoption_non_toxic_content[chirper] = len(chirper_toxic_adopted_non_toxic_content)
        count_chirper_toxic_adoption_toxic_content[chirper] = len(chirper_toxic_adopted_toxic_content)
    
        count_chirper_non_toxic_adoption_non_toxic_content[chirper] = len(chirper_non_toxic_adopted_non_toxic_content)
        count_chirper_non_toxic_adoption_toxic_content[chirper] = len(chirper_non_toxic_adopted_toxic_content)

    counters = [count_chirper_exposition_non_toxic_content, 
                count_chirper_exposition_toxic_content,
                count_chirper_toxic_adoption_non_toxic_content,
                count_chirper_toxic_adoption_toxic_content,
                count_chirper_non_toxic_adoption_non_toxic_content,
                count_chirper_non_toxic_adoption_toxic_content]
    
    for i in range(len(paths)):
        with open(os.path.join(root_dir, "processed", paths[i]), "w") as f:
            json.dump(counters[i], f)
else:
    with open(os.path.join(root_dir, "processed", paths[0]), "r") as f:
        count_chirper_exposition_non_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", paths[1]), "r") as f:
        count_chirper_exposition_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", paths[2]), "r") as f:
        count_chirper_toxic_adoption_non_toxic_content = json.load(f)
        
    with open(os.path.join(root_dir, "processed", paths[3]), "r") as f:
        count_chirper_toxic_adoption_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", paths[4]), "r") as f:
        count_chirper_non_toxic_adoption_non_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", paths[5]), "r") as f:
        count_chirper_non_toxic_adoption_toxic_content = json.load(f)

In [ ]:
CREATE = False
# COUNT TOXIC RESPONSES ONLY ONCE FOR EACH STIMULUS
paths = ["unique_count_chirper_exposition_non_toxic_content_dict.json",
     "unique_count_chirper_exposition_toxic_content_dict.json",
     "unique_count_chirper_toxic_adoption_non_toxic_content_dict.json",
     "unique_count_chirper_toxic_adoption_toxic_content_dict.json",
     "unique_count_chirper_non_toxic_adoption_non_toxic_content_dict.json",
     "unique_count_chirper_non_toxic_adoption_toxic_content_dict.json"]

if CREATE:
    count_chirper_exposition_non_toxic_content = defaultdict()
    count_chirper_exposition_toxic_content = defaultdict()
    
    unique_count_chirper_toxic_adoption_non_toxic_content = defaultdict()
    unique_count_chirper_toxic_adoption_toxic_content = defaultdict()
    
    unique_count_chirper_non_toxic_adoption_non_toxic_content = defaultdict()
    unique_count_chirper_non_toxic_adoption_toxic_content = defaultdict()
    
    for chirper, group in tqdm(grouped_exposure_df, total=len(grouped_exposure_df)):
    
        chirper_exposed_non_toxic_content = group[group["post_toxicity"] < toxicity_threshold]["chirp"].unique()
        chirper_exposed_toxic_content = group[group["post_toxicity"] >= toxicity_threshold]["chirp"].unique()
    
        # Toxic adoption
        chirper_toxic_adoption_df = toxic_adoption_df[toxic_adoption_df["chirper"] == chirper]
        chirper_toxic_adopted_non_toxic_content = chirper_toxic_adoption_df[chirper_toxic_adoption_df["chirp"].isin(chirper_exposed_non_toxic_content)]["chirp"].unique().tolist()
        chirper_toxic_adopted_toxic_content = chirper_toxic_adoption_df[chirper_toxic_adoption_df["chirp"].isin(chirper_exposed_toxic_content)]["chirp"].unique().tolist()
    
        # Non-toxic adoption
        chirper_non_toxic_adoption_df = non_toxic_adoption_df[non_toxic_adoption_df["chirper"] == chirper]
        chirper_non_toxic_adopted_non_toxic_content = chirper_non_toxic_adoption_df[chirper_non_toxic_adoption_df["chirp"].isin(chirper_exposed_non_toxic_content)]["chirp"].unique().tolist()
        chirper_non_toxic_adopted_toxic_content = chirper_non_toxic_adoption_df[chirper_non_toxic_adoption_df["chirp"].isin(chirper_exposed_toxic_content)]["chirp"].unique().tolist()
    
        count_chirper_exposition_non_toxic_content[chirper] = len(chirper_exposed_non_toxic_content)
        count_chirper_exposition_toxic_content[chirper] = len(chirper_exposed_toxic_content)
    
        unique_count_chirper_toxic_adoption_non_toxic_content[chirper] = len(chirper_toxic_adopted_non_toxic_content)
        unique_count_chirper_toxic_adoption_toxic_content[chirper] = len(chirper_toxic_adopted_toxic_content)
    
        unique_count_chirper_non_toxic_adoption_non_toxic_content[chirper] = len(chirper_non_toxic_adopted_non_toxic_content)
        unique_count_chirper_non_toxic_adoption_toxic_content[chirper] = len(chirper_non_toxic_adopted_toxic_content)

    counters = [count_chirper_exposition_non_toxic_content, 
                count_chirper_exposition_toxic_content,
                unique_count_chirper_toxic_adoption_non_toxic_content,
                unique_count_chirper_toxic_adoption_toxic_content,
                unique_count_chirper_non_toxic_adoption_non_toxic_content,
                unique_count_chirper_non_toxic_adoption_toxic_content]
    
    for i in range(len(paths)):
        with open(os.path.join(root_dir, "processed", paths[i]), "w") as f:
            json.dump(counters[i], f)
else:
    with open(os.path.join(root_dir, "processed", paths[0]), "r") as f:
        count_chirper_exposition_non_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", paths[1]), "r") as f:
        count_chirper_exposition_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", paths[2]), "r") as f:
        unique_count_chirper_toxic_adoption_non_toxic_content = json.load(f)
        
    with open(os.path.join(root_dir, "processed", paths[3]), "r") as f:
        unique_count_chirper_toxic_adoption_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", paths[4]), "r") as f:
        unique_count_chirper_non_toxic_adoption_non_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", paths[5]), "r") as f:
        unique_count_chirper_non_toxic_adoption_toxic_content = json.load(f)

In [ ]:
scatterplot_data = []

for chirper in count_chirper_exposition_non_toxic_content:

    label = "Non-Prompted"
    if chirper in agents_prompted_to_be_toxic:
        label = "Prompted"

    non_toxic_exp = count_chirper_exposition_non_toxic_content[chirper]
    toxic_exp = count_chirper_exposition_toxic_content[chirper]

    toxic_adop_non_toxic_exp = count_chirper_toxic_adoption_non_toxic_content[chirper]
    toxic_adop_toxic_exp = count_chirper_toxic_adoption_toxic_content[chirper]

    non_toxic_adop_non_toxic_exp = count_chirper_non_toxic_adoption_non_toxic_content[chirper]
    non_toxic_adop_toxic_exp = count_chirper_non_toxic_adoption_toxic_content[chirper]

    scatterplot_data.append((toxic_adop_non_toxic_exp, non_toxic_exp, "Toxic_Non-Toxic", label))
    scatterplot_data.append((toxic_adop_toxic_exp, toxic_exp, "Toxic_Toxic", label))

    scatterplot_data.append((non_toxic_adop_non_toxic_exp, non_toxic_exp, "Non-Toxic_Non-Toxic", label))
    scatterplot_data.append((non_toxic_adop_toxic_exp, toxic_exp, "Non-Toxic_Toxic", label))

In [ ]:
table = np.array([
    [sum(list(count_chirper_toxic_adoption_toxic_content.values())),  sum(list(count_chirper_non_toxic_adoption_toxic_content.values()))],   # Toxic Adoption:     Toxic Exposure | Non-Toxic Exposure
    [sum(list(count_chirper_toxic_adoption_non_toxic_content.values())), sum(list(count_chirper_non_toxic_adoption_non_toxic_content.values()))], # Non-Toxic Adoption: Toxic Exposure | Non-Toxic Exposure
])

In [ ]:
norm_table = table/np.sum(table, axis=0)
norm_table

In [ ]:
odd = (table[0,0]/table[1,0])/(table[0,1]/table[1,1])

In [ ]:
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from scipy.stats import chi2_contingency
from scipy.stats import fisher_exact

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(table)

print(chi2, p_value, dof, expected)

# Phi Coefficient
N = table.sum()
phi = np.sqrt(chi2 / N)

print(phi)

# Odds Ratio (with Fisher's exact for p-value)
a, b, c, d = table[0,0], table[0,1], table[1,0], table[1,1]
odds_ratio = (a * d) / (b * c)
_, p_fisher = fisher_exact(table)

In [ ]:
print(odds_ratio, p_fisher)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
scatterplot_df = pd.DataFrame(scatterplot_data, columns=["Adoptions", "Exposures", "Type", "Agent"])


In [ ]:
palette = sns.color_palette()[:4]

In [ ]:
palette_dict = {'Toxic_Non-Toxic':palette[1], 'Toxic_Toxic':palette[-1],
                        "Non-Toxic_Non-Toxic": palette[2], "Non-Toxic_Toxic": palette[0]}
palette_keys = list(palette_dict.keys())

In [ ]:
fig, ax = plt.subplots(2, 2)
count = 0

import matplotlib.patches as mpatches
import matplotlib.lines as mlines

new_labels = [
    r'$\mathcal{R}_{T} \rightarrow \mathcal{S}_N$',
    r'$\mathcal{R}_T \rightarrow \mathcal{S}_T$',
    r'$\mathcal{R}_N \rightarrow \mathcal{S}_N$',
    r'$\mathcal{R}_N \rightarrow \mathcal{S}_T$'
]
handles = []
for i in range(2):
    for j in range(2):

        print(palette_keys[count])
        x = scatterplot_df[scatterplot_df["Type"] == palette_keys[count]]["Exposures"]
        y = scatterplot_df[scatterplot_df["Type"] == palette_keys[count]]["Adoptions"]

        res, p_value = stats.spearmanr(x, y)

        sns.regplot(data=scatterplot_df[scatterplot_df["Type"] == palette_keys[count]], x='Exposures', y='Adoptions',
                color=palette_dict[palette_keys[count]], ax=ax[i, j], scatter_kws={'alpha': 0.3})

        handles.append(mlines.Line2D([], [], color=palette_dict[palette_keys[count]],
                                      label=new_labels[count], marker='o', linestyle='None'))

        print(p_value)
        
        ax[i,j].text(0.05, 0.80,
         rf"$\rho = {res:.3f}$",
         transform=ax[i,j].transAxes,
         fontsize=18)

        #plt.plot(x, m*x + b, color='red', label='Regression Line')
        #ax[i,j].legend().set_visible(False)
        ax[i,j].grid(linestyle='--')

        ax[i,j].set_xlabel("\#Stimuli")
        ax[i,j].set_ylabel("\#Responses")

        if j == 1:
            ax[i,j].set_ylabel('')
        if i == 0:
            ax[i,j].set_xlabel('')

        
        count += 1
plt.tight_layout()

plt.savefig(os.path.join(figs_dir, "exposures_adoptions_type_correlation.pdf"), dpi=300, bbox_inches="tight")

fig_legend = plt.figure(figsize=(4, 1))   # adjust size as needed


legend = fig_legend.legend(handles, new_labels, ncol=4, frameon=True)

#ax[0,0].legend().set_visible(False)

# Remove axes
fig_legend.gca().set_axis_off()

# Save legend
fig_legend.savefig(os.path.join(figs_dir, "scatter_legend.pdf"), dpi=300, bbox_inches="tight")
plt.subplots_adjust(hspace=0.5)
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 2)
count = 0

import matplotlib.patches as mpatches
import matplotlib.lines as mlines

new_labels = [
    r'$\mathcal{R}_{T} \rightarrow \mathcal{S}_N$',
    r'$\mathcal{R}_T \rightarrow \mathcal{S}_T$',
    r'$\mathcal{R}_N \rightarrow \mathcal{S}_N$',
    r'$\mathcal{R}_N \rightarrow \mathcal{S}_T$'
]
handles = []
for i in range(2):
    for j in range(2):

        print(palette_keys[count])
        prompted_scatterplot_df = scatterplot_df[(scatterplot_df["Type"] == palette_keys[count]) & (scatterplot_df["Agent"] == "Prompted")]
        
        prompted_x = prompted_scatterplot_df["Exposures"]
        prompted_y = prompted_scatterplot_df["Adoptions"]

        non_prompted_scatterplot_df = scatterplot_df[(scatterplot_df["Type"] == palette_keys[count]) & (scatterplot_df["Agent"] == "Non-Prompted")]
        
        non_prompted_x = non_prompted_scatterplot_df["Exposures"]
        non_prompted_y = non_prompted_scatterplot_df["Adoptions"]

        # Subsample scatter points
        
        #ax[i,j].scatter(prompted_x, prompted_y,
        #            facecolors="white",
        #            edgecolors=palette_dict[palette_keys[count]],
        #            linewidths=1.5, s=60, zorder=10, marker="^")
        
        # Regression line only (no scatter)
        sns.regplot(x=prompted_x, y=prompted_y, scatter=False,
                    line_kws={"linewidth": 2.5, "zorder": 5, "marker": "o", 
                              "color": palette_dict[palette_keys[count]]}, ax=ax[i, j])

        #handles.append(mlines.Line2D([], [], color=palette_dict[palette_keys[count]],
        #                              label=new_labels[count]))

        handles.append(mpatches.Patch(color=palette_dict[palette_keys[count]], label=new_labels[count], alpha=0.5))

        line = ax[i,j].get_lines()[-1]

        x_fit = line.get_xdata()
        y_fit = line.get_ydata()

        line.remove()  # remove regplot's line
        
        # Replot with markevery
        ax[i,j].plot(x_fit, y_fit,
                     color=palette_dict[palette_keys[count]], linewidth=2.5,
                     marker="^", markevery=10,
                     markersize=9,
                     markerfacecolor="white",
                     markeredgecolor=palette_dict[palette_keys[count]],
                     markeredgewidth=1.5,
                     zorder=10)

        sns.regplot(x=non_prompted_x, y=non_prompted_y, scatter=False,
                    line_kws={"linewidth": 2.5, "zorder": 5, "marker": "o", 
                              "color": palette_dict[palette_keys[count]]}, ax=ax[i, j])

        line = ax[i,j].get_lines()[-1]

        x_fit = line.get_xdata()
        y_fit = line.get_ydata()
        line.remove()  # remove regplot's line
        
        # Replot with markevery
        ax[i,j].plot(x_fit, y_fit,
                     color=palette_dict[palette_keys[count]], linewidth=2.5,
                     marker="H", markevery=10,
                     markersize=9,
                     markerfacecolor="white",
                     markeredgecolor=palette_dict[palette_keys[count]],
                     markeredgewidth=1.5,
                     zorder=10)

    
        ax[i,j].set_xlabel("\#Stimuli")
        ax[i,j].set_ylabel("\#Responses")

        prompted_res, prompted_p_value = stats.spearmanr(prompted_x, prompted_y)
        non_prompted_res, non_prompted_p_value = stats.spearmanr(non_prompted_x, non_prompted_y)

         # ── Regression slope ───────────────────────────────────────────────
        prompted_slope, prompted_intercept, prompted_r, prompted_p, prompted_se = stats.linregress(prompted_x, prompted_y)
        #print(f"Slope: {prompted_slope:.4f} (for every 1 stimulus, {prompted_slope:.4f} more responses)")

        # ── Mean difference ────────────────────────────────────────────────
        prompted_mean_diff = np.mean(prompted_y) - np.mean(prompted_x)
        #print(f"Mean difference (responses - stimuli): {prompted_mean_diff:.4f}")

        # ── Cohen's d ──────────────────────────────────────────────────────
        prompted_pooled_std = np.sqrt((np.std(prompted_x, ddof=1)**2 + np.std(prompted_y, ddof=1)**2) / 2)
        prompted_cohens_d = (np.mean(prompted_y) - np.mean(prompted_x)) / prompted_pooled_std
        #print(f"Cohen's d: {prompted_cohens_d:.4f}")


        non_prompted_slope, non_prompted_intercept, non_prompted_r, non_prompted_p, non_prompted_se = stats.linregress(non_prompted_x, non_prompted_y)
        #print(f"Slope: {non_prompted_slope:.4f} (for every 1 stimulus, {non_prompted_slope:.4f} more responses)")

        # ── Mean difference ────────────────────────────────────────────────
        non_prompted_mean_diff = np.mean(non_prompted_y) - np.mean(non_prompted_x)
        #print(f"Mean difference (responses - stimuli): {non_prompted_mean_diff:.4f}")

        # ── Cohen's d ──────────────────────────────────────────────────────
        non_prompted_pooled_std = np.sqrt((np.std(non_prompted_x, ddof=1)**2 + np.std(non_prompted_y, ddof=1)**2) / 2)
        non_prompted_cohens_d = (np.mean(non_prompted_y) - np.mean(non_prompted_x)) / non_prompted_pooled_std
        #print(f"Cohen's d: {non_prompted_cohens_d:.4f}")


        print("R2 prompted", prompted_r**2) # r comes from linregress
        print("R2 non-prompted", non_prompted_r**2)
        #ax[i,j].text(0.05, 0.85,
        # rf"$m^P = {prompted_slope:.2f}$",
        # transform=ax[i,j].transAxes,
        # fontsize=16)

        #ax[i,j].text(0.05, 0.7,
        # rf"$m^N = {non_prompted_slope:.2f}$",
        # transform=ax[i,j].transAxes,
        # fontsize=16)

        #plt.plot(x, m*x + b, color='red', label='Regression Line')
        #ax[i,j].legend().set_visible(False)
        ax[i,j].grid(linestyle='--')

        ax[i,j].set_xlabel("\#Stimuli")
        ax[i,j].set_ylabel("\#Responses")

        if j == 1:
            ax[i,j].set_ylabel('')
        if i == 0:
            ax[i,j].set_xlabel('')

        
        count += 1
plt.tight_layout()

plt.savefig(os.path.join(figs_dir, "prompted_non_prompted_exposures_adoptions_type_correlation.pdf"), dpi=300, bbox_inches="tight")

#fig_legend = plt.figure(figsize=(4, 1))   # adjust size as needed


#legend = fig_legend.legend(handles, new_labels, ncol=4, frameon=True)

#ax[0,0].legend().set_visible(False)

# Remove axes
#fig_legend.gca().set_axis_off()




In [ ]:

fig_legend, ax_legend = plt.subplots(figsize=(4, 2))

triangle = mlines.Line2D([], [], color="gray", marker="^", linestyle="None",
                         markersize=9, label="Prompted",
                         markerfacecolor="white", markeredgewidth=1.5)
hexagon  = mlines.Line2D([], [], color="gray", marker="h", linestyle="None",
                         markersize=9, label="Non-prompted",
                         markerfacecolor="white", markeredgewidth=1.5)

toxic_stimulus    = mpatches.Patch(color=sns.color_palette()[1], label="Toxic")
nontoxic_stimulus = mpatches.Patch(color=sns.color_palette()[0], label="Non-toxic")

ax_legend.legend(
    handles=[
        mlines.Line2D([], [], linestyle="none", label=" Interaction "),
        *handles,
        mlines.Line2D([], [], linestyle="none", label=" Agent "),
        triangle, hexagon,
        mlines.Line2D([], [], linestyle="none", label=" "),
        mlines.Line2D([], [], linestyle="none", label=" "),
    ],
    fontsize=17,
    handletextpad=0.5,
    framealpha=0.9,
    title="",
    title_fontsize=10,
    #loc="center",   
    ncol=2# center it in the blank figure
)

ax_legend.axis("off")   # hide the empty axes

plt.tight_layout()

# Save legend
fig_legend.savefig(os.path.join(figs_dir, "prompted_non_prompted_scatter_legend.pdf"), dpi=300, bbox_inches="tight")
plt.subplots_adjust(hspace=0.5)
plt.show()

In [ ]:
users_ors = []
users_toxic_rrs = []
users_non_toxic_rrs = []


chirpers = exposure_df["chirper"].unique()
for chirper in chirpers:

    # a quanti stimuli non tossici viene risposto in maniera tossica ecc?

    non_toxic_exp = count_chirper_exposition_non_toxic_content[chirper]
    toxic_exp = count_chirper_exposition_toxic_content[chirper]

    toxic_adop_non_toxic_exp = unique_count_chirper_toxic_adoption_non_toxic_content[chirper]
    toxic_adop_toxic_exp = unique_count_chirper_toxic_adoption_toxic_content[chirper]

    non_toxic_adop_non_toxic_exp = unique_count_chirper_non_toxic_adoption_non_toxic_content[chirper]
    non_toxic_adop_toxic_exp = unique_count_chirper_non_toxic_adoption_toxic_content[chirper]

    toxic_adop = toxic_adop_non_toxic_exp + toxic_adop_toxic_exp

    # toxic response - toxic stimulus, toxic response - non toxic stimulus, 
    # non toxic response - toxic stimulus, non toxic response - non toxic stimululs
    users_ors.append((toxic_adop_toxic_exp, toxic_adop_non_toxic_exp, non_toxic_adop_toxic_exp, non_toxic_adop_non_toxic_exp))

    # cioè voglio vedere che c'è più rischio di toxic adopt su toxic exp E più rischio di non toxic adopt su non toxic adopt
    users_toxic_rrs.append((toxic_adop_toxic_exp, toxic_exp, toxic_adop_non_toxic_exp, non_toxic_exp))
    users_non_toxic_rrs.append((non_toxic_adop_non_toxic_exp, non_toxic_exp, non_toxic_adop_toxic_exp, toxic_exp))

In [ ]:
prompted_users_ors = []

prompted_users_not_sex_ors = []

prompted_users_toxic_rrs = []
prompted_users_non_toxic_rrs = []

non_prompted_users_ors = []
non_prompted_users_toxic_rrs = []
non_prompted_users_non_toxic_rrs = []


chirpers = exposure_df["chirper"].unique()
for chirper in chirpers:

    # a quanti stimuli non tossici viene risposto in maniera tossica ecc?

    non_toxic_exp = count_chirper_exposition_non_toxic_content[chirper]
    toxic_exp = count_chirper_exposition_toxic_content[chirper]

    toxic_adop_non_toxic_exp = unique_count_chirper_toxic_adoption_non_toxic_content[chirper]
    toxic_adop_toxic_exp = unique_count_chirper_toxic_adoption_toxic_content[chirper]

    non_toxic_adop_non_toxic_exp = unique_count_chirper_non_toxic_adoption_non_toxic_content[chirper]
    non_toxic_adop_toxic_exp = unique_count_chirper_non_toxic_adoption_toxic_content[chirper]

    toxic_adop = toxic_adop_non_toxic_exp + toxic_adop_toxic_exp

    # toxic response - toxic stimulus, toxic response - non toxic stimulus, 
    # non toxic response - toxic stimulus, non toxic response - non toxic stimululs

    if chirper in agents_prompted_to_be_toxic:
        prompted_users_ors.append((toxic_adop_toxic_exp, toxic_adop_non_toxic_exp, non_toxic_adop_toxic_exp, non_toxic_adop_non_toxic_exp))
        if chirper in agents_prompted_to_be_toxic_non_sex:
            prompted_users_not_sex_ors.append((toxic_adop_toxic_exp, toxic_adop_non_toxic_exp, non_toxic_adop_toxic_exp, non_toxic_adop_non_toxic_exp))

    else:
        non_prompted_users_ors.append((toxic_adop_toxic_exp, toxic_adop_non_toxic_exp, non_toxic_adop_toxic_exp, non_toxic_adop_non_toxic_exp))

    # cioè voglio vedere che c'è più rischio di toxic adopt su toxic exp E più rischio di non toxic adopt su non toxic adopt
    #users_toxic_rrs.append((toxic_adop_toxic_exp, toxic_exp, toxic_adop_non_toxic_exp, non_toxic_exp))
    #users_non_toxic_rrs.append((non_toxic_adop_non_toxic_exp, non_toxic_exp, non_toxic_adop_toxic_exp, toxic_exp))

In [ ]:
import statsmodels.stats.contingency_tables as ct
import numpy as np

tables = [np.array([[a, b], [c, d]]) for a, b, c, d in users_ors]
result = ct.StratifiedTable(tables)
print(result.oddsratio_pooled)
print(">1 means that the probability of a type of adoption (toxic/non-toxic) is higher with the same type of exposure (toxic/non-toxic)")# MH OR
print(result.oddsratio_pooled_confint())  # 95% CI

In [ ]:
import numpy as np, statsmodels.stats.contingency_tables as ct

tabs = [np.asarray(t, float) for t in tables]

def informative(t):
    return t.sum() > 1 and (t.sum(1) > 0).all() and (t.sum(0) > 0).all()

# diagnostics — worth reporting in the paper
n = np.array([t.sum() for t in tabs])
print("n == 0:", (n == 0).sum(), " n == 1:", (n == 1).sum())
print("zero-margin:", sum(not ((t.sum(1) > 0).all() and (t.sum(0) > 0).all()) for t in tabs))

kept = [t for t in tabs if informative(t)]
print(f"informative strata: {len(kept)} / {len(tabs)}")

res = ct.StratifiedTable(kept)
print(res.summary())
print(res.test_null_odds())
print(res.test_equal_odds())

# sanity check: the OR should be essentially unchanged
print(result.oddsratio_pooled, res.oddsratio_pooled)

In [ ]:
import numpy as np

def rates(or_tuples):
    """or_tuples: list of (a, b, c, d) = (RT-ST, RT-SN, RN-ST, RN-SN)"""
    A = np.asarray(or_tuples, dtype=float)

    # --- marginal (volume-weighted) ---
    a, b, c, d = A.sum(axis=0)
    p_st, p_sn = a / (a + c), b / (b + d)

    # --- median per-agent (equal-weighted), agents with the denominator > 0 ---
    col_T, col_N = A[:, 0] + A[:, 2], A[:, 1] + A[:, 3]
    m_st = np.median(A[col_T > 0, 0] / col_T[col_T > 0])
    m_sn = np.median(A[col_N > 0, 1] / col_N[col_N > 0])

    return p_st, p_sn, m_st, m_sn

for name, tuples in [("all", prompted_users_ors + non_prompted_users_ors),
                     ("prompted", prompted_users_ors),
                     ("non-prompted", non_prompted_users_ors)]:
    p_st, p_sn, m_st, m_sn = rates(tuples)
    print(f"{name:>13}  marginal: P(RT|ST)={100*p_st:5.2f}%  P(RT|SN)={100*p_sn:5.2f}%  "
          f"RR={p_st/p_sn:.2f}   |   median per-agent: {100*m_st:5.2f}% vs {100*m_sn:5.2f}%")

In [ ]:
conf_int = result.oddsratio_pooled_confint()
print((conf_int[1] - conf_int[0])/3.92)

In [ ]:
import numpy as np
rows = []
for chirper in chirpers:
    n_T = count_chirper_exposition_toxic_content[chirper]
    n_N = count_chirper_exposition_non_toxic_content[chirper]
    r_T = (unique_count_chirper_toxic_adoption_toxic_content[chirper]
           + unique_count_chirper_non_toxic_adoption_toxic_content[chirper])
    r_N = (unique_count_chirper_toxic_adoption_non_toxic_content[chirper]
           + unique_count_chirper_non_toxic_adoption_non_toxic_content[chirper])
    rows.append((n_T, n_N, r_T, r_N))

R = np.asarray(rows, float)
print("NaN present:", np.isnan(R).any(axis=0))          # per column
print("resp_T > exp_T:", (R[:,2] > R[:,0]).sum(), "agents")
print("resp_N > exp_N:", (R[:,3] > R[:,1]).sum(), "agents")
print("exp_T == 0:", (R[:,0] == 0).sum(), " exp_N == 0:", (R[:,1] == 0).sum())

In [ ]:
import statsmodels.stats.contingency_tables as ct
import numpy as np

prompted_tables = [np.array([[a, b], [c, d]]) for a, b, c, d in prompted_users_ors]

prompted_tables_not_sex = [np.array([[a, b], [c, d]]) for a, b, c, d in prompted_users_not_sex_ors]


prompted_result = ct.StratifiedTable(prompted_tables)
print(prompted_result.oddsratio_pooled)
print(">1 means that the probability of a type of adoption (toxic/non-toxic) is higher with the same type of exposure (toxic/non-toxic)")# MH OR
print(prompted_result.oddsratio_pooled_confint())  # 95% CI

In [ ]:

def informative(t):
    return t.sum() > 1 and (t.sum(1) > 0).all() and (t.sum(0) > 0).all()

In [ ]:
all_tabs = [np.asarray(t, float) for t in prompted_tables + non_prompted_tables]

kept = [t for t in all_tabs if informative(t)]
print(f"informative strata: {len(kept)} / {len(tabs)}")

res = ct.StratifiedTable(kept)
print(res.summary())
print(res.test_null_odds())
print(res.test_equal_odds())

print(res.oddsratio_pooled)

In [ ]:
import numpy as np, statsmodels.stats.contingency_tables as ct

tabs = [np.asarray(t, float) for t in prompted_tables]


# diagnostics — worth reporting in the paper
n = np.array([t.sum() for t in tabs])
print("n == 0:", (n == 0).sum(), " n == 1:", (n == 1).sum())
print("zero-margin:", sum(not ((t.sum(1) > 0).all() and (t.sum(0) > 0).all()) for t in tabs))

kept = [t for t in tabs if informative(t)]
print(f"informative strata: {len(kept)} / {len(tabs)}")

res = ct.StratifiedTable(kept)
print(res.summary())
print(res.test_null_odds())
print(res.test_equal_odds())

# sanity check: the OR should be essentially unchanged
print(prompted_result.oddsratio_pooled, res.oddsratio_pooled)

In [ ]:
from scipy.stats import wilcoxon
deltas = [a/(a+c) - b/(b+d) for a, b, c, d in prompted_users_ors if (a+c) > 0 and (b+d) > 0]
print(np.median(deltas), wilcoxon(deltas))

In [ ]:
import statsmodels.stats.contingency_tables as ct
import numpy as np

non_prompted_tables = [np.array([[a, b], [c, d]]) for a, b, c, d in non_prompted_users_ors]
non_prompted_result = ct.StratifiedTable(non_prompted_tables)
print(non_prompted_result.oddsratio_pooled)
print(">1 means that the probability of a type of adoption (toxic/non-toxic) is higher with the same type of exposure (toxic/non-toxic)")# MH OR
print(non_prompted_result.oddsratio_pooled_confint())  # 95% CI

In [ ]:
import numpy as np, statsmodels.stats.contingency_tables as ct

tabs = [np.asarray(t, float) for t in non_prompted_tables]

def informative(t):
    return t.sum() > 1 and (t.sum(1) > 0).all() and (t.sum(0) > 0).all()

# diagnostics — worth reporting in the paper
n = np.array([t.sum() for t in tabs])
print("n == 0:", (n == 0).sum(), " n == 1:", (n == 1).sum())
print("zero-margin:", sum(not ((t.sum(1) > 0).all() and (t.sum(0) > 0).all()) for t in tabs))

kept = [t for t in tabs if informative(t)]
print(f"informative strata: {len(kept)} / {len(tabs)}")

res = ct.StratifiedTable(kept)
print(res.summary())
print(res.test_null_odds())
print(res.test_equal_odds())

# sanity check: the OR should be essentially unchanged
print(non_prompted_result.oddsratio_pooled, res.oddsratio_pooled)

In [ ]:
both = (R[:,0] > 0) & (R[:,1] > 0)
print(both.sum(), "exposed to both;", 9700, "informative")

In [ ]:
prompted_mask, non_prompted_mask = [], []
for chirper in chirpers:
    n_T = count_chirper_exposition_toxic_content[chirper]
    n_N = count_chirper_exposition_non_toxic_content[chirper]
    ok = (n_T > 0) and (n_N > 0)          # exposed to both stimulus types
    if chirper in agents_prompted_to_be_toxic:
        prompted_mask.append(ok)
    else:
        non_prompted_mask.append(ok)

assert len(prompted_mask) == len(prompted_tables)
assert len(non_prompted_mask) == len(non_prompted_tables)

def haldane(tables, mask):
    tc = [np.asarray(t, float) + 0.5 for t, k in zip(tables, mask) if k]
    r = ct.StratifiedTable(tc)
    return len(tc), round(r.oddsratio_pooled, 3), r.oddsratio_pooled_confint()

print("all         ", haldane(prompted_tables + non_prompted_tables,
                              prompted_mask + non_prompted_mask))
print("prompted    ", haldane(prompted_tables, prompted_mask))
print("non-prompted", haldane(non_prompted_tables, non_prompted_mask))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

means = np.array([result.oddsratio_pooled, prompted_result.oddsratio_pooled, non_prompted_result.oddsratio_pooled])

colors = ["gray"] + sns.color_palette()[2:4][::-1]

x_labels = ["All", "Prompted", "Non-Prompted"]

x_pos = np.arange(len(x_labels))

lower_errors = np.array([means[0]-result.oddsratio_pooled_confint()[0], means[1]-prompted_result.oddsratio_pooled_confint()[0], means[2]-non_prompted_result.oddsratio_pooled_confint()[0]])
upper_errors = np.array([result.oddsratio_pooled_confint()[1]-means[0], prompted_result.oddsratio_pooled_confint()[1]-means[1], non_prompted_result.oddsratio_pooled_confint()[1]-means[2]])

for i, c in enumerate(colors):
    if i == 0:
        marker = "o"
    elif i == 1:
        marker = "^"
    else:
        marker = "H"
    ax.plot(x_labels[i], means[i], 
            linewidth=0, marker=marker, color=c, 
            markersize=12, markerfacecolor="white", markeredgewidth=2)#, label=x_labels[i])
    
 
    # Vertical error bars on top for precision
    ax.errorbar(
        x_pos[i], means[i],
        yerr=[[lower_errors[i]], [upper_errors[i]]],
        fmt="none",
        ecolor=c,
        elinewidth=1.5,
        capsize=5,
        capthick=1.5,
    )

#ax.fill_between(
##    x_pos,
#    means - lower_errors,
#    means + upper_errors,
    #color=ci_color,
#    alpha=0.40,
#    label=f"{int(CONFIDENCE*100)}% CI",
#)

CONFIDENCE = 0.95
# Styling
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, fontsize=15)
ax.set_ylabel("Odds Ratio", fontsize=17)
#ax.set_title(f"Group Means with {int(CONFIDENCE*100)}% Confidence Intervals", fontsize=14, fontweight="bold")
#ax.legend(framealpha=0.9, fontsize=10, loc="best")
#ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.grid(axis="x", linestyle="--", alpha=0)

#ax.set_ylim(2, 3.6)
#ax.set_yticks(np.arange(0, 3.6, 1.5))
plt.tight_layout()

fig.savefig(os.path.join(figs_dir, "odds_ratio.pdf"), dpi=200, bbox_inches="tight")

plt.show()

In [ ]:
"""
Temporal structure of toxic exposure — analyses addressing Reviewer [N]'s comment
on "cumulative toxic exposure (repeated over time)".

Assumes the following are already in scope from ChirperAI-LocalPlots.ipynb:
    comments_to_post          (cell 90)
    toxicity_threshold        (cell 102, = 0.5)
    agents_prompted_to_be_toxic

Note on units: exposure_df.exposure_date is comments_df.createdAt converted to
MILLISECONDS since epoch (cell 88: .astype(int) // 1_000_000).
"""

toxicity_threshold = 0.5
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 0. Build an event-level frame. One row = one comment = one exposure/response
#    pair. We rebuild from comments_to_post rather than exposure_df because we
#    need chirper_y, the AUTHOR of the parent post (the source of the stimulus),
#    which exposure_df drops.
# ---------------------------------------------------------------------------



print(f"events: {len(ev):,}   agents: {ev.agent.nunique():,}")
print(f"observation window: {ev.t.min()}  ->  {ev.t.max()}"
      f"  ({(ev.t.max() - ev.t.min()).days} days)")

# ---------------------------------------------------------------------------
# 1. TEMPORAL SPREAD of toxic exposure, per agent.
#    Directly answers "is this repeated over time, or concentrated?"
# ---------------------------------------------------------------------------
tox = ev[ev.stim_toxic == 1]

def burstiness(times_ms):
    """Goh & Barabasi burstiness of inter-event times: -1 regular, 0 Poisson, +1 bursty."""
    if len(times_ms) < 3:
        return np.nan
    d = np.diff(np.sort(times_ms))
    m, s = d.mean(), d.std()
    return np.nan if (m + s) == 0 else (s - m) / (s + m)

spread = tox.groupby("agent").agg(
    n_toxic_exp=("t_ms", "size"),
    n_distinct_days=("t", lambda s: s.dt.floor("D").nunique()),
    n_distinct_hours=("t", lambda s: s.dt.floor("h").nunique()),
    span_days=("t_ms", lambda s: (s.max() - s.min()) / 86_400_000),
    burst=("t_ms", burstiness),
)

multi = spread[spread.n_toxic_exp >= 2]
print("\n--- temporal spread of toxic exposure ---")
print(f"agents with >=2 toxic exposures: {len(multi):,}")
print(f"  median span:            {multi.span_days.median():.1f} days")
print(f"  median distinct days:   {multi.n_distinct_days.median():.0f}")
print(f"  all exposures same day: {100 * (multi.n_distinct_days == 1).mean():.1f}% of agents")
print(f"  all within one hour:    {100 * (multi.n_distinct_hours == 1).mean():.1f}% of agents")
print(f"  median burstiness B:    {multi.burst.median():.3f}")

# ---------------------------------------------------------------------------
# 2. SOURCE CONCENTRATION.
#    Answers the "one commenter sending many toxic stimuli" version directly.
# ---------------------------------------------------------------------------
def hhi(sources):
    p = sources.value_counts(normalize=True).values
    return float((p ** 2).sum())

conc = tox.groupby("agent").agg(
    n_toxic_exp=("source", "size"),
    n_sources=("source", "nunique"),
    top_share=("source", lambda s: s.value_counts(normalize=True).iloc[0]),
    hhi=("source", hhi),
)
concm = conc[conc.n_toxic_exp >= 2]
print("\n--- source concentration of toxic exposure ---")
print(f"  median distinct sources:      {concm.n_sources.median():.0f}")
print(f"  median share from top source: {100 * concm.top_share.median():.1f}%")
print(f"  single-source agents:         {100 * (concm.n_sources == 1).mean():.1f}%")
print(f"  median HHI:                   {concm.hhi.median():.3f}")


In [ ]:
# ---------------------------------------------------------------------------
# 3. SEQUENTIAL TEST — the analysis the reviewer is actually asking for.
#    For each event, count the agent's toxic exposures STRICTLY BEFORE it, and
#    separately those in the preceding 24h. The first term is long-run
#    accumulation; the second is recent-burst. Distinguishing them is exactly
#    the distinction the reviewer draws.
# ---------------------------------------------------------------------------
# ev is already sorted by (agent, t_ms), so everything below is pure numpy on
# contiguous per-agent blocks. No groupby.apply.
codes = pd.factorize(ev["agent"])[0]           # increasing, since ev is agent-sorted
s = ev["stim_toxic"].to_numpy()
t = ev["t_ms"].to_numpy().astype("int64")
n = len(ev)
 
first = np.flatnonzero(np.r_[True, codes[1:] != codes[:-1]])   # block starts
sizes = np.diff(np.r_[first, n])
first_of = np.repeat(first, sizes)                             # block start per row
 
csum = np.cumsum(s)
base = np.repeat(csum[first] - s[first], sizes)
cum_prior = csum - base - s                    # toxic exposures strictly before row i
ev["cum_prior_toxic"] = cum_prior
ev["event_idx"] = np.arange(n) - first_of
 
# Rolling 24h count. Offsetting each agent's clock by more than the full
# observation span keeps searchsorted inside the agent's own block.
WINDOW_MS = 24 * 3_600_000
OFF = (t.max() - t.min()) + WINDOW_MS + 1
tg = t + codes.astype("int64") * OFF
left = np.maximum(np.searchsorted(tg, tg - WINDOW_MS, side="left"), first_of)
ev["recent_toxic_24h"] = cum_prior - cum_prior[left]
 
ev["log_cum_prior"] = np.log1p(ev.cum_prior_toxic)
ev["log_recent"] = np.log1p(ev.recent_toxic_24h)
 
# --- sanity checks: run these before fitting anything -----------------------
assert (ev.cum_prior_toxic >= 0).all()
assert (ev.recent_toxic_24h >= 0).all()
assert (ev.recent_toxic_24h <= ev.cum_prior_toxic).all()
print(ev[["stim_toxic", "cum_prior_toxic", "recent_toxic_24h",
          "resp_toxic"]].describe().T)
 
# ---------------------------------------------------------------------------
# 3b. Model. Agent identity must be absorbed, so only within-agent variation
#     identifies the coefficients — same logic as the RQ1 stratification.
#     Option A is exact but heavy; Option B is robust and scales.
# ---------------------------------------------------------------------------
vary = ev.groupby("agent")["resp_toxic"].transform("nunique") > 1
sub = ev[vary].copy()
print(f"\n--- sequential model ---\nagents contributing: {sub.agent.nunique():,} "
      f"({len(sub):,} events)")
 
COLS = ["stim_toxic", "log_cum_prior", "log_recent"]
 
# --- Option A: conditional (fixed-effects) logit, on a subsample of agents ---
def fit_clogit(frame, n_agents=3000, seed=0):
    from statsmodels.discrete.conditional_models import ConditionalLogit
    ids = frame.agent.unique()
    if len(ids) > n_agents:
        ids = np.random.default_rng(seed).choice(ids, n_agents, replace=False)
    f = frame[frame.agent.isin(ids)]
    m = ConditionalLogit(f["resp_toxic"].to_numpy(float),
                         f[COLS].to_numpy(float),
                         groups=pd.factorize(f["agent"])[0]).fit(disp=False)
    return pd.DataFrame({"coef": m.params, "OR": np.exp(m.params),
                         "se": m.bse, "p": m.pvalues}, index=COLS)
 
# Repeat over a few seeds to confirm the subsample isn't driving anything:
# for sd in range(3): print(fit_clogit(sub, seed=sd))
 
# --- Option B: correlated random effects (Mundlak), cluster-robust SEs -------
# Adding each covariate's agent-level mean makes the remaining coefficient the
# WITHIN-agent effect, which is the same estimand as Option A, without the
# per-group optimization. Scales to the full dataset.
import statsmodels.api as sm
 
for c in COLS:
    sub[c + "_bar"] = sub.groupby("agent")[c].transform("mean")
 
Xc = sm.add_constant(sub[COLS + [c + "_bar" for c in COLS]].astype(float))
mB = sm.Logit(sub["resp_toxic"].astype(float), Xc).fit(
    disp=False, cov_type="cluster",
    cov_kwds={"groups": pd.factorize(sub["agent"])[0]})
print(mB.summary())
print("\nwithin-agent odds ratios:\n", np.exp(mB.params[COLS]))
 
# ---------------------------------------------------------------------------
# 4. SIMPLE WITHIN-AGENT EARLY-vs-LATE CHECK.
#    Assumption-light, easy to describe, and persuasive on its own.
# ---------------------------------------------------------------------------
from scipy.stats import wilcoxon
 
ev["half"] = (ev.event_idx
              >= ev.groupby("agent")["event_idx"].transform("max") / 2).astype(int)
h = (ev.groupby(["agent", "half"])
       .agg(rate=("resp_toxic", "mean"), n=("resp_toxic", "size"))
       .unstack("half"))
h.columns = ["rate_early", "rate_late", "n_early", "n_late"]
h = h.dropna()
h = h[(h.n_early >= 10) & (h.n_late >= 10)]
d = (h.rate_late - h.rate_early).values
 
print(f"\n--- early vs late (>=10 events per half) ---")
print(f"agents: {len(h):,}   median change: {100 * np.median(d):+.2f} pp   "
      f"increased: {100 * (d > 0).mean():.1f}%")
print(wilcoxon(d))
 
# ---------------------------------------------------------------------------
# 5. Split by arm, for the prompted / non-prompted contrast.
# ---------------------------------------------------------------------------
for name, mask in [("prompted", h.index.isin(agents_prompted_to_be_toxic)),
                   ("non-prompted", ~h.index.isin(agents_prompted_to_be_toxic))]:
    dd = (h.rate_late - h.rate_early).values[mask]
    if len(dd) > 10:
        print(f"{name:>13}: n={len(dd):,}  median {100 * np.median(dd):+.2f} pp  "
              f"increased {100 * (dd > 0).mean():.1f}%")

In [ ]:
tx = ev[ev.stim_toxic == 1].copy()
tx["k"] = tx.groupby("agent").cumcount()
tx["half"] = (tx.k >= tx.groupby("agent")["k"].transform("max") / 2).astype(int)
h2 = (tx.groupby(["agent", "half"])
        .agg(rate=("resp_toxic", "mean"), n=("resp_toxic", "size")).unstack("half"))
h2.columns = ["r_early", "r_late", "n_early", "n_late"]
h2 = h2.dropna()
for k in (3, 5):
    q = h2[(h2.n_early >= k) & (h2.n_late >= k)]
    d = (q.r_late - q.r_early).values
    print(f"min {k}/half: agents={len(q):,}  median {100*np.median(d):+.2f} pp  "
          f"increased {100*(d>0).mean():.1f}%")

In [ ]:
sub["stim_x_cum"] = sub.stim_toxic * sub.log_cum_prior
COLS2 = ["stim_toxic", "log_cum_prior", "log_recent", "stim_x_cum"]
for c in COLS2:
    sub[c + "_bar"] = sub.groupby("agent")[c].transform("mean")

Xc2 = sm.add_constant(sub[COLS2 + [c + "_bar" for c in COLS2]].astype(float))
m2 = sm.Logit(sub["resp_toxic"].astype(float), Xc2).fit(
    disp=False, cov_type="cluster",
    cov_kwds={"groups": pd.factorize(sub["agent"])[0]})
print(m2.summary())
print(np.exp(m2.params[COLS2]))

In [ ]:
sub["log_prior_events"] = np.log1p(sub.event_idx)
sub["stim_x_events"] = sub.stim_toxic * sub.log_prior_events

COLS3 = ["stim_toxic", "log_cum_prior", "log_recent",
         "stim_x_cum", "log_prior_events", "stim_x_events"]
for c in COLS3:
    sub[c + "_bar"] = sub.groupby("agent")[c].transform("mean")

X3 = sm.add_constant(sub[COLS3 + [c + "_bar" for c in COLS3]].astype(float))
m3 = sm.Logit(sub["resp_toxic"].astype(float), X3).fit(
    disp=False, cov_type="cluster",
    cov_kwds={"groups": pd.factorize(sub["agent"])[0]})
print(m3.summary())
print(np.exp(m3.params[COLS3]))

In [ ]:
"""
Addressing Reviewer [N]'s comment on repeated exposure.

Two additions to the event-level analysis already built in temporal_analysis.py:
  (A) null models for the P(adopt | n_e) curve in Figures 5-11
  (B) a within-agent permutation null for the event-level model
plus additional clustering / time controls.

Requires from temporal_analysis.py:  ev, sub, COLS3
"""

import numpy as np, pandas as pd, statsmodels.api as sm
import matplotlib.pyplot as plt

# ===========================================================================
# 0. What metadata do we actually have for the extra random-effect levels?
#    Run this first — the answer determines what is feasible.
# ===========================================================================
print("chirpers_df columns:", chirpers_df.columns.tolist())
# Looking for: creator / owner / user id, model / llm / engine.
# post id and time we already have (ev.post_id, ev.t).

# ===========================================================================
# (A) NULL MODEL for the "at least one toxic response" curve.
#     Reviewer's point: P(>=1 toxic | n) = 1 - (1-p)^n rises and saturates
#     with no change in the underlying propensity. We quantify how much of
#     the observed curve this accounts for.
# ===========================================================================
agg = ev.groupby("agent").agg(n=("resp_toxic", "size"), k=("resp_toxic", "sum"))
p_hat = ev.resp_toxic.mean()
rates = (agg.k / agg.n).values          # per-agent toxicity rates
n_arr = agg.n.values

obs = (agg.assign(adopt=agg.k > 0).groupby("n")["adopt"]
          .agg(["mean", "size"]).rename(columns={"mean": "p_adopt", "size": "agents"}))
ns = obs.index.values

def sim_curve(hetero, R=200, seed=0):
    """Simulate the P(adopt|n) curve under a null with no exposure effect.

    hetero=False : every response toxic with the same probability p_hat
                   (the reviewer's 1-(1-p)^n null, simulated).
    hetero=True  : each agent keeps a toxicity RATE drawn from the observed
                   distribution, but that rate is assigned INDEPENDENTLY of
                   the agent's activity level. Preserves both marginals,
                   destroys only the association. This is the stronger null.
    """
    rng = np.random.default_rng(seed)
    out = np.full((R, len(ns)), np.nan)
    for r in range(R):
        p_i = rng.choice(rates, size=len(n_arr)) if hetero else np.full(len(n_arr), p_hat)
        k = rng.binomial(n_arr, p_i)
        d = pd.DataFrame({"n": n_arr, "adopt": k > 0})
        out[r] = d.groupby("n")["adopt"].mean().reindex(ns).values
    return np.nanpercentile(out, [2.5, 50, 97.5], axis=0)

lo_h, mid_h, hi_h = sim_curve(hetero=False)     # homogeneous p
lo_t, mid_t, hi_t = sim_curve(hetero=True)      # heterogeneous p

w = obs.agents.values                            # weight by number of agents
def wdiff(mid):
    m = ~np.isnan(mid)
    return np.average(obs.p_adopt.values[m] - mid[m], weights=w[m])

print(f"\np_hat = {p_hat:.4f}")
print(f"mean excess over homogeneous null (agent-weighted):   {wdiff(mid_h):+.4f}")
print(f"mean excess over heterogeneous null (agent-weighted): {wdiff(mid_t):+.4f}")
print(f"observed curve above heterogeneous 97.5th pct at "
      f"{100 * np.nanmean(obs.p_adopt.values > hi_t):.1f}% of n values")

# Figure: observed curve against the two null bands. This REPLACES the
# unadorned version of Figures 5-11 -- the null band is the point.
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.fill_between(ns, lo_t, hi_t, alpha=.25, label="null (95%): activity + rate preserved")
ax.plot(ns, mid_h, ls="--", lw=1, label=r"null: $1-(1-p)^n$")
ax.plot(ns, obs.p_adopt.values, lw=1.5, label="observed")
ax.set_xscale("log"); ax.set_xlabel(r"$n_e$"); ax.set_ylabel(r"$P(A \mid n_e)$")
ax.legend(fontsize=7); fig.tight_layout()
fig.savefig("fig_adoption_vs_null.pdf")

# NOTE: to reproduce the published figure exactly, swap `agg` for the paper's
# own exposures_per_follower / toxic_adoptions_per_follower dictionaries.
# The null construction is unchanged.

# ===========================================================================
# (B) WITHIN-AGENT PERMUTATION NULL for the event-level model.
#     Shuffling resp_toxic within an agent preserves that agent's activity
#     level and toxicity rate EXACTLY -- precisely the null the reviewer
#     specifies -- while destroying any dependence on prior exposure.
# ===========================================================================
def permutation_null(sub, cols, R=50, n_agents=5000, seed=0):
    rng = np.random.default_rng(seed)
    ids = rng.choice(sub.agent.unique(),
                     min(n_agents, sub.agent.nunique()), replace=False)
    f = (sub[sub.agent.isin(ids)]
         .sort_values(["agent", "t_ms"]).reset_index(drop=True))
    codes = pd.factorize(f.agent)[0]
    starts = np.flatnonzero(np.r_[True, codes[1:] != codes[:-1]])
    bounds = list(zip(starts, np.r_[starts[1:], len(f)]))

    X = sm.add_constant(f[cols + [c + "_bar" for c in cols]].astype(float))
    y = f.resp_toxic.to_numpy(float)
    observed = sm.Logit(y, X).fit(disp=False).params

    null = []
    for _ in range(R):
        yp = y.copy()
        for s, e in bounds:
            yp[s:e] = rng.permutation(yp[s:e])
        null.append(sm.Logit(yp, X).fit(disp=False).params)
    null = pd.DataFrame(null)

    res = pd.DataFrame({
        "observed": observed[cols],
        "null_mean": null[cols].mean(),
        "null_sd": null[cols].std(),
        "null_p025": null[cols].quantile(.025),
        "null_p975": null[cols].quantile(.975),
    })
    res["z_vs_null"] = (res.observed - res.null_mean) / res.null_sd
    return res

print("\n--- permutation null (within-agent shuffle) ---")
print(permutation_null(sub, COLS3, R=50, n_agents=5000))

# ===========================================================================
# (C) ADDITIONAL CLUSTERING AND TIME.
#     statsmodels has no native two-way clustering for Logit, so we report
#     one-way clusterings side by side and add month fixed effects.
# ===========================================================================
sub["month"] = sub.t.dt.to_period("M").astype(str)
M = pd.get_dummies(sub["month"], prefix="m", drop_first=True).astype(float)

Xf = sm.add_constant(pd.concat(
    [sub[COLS3 + [c + "_bar" for c in COLS3]].astype(float).reset_index(drop=True),
     M.reset_index(drop=True)], axis=1))
y = sub["resp_toxic"].astype(float).reset_index(drop=True)

for label, grp in [("cluster: agent", pd.factorize(sub["agent"])[0]),
                   ("cluster: post",  pd.factorize(sub["post_id"])[0])]:
    m = sm.Logit(y, Xf).fit(disp=False, cov_type="cluster",
                            cov_kwds={"groups": grp})
    print(f"\n=== {label} (month FE included) ===")
    print(pd.DataFrame({"coef": m.params[COLS3], "OR": np.exp(m.params[COLS3]),
                        "se": m.bse[COLS3], "p": m.pvalues[COLS3]}))

# If a creator or model column exists in chirpers_df, map it onto ev.agent and
# add it as a third clustering level in the loop above. If both agent and post
# clusterings give similar standard errors, two-way clustering would add little
# and reporting the more conservative of the two is defensible.

In [ ]:
edges = np.unique(np.round(np.logspace(0, np.log10(n_arr.max()), 20)).astype(int))
b = pd.cut(agg.n, edges, include_lowest=True)
o = agg.assign(ad=agg.k > 0).groupby(b, observed=True)["ad"].agg(["mean", "size"])

rng = np.random.default_rng(0)
sim = np.array([pd.DataFrame({"b": b, "ad": rng.binomial(n_arr, rng.choice(rates, len(n_arr))) > 0})
                  .groupby("b", observed=True)["ad"].mean().values for _ in range(200)])
band = np.percentile(sim, [2.5, 97.5], axis=0)
print(pd.DataFrame({"obs": o["mean"].values, "agents": o["size"].values,
                    "null_lo": band[0], "null_hi": band[1]}, index=o.index))

In [ ]:
print(chirpers_df[["user", "spec", "world", "lang"]].nunique())
print(chirpers_df["spec"].dropna().iloc[:3])

In [ ]:
"""
(1) Figure: observed P(A | n_e) against the null band, log-binned.
(2) Regression clustered on human creator (chirpers_df.user) and world.

Requires: ev, sub, COLS3, chirpers_df, figs_dir
"""

import numpy as np, pandas as pd, statsmodels.api as sm
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, FuncFormatter

agg = ev.groupby("agent").agg(n=("resp_toxic", "size"), k=("resp_toxic", "sum"))
#agg = ev.groupby("agent").agg(
#    n=("post_id", "nunique"),
#    k=("post_id", lambda s: s[ev.loc[s.index, "resp_toxic"] == 1].nunique()),
#)
n_arr = agg.n.to_numpy()
rates = (agg.k / agg.n).to_numpy()
adopt = (agg.k > 0).to_numpy()

N_BINS, R, MIN_AGENTS = 20, 500, 100

edges = np.unique(np.round(np.logspace(0, np.log10(n_arr.max()), N_BINS)).astype(int))
edges = np.r_[edges[0] - 1, edges]                      # so the first bin includes n=1
bin_id = np.digitize(n_arr, edges[1:], right=True)
keep = bin_id < len(edges) - 1

# geometric bin centres, for a log x-axis
lo, hi = edges[:-1] + 1, edges[1:]
centre = np.sqrt(lo * hi)

obs_p = np.array([adopt[bin_id == b].mean() if (bin_id == b).any() else np.nan
                  for b in range(len(centre))])
obs_n = np.array([(bin_id == b).sum() for b in range(len(centre))])

# Null: each agent keeps a toxicity RATE drawn from the observed distribution,
# assigned independently of its activity level. Preserves both marginals and
# destroys only the association between them.
rng = np.random.default_rng(0)
sim = np.full((R, len(centre)), np.nan)
for r in range(R):
    k = rng.binomial(n_arr, rng.choice(rates, size=len(n_arr)))
    a = k > 0
    for b in range(len(centre)):
        m = bin_id == b
        if m.any():
            sim[r, b] = a[m].mean()

null_lo, null_md, null_hi = np.nanpercentile(sim, [2.5, 50, 97.5], axis=0)

p_hat = ev.resp_toxic.mean()
homog = 1 - (1 - p_hat) ** centre

# --- plot ------------------------------------------------------------------
plt.rcParams.update({"font.size": 15, "axes.spines.top": False,
                     "axes.spines.right": False})
fig, ax = plt.subplots(figsize=(4.6, 3.1))

ax.fill_between(centre, null_lo, null_hi, color="#9ecae1", alpha=.55, lw=0, zorder=1,
                label="null: activity and toxicity rate preserved (95\\%)")
ax.plot(centre, null_md, color="#3182bd", lw=1, zorder=2)
ax.plot(centre, homog, color="#3182bd", ls=":", lw=1.2, zorder=2,
        label=r"null: $1-(1-p)^{n_{\mathcal{S}}}$")

solid = obs_n >= MIN_AGENTS
ax.plot(centre, obs_p, color="#d95f02", lw=1.6, zorder=4, label="observed")
ax.scatter(centre[solid], obs_p[solid], s=18, color="#d95f02",
           edgecolor="white", lw=.6, zorder=5)
ax.scatter(centre[~solid], obs_p[~solid], s=16, facecolor="white",
           edgecolor="#d95f02", lw=1.0, zorder=5)

ax.set_xscale("log")
ax.set_xlabel(r"$n_{\mathcal{S}}$")
ax.set_ylabel(r"P$(\mathcal{R}_{T} \mid n_{\mathcal{S}})$")
ax.set_ylim(-0.02, 1.03)
ax.xaxis.set_major_locator(LogLocator(base=10))
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.legend(frameon=False, fontsize=11, loc="lower right")
fig.tight_layout()


handles, labels = ax.get_legend_handles_labels()

# Create a new figure for the legend
fig_legend = plt.figure(figsize=(4, 1))   # adjust size as needed

legend = fig_legend.legend(handles, labels, ncol=3, frameon=True)

ax.legend().set_visible(False)

# Remove axes
fig_legend.gca().set_axis_off()

# Save legend
fig_legend.savefig(f"{figs_dir}/adoption_vs_null_legend.pdf", dpi=300, bbox_inches="tight")


fig.savefig(f"{figs_dir}/adoption_vs_null.pdf", bbox_inches="tight")
plt.show()

# Open markers = bins with fewer than MIN_AGENTS agents; mention this in the
# caption so the sparse tail is visibly down-weighted rather than silently used.

In [ ]:



tbl = pd.DataFrame({"n_lo": lo, "n_hi": hi, "agents": obs_n, "observed": obs_p,
                    "null_lo": null_lo, "null_md": null_md, "null_hi": null_hi})
tbl["excess"] = tbl.observed - tbl.null_md
print(tbl.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
tbl.to_csv("adoption_vs_null.csv", index=False)

# Agent-weighted excess, restricted to bins that carry real mass
m = tbl.agents >= MIN_AGENTS
print(f"\nagent-weighted excess over null median (bins with >={MIN_AGENTS} agents): "
      f"{np.average(tbl.excess[m], weights=tbl.agents[m]):+.4f}")
print(f"bins above the null 97.5th pct: "
      f"{int((tbl.observed[m] > tbl.null_hi[m]).sum())} / {int(m.sum())}")

# ===========================================================================
# 2. CLUSTERING ON HUMAN CREATOR AND WORLD
# ===========================================================================
meta = chirpers_df.set_index("id")[["user", "world", "lang"]]
sub = sub.drop(columns=["user", "world", "lang"], errors="ignore").join(meta, on="agent")
print("\nunmatched agents:", sub[["user", "world"]].isna().mean().to_dict())

# Unmatched agents become singleton clusters rather than one giant NaN cluster.
for c in ["user", "world", "lang"]:
    sub[c] = sub[c].astype("object").where(sub[c].notna(), "NA_" + sub["agent"].astype(str))

# Rebuild the design matrix once, index-aligned, to avoid silent misalignment.
f = sub.reset_index(drop=True)
f["month"] = f.t.dt.tz_localize(None).dt.to_period("M").astype(str)
M = pd.get_dummies(f["month"], prefix="m", drop_first=True).astype(float)
X = sm.add_constant(pd.concat(
    [f[COLS3 + [c + "_bar" for c in COLS3]].astype(float), M], axis=1))
y = f["resp_toxic"].astype(float)

base = sm.Logit(y, X).fit(disp=False)          # coefficients are identical across
                                               # clusterings; only the SEs change
rows = {}
for label, col in [("agent", "agent"), ("post", "post_id"),
                   ("creator", "user"), ("world", "world")]:
    g = pd.factorize(f[col].astype(str))[0]
    m_ = sm.Logit(y, X).fit(disp=False, cov_type="cluster", cov_kwds={"groups": g})
    rows[label] = pd.Series(m_.bse[COLS3])
    print(f"\n=== cluster: {label}  ({len(np.unique(g)):,} groups) ===")
    print(pd.DataFrame({"OR": np.exp(m_.params[COLS3]), "se": m_.bse[COLS3],
                        "z": m_.tvalues[COLS3], "p": m_.pvalues[COLS3]}))

print("\n--- standard errors by clustering level ---")
se = pd.DataFrame(rows)
se["max"] = se.max(axis=1)
se["widest"] = se[list(rows)].idxmax(axis=1)
print(se)
print("\nReport the widest as primary; the others as robustness.")

In [ ]:
"""
Addressing the reviewer's comment on dependence between observations.

(1) Cluster bootstrap of the Mantel-Haenszel OR, resampling at the AGENT and
    at the CREATOR level (the reviewer's explicit request).
(2) Quantify the agent-post pairs counted in both rows of the 2x2, so the
    observational unit can be defined precisely in the manuscript.
(3) Comment-level replication, to show the unit choice is not material.

Requires: prompted_tables, non_prompted_tables, chirpers, agents_prompted_to_be_toxic,
          informative(), toxic_adoption_df, non_toxic_adoption_df, chirpers_df, ev
"""

import numpy as np, pandas as pd

# ===========================================================================
# 1. CLUSTER BOOTSTRAP OF THE MH ODDS RATIO
# ===========================================================================
# Rebuild tables WITH their agent ids attached, in the same loop order used to
# build prompted_tables / non_prompted_tables, so nothing can misalign.
agent_ids_p, agent_ids_np = [], []
agent_ids_p_not_sex = []
for chirper in chirpers:
    (agent_ids_p if chirper in agents_prompted_to_be_toxic else agent_ids_np).append(chirper)
    if chirper in agents_prompted_to_be_toxic_non_sex:
        agent_ids_p_not_sex.append(chirper)

assert len(agent_ids_p) == len(prompted_tables)
assert len(agent_ids_np) == len(non_prompted_tables)

def mh_parts(tables):
    """Per-stratum numerator and denominator of the MH estimator."""
    A = np.stack([np.asarray(t, float) for t in tables])
    n = A.sum(axis=(1, 2))
    return A[:, 0, 0] * A[:, 1, 1] / n, A[:, 0, 1] * A[:, 1, 0] / n

def cluster_boot(tables, agent_ids, cluster_of=None, R=2000, seed=0):
    """Resample clusters with replacement; recompute the pooled MH OR each time.

    cluster_of : dict agent -> cluster id. If None, each agent is its own
                 cluster (agent-level bootstrap).
    """
    keep = [i for i, t in enumerate(tables) if informative(np.asarray(t, float))]
    tabs = [tables[i] for i in keep]
    ids = [agent_ids[i] for i in keep]
    num, den = mh_parts(tabs)

    cl = np.array([cluster_of.get(a, f"NA_{a}") if cluster_of else a for a in ids])
    uniq, inv = np.unique(cl, return_inverse=True)
    members = [np.flatnonzero(inv == j) for j in range(len(uniq))]

    rng = np.random.default_rng(seed)
    out = np.empty(R)
    for r in range(R):
        pick = rng.integers(0, len(uniq), len(uniq))
        sel = np.concatenate([members[j] for j in pick])
        out[r] = num[sel].sum() / den[sel].sum()
    return num.sum() / den.sum(), np.percentile(out, [2.5, 97.5]), len(uniq)

creator_of = chirpers_df.set_index("id")["user"].to_dict()

for name, tabs, ids in [("all", prompted_tables + non_prompted_tables,
                                agent_ids_p + agent_ids_np),
                        ("prompted", prompted_tables, agent_ids_p),
                        ("prompted-not-sex", prompted_tables_not_sex, agent_ids_p_not_sex),
                        ("non-prompted", non_prompted_tables, agent_ids_np)]:
    pt, ci_a, k_a = cluster_boot(tabs, ids, None)
    _,  ci_c, k_c = cluster_boot(tabs, ids, creator_of)
    print(f"{name:>13}  MH={pt:.3f} | agent boot {np.round(ci_a,3)} ({k_a:,} clusters)"
          f" | creator boot {np.round(ci_c,3)} ({k_c:,} clusters)")

# Compare these to the analytic CIs already reported (3.40 [3.33, 3.48],
# 2.98 [2.89, 3.07], 4.11 [3.97, 4.24]). If the creator bootstrap is wider,
# report it as primary; if comparable, report that the analytic intervals
# are not materially understated.

# ===========================================================================
# 2. OVERLAP: agent-post pairs counted in BOTH rows of the 2x2
# ===========================================================================
tox_pairs = toxic_adoption_df[["chirper", "chirp"]].drop_duplicates()
non_pairs = non_toxic_adoption_df[["chirper", "chirp"]].drop_duplicates()
both = tox_pairs.merge(non_pairs, on=["chirper", "chirp"])

total_cells = len(tox_pairs) + len(non_pairs)
print(f"\nagent-post pairs, toxic row:     {len(tox_pairs):,}")
print(f"agent-post pairs, non-toxic row: {len(non_pairs):,}")
print(f"pairs counted in BOTH rows:      {len(both):,} "
      f"({100*len(both)/total_cells:.2f}% of all cell entries)")
print(f"agents affected:                 {both.chirper.nunique():,}")

# ===========================================================================
# 3. COMMENT-LEVEL REPLICATION (one row per response, no double counting)
# ===========================================================================
import statsmodels.stats.contingency_tables as ct

cl_tabs, cl_ids = [], []
for agent, g in ev.groupby("agent", sort=False):
    a = ((g.resp_toxic == 1) & (g.stim_toxic == 1)).sum()
    b = ((g.resp_toxic == 1) & (g.stim_toxic == 0)).sum()
    c = ((g.resp_toxic == 0) & (g.stim_toxic == 1)).sum()
    d = ((g.resp_toxic == 0) & (g.stim_toxic == 0)).sum()
    t = np.array([[a, b], [c, d]], float)
    if informative(t):
        cl_tabs.append(t); cl_ids.append(agent)

res = ct.StratifiedTable(cl_tabs)
print(f"\ncomment-level MH OR: {res.oddsratio_pooled:.3f} "
      f"{np.round(res.oddsratio_pooled_confint(), 3)}  ({len(cl_tabs):,} strata)")
_, ci_c, k_c = cluster_boot(cl_tabs, cl_ids, creator_of)
print(f"creator bootstrap CI: {np.round(ci_c, 3)} ({k_c:,} clusters)")

In [ ]:
boot_lo = np.array([3.151, 2.772, 3.567])
boot_hi = np.array([3.745, 3.241, 4.897])


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

means = np.array([result.oddsratio_pooled, prompted_result.oddsratio_pooled, non_prompted_result.oddsratio_pooled])


lower_errors = means - boot_lo
upper_errors = boot_hi - means
colors = ["gray"] + sns.color_palette()[2:4][::-1]

x_labels = ["All", "Prompted", "Non-Prompted"]

x_pos = np.arange(len(x_labels))

lower_errors = np.array([means[0]-result.oddsratio_pooled_confint()[0], means[1]-prompted_result.oddsratio_pooled_confint()[0], means[2]-non_prompted_result.oddsratio_pooled_confint()[0]])
upper_errors = np.array([result.oddsratio_pooled_confint()[1]-means[0], prompted_result.oddsratio_pooled_confint()[1]-means[1], non_prompted_result.oddsratio_pooled_confint()[1]-means[2]])

for i, c in enumerate(colors):
    if i == 0:
        marker = "o"
    elif i == 1:
        marker = "^"
    else:
        marker = "H"
    ax.plot(x_labels[i], means[i], 
            linewidth=0, marker=marker, color=c, 
            markersize=12, markerfacecolor="white", markeredgewidth=2)#, label=x_labels[i])
    
 
    # Vertical error bars on top for precision
    ax.errorbar(
        x_pos[i], means[i],
        yerr=[[lower_errors[i]], [upper_errors[i]]],
        fmt="none",
        ecolor=c,
        elinewidth=1.5,
        capsize=5,
        capthick=1.5,
    )

#ax.fill_between(
##    x_pos,
#    means - lower_errors,
#    means + upper_errors,
    #color=ci_color,
#    alpha=0.40,
#    label=f"{int(CONFIDENCE*100)}% CI",
#)

CONFIDENCE = 0.95
# Styling
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, fontsize=15)
ax.set_ylabel("Odds Ratio", fontsize=17)
#ax.set_title(f"Group Means with {int(CONFIDENCE*100)}% Confidence Intervals", fontsize=14, fontweight="bold")
#ax.legend(framealpha=0.9, fontsize=10, loc="best")
#ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.grid(axis="x", linestyle="--", alpha=0)

#ax.set_ylim(2, 3.6)
#ax.set_yticks(np.arange(0, 3.6, 1.5))
plt.tight_layout()

fig.savefig(os.path.join(figs_dir, "odds_ratio.pdf"), dpi=200, bbox_inches="tight")

plt.show()

In [ ]:
agent_ids_n = agent_ids_np

In [ ]:
def boot_ratio(R=2000, seed=0):
    keep_p = [i for i, t in enumerate(prompted_tables) if informative(np.asarray(t, float))]
    keep_n = [i for i, t in enumerate(non_prompted_tables) if informative(np.asarray(t, float))]
    num_p, den_p = mh_parts([prompted_tables[i] for i in keep_p])
    num_n, den_n = mh_parts([non_prompted_tables[i] for i in keep_n])
    cl_p = np.array([creator_of.get(agent_ids_p[i], f"NA_{agent_ids_p[i]}") for i in keep_p])
    cl_n = np.array([creator_of.get(agent_ids_n[i], f"NA_{agent_ids_n[i]}") for i in keep_n])

    uniq = np.unique(np.concatenate([cl_p, cl_n]))
    mem_p = {c: np.flatnonzero(cl_p == c) for c in uniq}
    mem_n = {c: np.flatnonzero(cl_n == c) for c in uniq}

    rng, out = np.random.default_rng(seed), []
    for _ in range(R):
        pick = uniq[rng.integers(0, len(uniq), len(uniq))]
        ip = np.concatenate([mem_p[c] for c in pick]); iN = np.concatenate([mem_n[c] for c in pick])
        if len(ip) and len(iN):
            out.append((num_n[iN].sum()/den_n[iN].sum()) / (num_p[ip].sum()/den_p[ip].sum()))
    return np.percentile(out, [2.5, 50, 97.5])

print(boot_ratio())

In [ ]:
print("fallback rate:", sub.groupby("agent")["user"].first().astype(str).str.startswith("NA_").mean())
print("rows:", len(f), "(expect 3,925,541)")

In [ ]:
chirper_exposures = []
chirper_prop_adoptions_over_exposures = []

for chirper in count_chirper_toxic_adoption_non_toxic_content:

    exposure_non_toxic_content = count_chirper_exposition_non_toxic_content[chirper]
    exposure_toxic_content = count_chirper_exposition_toxic_content[chirper]

    n_exposures = exposure_non_toxic_content + exposure_toxic_content

    toxic_adoption_non_toxic_content = count_chirper_toxic_adoption_non_toxic_content[chirper]
    toxic_adoption_toxic_content = count_chirper_toxic_adoption_toxic_content[chirper]

    non_toxic_adoption_non_toxic_content = count_chirper_non_toxic_adoption_non_toxic_content[chirper]
    non_toxic_adoption_toxic_content = count_chirper_non_toxic_adoption_toxic_content[chirper]

    total_adoption = toxic_adoption_non_toxic_content + toxic_adoption_toxic_content + non_toxic_adoption_non_toxic_content + non_toxic_adoption_toxic_content

    #chirper_exposure = exposure_non_toxic_content + exposure_toxic_content
    chirper_exposures.append((exposure_non_toxic_content/n_exposures, "Non-Toxic"))
    chirper_exposures.append((exposure_toxic_content/n_exposures, "Toxic"))

    chirper_prop_adoptions_over_exposures.append((toxic_adoption_non_toxic_content, "Toxic_Over_Non-Toxic"))
    chirper_prop_adoptions_over_exposures.append((non_toxic_adoption_non_toxic_content, "Non-Toxic_Over_Non-Toxic"))
        
    chirper_prop_adoptions_over_exposures.append((toxic_adoption_toxic_content, "Toxic_Over_Toxic"))
    chirper_prop_adoptions_over_exposures.append((non_toxic_adoption_toxic_content, "Non-Toxic_Over_Toxic"))

In [ ]:
raw_adoptions = np.array([[sum(count_chirper_toxic_adoption_toxic_content.values()), sum(count_chirper_toxic_adoption_non_toxic_content.values())],
                         [sum(count_chirper_non_toxic_adoption_toxic_content.values()), sum(count_chirper_non_toxic_adoption_non_toxic_content.values())]])

In [ ]:
sum(sum(raw_adoptions))

In [ ]:
prompted_count_chirper_toxic_adoption_toxic_content = sum([v for k, v in count_chirper_toxic_adoption_toxic_content.items() if k in agents_prompted_to_be_toxic])
prompted_count_chirper_toxic_adoption_non_toxic_content = sum([v for k, v in count_chirper_toxic_adoption_non_toxic_content.items() if k in agents_prompted_to_be_toxic])
prompted_count_chirper_non_toxic_adoption_toxic_content = sum([v for k, v in count_chirper_non_toxic_adoption_toxic_content.items() if k in agents_prompted_to_be_toxic])
prompted_count_chirper_non_toxic_adoption_non_toxic_content = sum([v for k, v in count_chirper_non_toxic_adoption_non_toxic_content.items() if k in agents_prompted_to_be_toxic])

prompted_raw_adoptions = np.array([[prompted_count_chirper_toxic_adoption_toxic_content, prompted_count_chirper_toxic_adoption_non_toxic_content],
                         [prompted_count_chirper_non_toxic_adoption_toxic_content, prompted_count_chirper_non_toxic_adoption_non_toxic_content]])

In [ ]:
non_prompted_count_chirper_toxic_adoption_toxic_content = sum([v for k, v in count_chirper_toxic_adoption_toxic_content.items() if k not in agents_prompted_to_be_toxic])
non_prompted_count_chirper_toxic_adoption_non_toxic_content = sum([v for k, v in count_chirper_toxic_adoption_non_toxic_content.items() if k not in agents_prompted_to_be_toxic])
non_prompted_count_chirper_non_toxic_adoption_toxic_content = sum([v for k, v in count_chirper_non_toxic_adoption_toxic_content.items() if k not in agents_prompted_to_be_toxic])
non_prompted_count_chirper_non_toxic_adoption_non_toxic_content = sum([v for k, v in count_chirper_non_toxic_adoption_non_toxic_content.items() if k not in agents_prompted_to_be_toxic])

non_prompted_raw_adoptions = np.array([[non_prompted_count_chirper_toxic_adoption_toxic_content, non_prompted_count_chirper_toxic_adoption_non_toxic_content],
                         [non_prompted_count_chirper_non_toxic_adoption_toxic_content, non_prompted_count_chirper_non_toxic_adoption_non_toxic_content]])

In [ ]:
np.sum(raw_adoptions)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_style('white')
disp = ConfusionMatrixDisplay(confusion_matrix=raw_adoptions/np.sum(raw_adoptions),
                              display_labels=["T", "N"],
                             )

disp.plot(cmap="Blues", colorbar=False)

plt.xlabel("$\mathcal{S}$", fontsize=23)
plt.ylabel("$\mathcal{R}$", fontsize=23)

plt.tight_layout()

plt.savefig(os.path.join(figs_dir, "raw_adoptions_heatmap.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_style('white')
disp = ConfusionMatrixDisplay(confusion_matrix=prompted_raw_adoptions/np.sum(prompted_raw_adoptions),
                              display_labels=["T", "N"],
                             )

disp.plot(cmap="Blues", colorbar=False)

plt.xlabel("$\mathcal{S}$", fontsize=23)
plt.ylabel("$\mathcal{R}$", fontsize=23)

plt.tight_layout()

plt.savefig(os.path.join(figs_dir, "raw_adoptions_prompted_agents_heatmap.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_style('white')
disp = ConfusionMatrixDisplay(confusion_matrix=non_prompted_raw_adoptions/np.sum(non_prompted_raw_adoptions, axis=0),
                              display_labels=["T", "N"],
                             )

disp.plot(cmap="Blues", colorbar=False)

plt.xlabel("$\mathcal{S}$", fontsize=23)
plt.ylabel("$\mathcal{R}$", fontsize=23)

plt.tight_layout()

plt.savefig(os.path.join(figs_dir, "raw_adoptions_non_prompted_agents_heatmap.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
raw_adoptions/np.sum(raw_adoptions, axis=1)

In [ ]:
raw_adoptions[0]/np.sum(raw_adoptions, axis=1)[0]

In [ ]:
raw_adoptions[1]/np.sum(raw_adoptions, axis=1)[1]

In [ ]:
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(6, 0.3))  # wide and short for horizontal

norm = mcolors.Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap="Blues"),
    cax=ax,
    orientation="horizontal",
)
cbar.ax.tick_params(labelsize=17)
#cbar.set_label("Value", fontsize=12)  # optional label

fig.tight_layout()
fig.savefig(os.path.join(figs_dir, "colorbar.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_style('white')
disp = ConfusionMatrixDisplay(confusion_matrix=raw_adoptions/np.sum(raw_adoptions, axis=0),
                              display_labels=["T", "N"],
                             )

disp.plot(cmap="Blues", colorbar=False)

plt.xlabel("$\mathcal{S}$", fontsize=23)
plt.ylabel("$\mathcal{R}$", fontsize=23)

plt.tight_layout()

plt.savefig(os.path.join(figs_dir, "norm_by_stimuli_adoptions_heatmap.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_style('white')
disp = ConfusionMatrixDisplay(confusion_matrix=prompted_raw_adoptions/np.sum(prompted_raw_adoptions, axis=0),
                              display_labels=["T", "N"],
                             )

disp.plot(cmap="Blues", colorbar=False)

plt.xlabel("$\mathcal{S}$", fontsize=23)
plt.ylabel("$\mathcal{R}$", fontsize=23)

plt.tight_layout()

plt.savefig(os.path.join(figs_dir, "norm_by_stimuli_adoptions_prompted_agents_heatmap.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_style('white')
disp = ConfusionMatrixDisplay(confusion_matrix=non_prompted_raw_adoptions/np.sum(non_prompted_raw_adoptions, axis=0),
                              display_labels=["T", "N"],
                             )

disp.plot(cmap="Blues", colorbar=False)

plt.xlabel("$\mathcal{S}$", fontsize=23)
plt.ylabel("$\mathcal{R}$", fontsize=23)

plt.tight_layout()

plt.savefig(os.path.join(figs_dir, "norm_by_stimuli_adoptions_non_prompted_agents_heatmap.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:

norm_table = np.array([list(raw_adoptions[0]/np.sum(raw_adoptions, axis=1)[0]), list(raw_adoptions[1]/np.sum(raw_adoptions, axis=1)[1])])

In [ ]:
list(raw_adoptions[0]/np.sum(raw_adoptions, axis=1)[0])

In [ ]:
list(raw_adoptions[1]/np.sum(raw_adoptions, axis=1)[1])

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_style('white')
disp = ConfusionMatrixDisplay(confusion_matrix=norm_table,
                              display_labels=["T", "N"],
                             )

disp.plot(cmap="Blues", colorbar=False)

plt.xlabel("$\mathcal{S}$", fontsize=23)
plt.ylabel("$\mathcal{R}$", fontsize=23)

plt.tight_layout()

plt.savefig(os.path.join(figs_dir, "norm_by_responses_adoptions_heatmap.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
dist_exposure_df = pd.DataFrame(chirper_exposures, columns=["Exposure", "Type"])

In [ ]:
dist_adoptions_over_exposures_df = pd.DataFrame(chirper_prop_adoptions_over_exposures, columns=["Adoption", "Type"])

In [ ]:
dist_adoptions_over_exposures_df.groupby("Type").value_counts(normalize=True)

In [ ]:
colors = [palette[1], palette[2], palette[-1], palette[0]]

In [ ]:
sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(6, 4))

for (type_val, group), color in zip(
    dist_adoptions_over_exposures_df.groupby("Type"), colors
):
    vals = np.sort(group["Adoption"].dropna().values)
    survival = 1 - np.arange(1, len(vals) + 1) / len(vals)
    ax.step(vals, survival, where="post", color=color,
            linewidth=1.8, label=type_val)

#ax.set_yscale("log")
ax.set_xscale("log")

#ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:g}%"))

ax.set_xlabel(r"$|\mathcal{R}|$")
#ax.set_ylabel("Users with $\mathcal{R} ≥ x$")
ax.legend(title="Type", framealpha=0.85, edgecolor="0.8")
ax.grid(linestyle="--", linewidth=0.6, alpha=0.7)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(6, 4))

sns.ecdfplot(                          # axes-level, accepts ax=
    data=dist_adoptions_over_exposures_df,
    x="Adoption",
    hue="Type",
    palette=colors,
    ax=ax,
    stat="percent"
)

ax.get_legend().remove()               # safe legend removal

ax.set_xscale("log")                   # set on ax, not plt
#ax.set_yscale("log")

ax.grid(linestyle="--")
ax.set_xlabel(r"$\mathcal{R}$ ratio")
ax.set_ylabel("Chirpers")
fig.tight_layout()
plt.show()

In [ ]:
dist_adoptions_over_exposures_df[dist_adoptions_over_exposures_df["Adoption"] == 0]

In [ ]:
sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(
    data=dist_adoptions_over_exposures_df,
    x="Adoption",
    hue="Type",
    bins=200,
    element="step",
    fill=True,
    palette=colors,
    ax=ax,
    kde=True,
    #label="Type"
)
legend = ax.get_legend()
print(legend)
handles = legend.legend_handles
labels = [t.get_text() for t in legend.get_texts()]
print(labels)

order = [0, 2, 1, -1]  # indices of labels in the order you want

ax.legend(handles=[handles[i] for i in order],
          labels=[labels[i] for i in order])
legend = ax.get_legend()

handles = legend.legend_handles

#plt.yscale("log")
#plt.xscale("log")
# Get existing legend
# Suppose ax is any axes with a legend
plt.grid(linestyle='--')

ax.set_xlabel("Responses")
ax.set_ylabel("Chirpers")

#print(labels)

# Create a new figure for the legend
fig_legend = plt.figure(figsize=(4, 1))  # adjust size as needed

new_labels = [
    r'$\mathcal{R}_{T} \rightarrow \mathcal{S}_N$',
    r'$\mathcal{R}_T \rightarrow \mathcal{S}_T$',
    r'$\mathcal{R}_N \rightarrow \mathcal{S}_N$',
    r'$\mathcal{R}_N \rightarrow \mathcal{S}_T$'
]
legend = fig_legend.legend(handles, new_labels, ncol=4, frameon=True)

ax.legend().set_visible(False)

# Remove axes
fig_legend.gca().set_axis_off()
# Save legend
fig_legend.savefig(os.path.join(figs_dir, "line_legend.pdf"), dpi=300, bbox_inches="tight")

plt.tight_layout()

fig.savefig(os.path.join(figs_dir, "count_adoption_over_exposure_type.pdf"), dpi=200, bbox_inches="tight")

print(new_labels)

plt.show()

In [ ]:
(0.3-0.29)/((0.3 + 0.29)/2)

In [ ]:
from collections import defaultdict

In [ ]:
users_by_exposure_non_toxic_content = defaultdict(list)
users_by_exposure_toxic_content = defaultdict(list)

for user, n_e in count_chirper_exposition_non_toxic_content.items():
    users_by_exposure_non_toxic_content[n_e].append(user)

for user, n_e in count_chirper_exposition_toxic_content.items():
    users_by_exposure_toxic_content[n_e].append(user)

# 2. Compute P(A* | n_e = N)
prob_toxic_adoption_given_exposure_non_toxic_content = {}
prob_toxic_adoption_given_exposure_toxic_content = {}

# 3. Compute P(A | n_e = N)
prob_non_toxic_adoption_given_exposure_non_toxic_content = {}
prob_non_toxic_adoption_given_exposure_toxic_content = {}

minimum_toxic_adoption = 0 # np.percentile(toxic_adoptions, 25) # 0 (?)
minimum_non_toxic_adoption = 0 # np.percentile(non_toxic_adoptions, 25) # 0 (?)

for N, users in users_by_exposure_non_toxic_content.items():
    # users with exposure N to non-toxic content who adopted
    toxic_adopted = sum(1 for u in users if count_chirper_toxic_adoption_non_toxic_content[u] > minimum_toxic_adoption)
    non_toxic_adopted = sum(1 for u in users if count_chirper_non_toxic_adoption_non_toxic_content[u] > minimum_non_toxic_adoption)
    
    total = len(users)

    prob_toxic_adoption_given_exposure_non_toxic_content[N] = toxic_adopted / total if total > 0 else 0.0
    prob_non_toxic_adoption_given_exposure_non_toxic_content[N] = non_toxic_adopted / total if total > 0 else 0.0

for N, users in users_by_exposure_toxic_content.items():
    # users with exposure N to toxic content who adopted
    toxic_adopted = sum(1 for u in users if count_chirper_toxic_adoption_toxic_content[u] > minimum_toxic_adoption)
    non_toxic_adopted = sum(1 for u in users if count_chirper_non_toxic_adoption_toxic_content[u] > minimum_non_toxic_adoption)
    
    total = len(users)

    prob_toxic_adoption_given_exposure_toxic_content[N] = toxic_adopted / total if total > 0 else 0.0
    prob_non_toxic_adoption_given_exposure_toxic_content[N] = non_toxic_adopted / total if total > 0 else 0.0

In [ ]:
np.percentile(toxic_adoptions, 25)

In [ ]:
np.percentile(non_toxic_adoptions, 25)

In [ ]:
toxic_N_vals_non_toxic_content = np.array(list(prob_toxic_adoption_given_exposure_non_toxic_content.keys()))
toxic_Pr_vals_non_toxic_content = np.array(list(prob_toxic_adoption_given_exposure_non_toxic_content.values()))

non_toxic_N_vals_non_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_non_toxic_content.keys()))
non_toxic_Pr_vals_non_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_non_toxic_content.values()))

toxic_N_vals_toxic_content = np.array(list(prob_toxic_adoption_given_exposure_toxic_content.keys()))
toxic_Pr_vals_toxic_content = np.array(list(prob_toxic_adoption_given_exposure_toxic_content.values()))

non_toxic_N_vals_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_toxic_content.keys()))
non_toxic_Pr_vals_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_toxic_content.values()))

In [ ]:
from scipy.optimize import curve_fit

def get_x_y_fit(N_vals, Pr_vals):

    def log_func(x, a, b):
        return a * np.log(x) + b
    
    # Avoid log(0) → only fit points where N > 0
    fit_mask = N_vals > 0
    
    popt, _ = curve_fit(log_func, N_vals[fit_mask], Pr_vals[fit_mask], bounds=(0.,1))
    a, b = popt
    
    # smooth curve for plotting
    x_fit = np.linspace(1, N_vals.max(), 500)
    #y_fit = log_func(x_fit, a, b)

    y_fit = np.clip(log_func(x_fit, a, b), 0, 1)

    return x_fit, y_fit

In [ ]:
toxic_x_fit_non_toxic, toxic_y_fit_non_toxic = get_x_y_fit(toxic_N_vals_non_toxic_content, toxic_Pr_vals_non_toxic_content)

non_toxic_x_fit_non_toxic, non_toxic_y_fit_non_toxic = get_x_y_fit(non_toxic_N_vals_non_toxic_content, non_toxic_Pr_vals_non_toxic_content)

toxic_x_fit_toxic, toxic_y_fit_toxic = get_x_y_fit(toxic_N_vals_toxic_content, toxic_Pr_vals_toxic_content)

non_toxic_x_fit_toxic, non_toxic_y_fit_toxic = get_x_y_fit(non_toxic_N_vals_toxic_content, non_toxic_Pr_vals_toxic_content)

In [ ]:
plt.figure(figsize=(5,4))

# scatter
plt.scatter(toxic_N_vals_non_toxic_content, toxic_Pr_vals_non_toxic_content, alpha=0.7, label="Non-Toxic")
plt.scatter(toxic_N_vals_toxic_content, toxic_Pr_vals_toxic_content, alpha=0.7, label="Toxic")

# regression curve
plt.plot(toxic_x_fit_non_toxic, toxic_y_fit_non_toxic, linewidth=2.5)
plt.plot(toxic_x_fit_toxic, toxic_y_fit_toxic, linewidth=2.5)

# labels
plt.xlabel("Number of Exposures $n_e$", fontsize=12)
plt.ylabel("Probability of Toxic Adoption $Pr(A^*|n_e)$", fontsize=12)

#plt.xlim(0, N_vals.max() + 10)
#plt.ylim(0, max(Pr_vals) * 1.1)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
treshold = 150

restricted_toxic_N_vals_non_toxic_content = toxic_N_vals_non_toxic_content[toxic_N_vals_non_toxic_content <= treshold]
restricted_toxic_Pr_vals_non_toxic_content = toxic_Pr_vals_non_toxic_content[toxic_N_vals_non_toxic_content <= treshold]

restricted_non_toxic_N_vals_non_toxic_content = non_toxic_N_vals_non_toxic_content[non_toxic_N_vals_non_toxic_content <= treshold]
restricted_non_toxic_Pr_vals_non_toxic_content = non_toxic_Pr_vals_non_toxic_content[non_toxic_N_vals_non_toxic_content <= treshold]

restricted_toxic_N_vals_toxic_content = toxic_N_vals_toxic_content[toxic_N_vals_toxic_content <= treshold]
restricted_toxic_Pr_vals_toxic_content = toxic_Pr_vals_toxic_content[toxic_N_vals_toxic_content <= treshold]

restricted_non_toxic_N_vals_toxic_content = non_toxic_N_vals_toxic_content[non_toxic_N_vals_toxic_content <= treshold]
restricted_non_toxic_Pr_vals_toxic_content = non_toxic_Pr_vals_toxic_content[non_toxic_N_vals_toxic_content <= treshold]

In [ ]:
restricted_toxic_x_fit_non_toxic, restricted_toxic_y_fit_non_toxic = get_x_y_fit(restricted_toxic_N_vals_non_toxic_content, restricted_toxic_Pr_vals_non_toxic_content)

restricted_non_toxic_x_fit_non_toxic, restricted_non_toxic_y_fit_non_toxic = get_x_y_fit(restricted_non_toxic_N_vals_non_toxic_content, restricted_non_toxic_Pr_vals_non_toxic_content)

restricted_toxic_x_fit_toxic, restricted_toxic_y_fit_toxic = get_x_y_fit(restricted_toxic_N_vals_toxic_content, restricted_toxic_Pr_vals_toxic_content)

restricted_non_toxic_x_fit_toxic, restricted_non_toxic_y_fit_toxic = get_x_y_fit(restricted_non_toxic_N_vals_toxic_content, restricted_non_toxic_Pr_vals_toxic_content)

In [ ]:
plt.figure()
plt.boxplot(restricted_toxic_Pr_vals_toxic_content, showfliers=False)
plt.show()

In [ ]:
plt.figure(figsize=(5,4))
sns.set_style("whitegrid")
# scatter
plt.scatter(restricted_toxic_N_vals_non_toxic_content, restricted_toxic_Pr_vals_non_toxic_content, alpha=0.7, label="Non-toxic")
plt.scatter(restricted_toxic_N_vals_toxic_content, restricted_toxic_Pr_vals_toxic_content, alpha=0.7, label="Toxic")


print(mk.original_test(restricted_toxic_Pr_vals_non_toxic_content))
print(mk.original_test(restricted_toxic_Pr_vals_toxic_content))

from scipy.stats import mannwhitneyu
U1, p = mannwhitneyu(restricted_toxic_Pr_vals_non_toxic_content, restricted_toxic_Pr_vals_toxic_content)
print(U1, p)

# regression curve
plt.plot(restricted_toxic_x_fit_non_toxic, restricted_toxic_y_fit_non_toxic, linewidth=2.5)
plt.plot(restricted_toxic_x_fit_toxic, restricted_toxic_y_fit_toxic, linewidth=2.5)

# labels
plt.xlabel("$n_{\mathcal{S}}$", fontsize=16)
plt.ylabel(r"$\textrm{P}(\mathcal{R}_T|n_{\mathcal{S}})$", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

#plt.xlim(0, N_vals.max() + 10)
#plt.ylim(0, max(Pr_vals) * 1.1)
plt.legend(title="Stimulus", fontsize=12, title_fontsize=14, ncols=2)
plt.grid(linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(figs_dir, 'probability_of_toxic_adoption_by_post_toxicity.png'), dpi=200)
plt.savefig(os.path.join(figs_dir, 'probability_of_toxic_adoption_by_post_toxicity.pdf'), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(5,4))
sns.set_style("whitegrid")
# scatter
plt.scatter(restricted_non_toxic_N_vals_non_toxic_content, restricted_non_toxic_Pr_vals_non_toxic_content, alpha=0.7, label="Non-Toxic")
plt.scatter(restricted_non_toxic_N_vals_toxic_content, restricted_non_toxic_Pr_vals_toxic_content, alpha=0.7, label="Toxic")

# regression curve
plt.plot(restricted_non_toxic_x_fit_non_toxic, restricted_non_toxic_y_fit_non_toxic, linewidth=2.5)
plt.plot(restricted_non_toxic_x_fit_toxic, restricted_non_toxic_y_fit_toxic, linewidth=2.5)

# labels
plt.xlabel("$n_e$", fontsize=16)
plt.ylabel("$Pr(\\tilde{A}|n_e)$", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

#plt.xlim(0, N_vals.max() + 10)
#plt.ylim(0, max(Pr_vals) * 1.1)
plt.legend(title="Original Post", fontsize=12, title_fontsize=14, ncols=2)
plt.grid(linestyle='--')
plt.tight_layout()
#plt.savefig('probability_of_non_toxic_adoption_by_post_toxicity.png', dpi=200)
plt.show()

# COMPARING PROMPTED/NON_PROMPTED AGENTS TO TOXICITY

In [ ]:
prompted_users_by_exposure_non_toxic_content = defaultdict(list)
non_prompted_users_by_exposure_non_toxic_content = defaultdict(list)

prompted_users_by_exposure_toxic_content = defaultdict(list)
non_prompted_users_by_exposure_toxic_content = defaultdict(list)

for user, n_e in count_chirper_exposition_non_toxic_content.items():
    if user in agents_prompted_to_be_toxic:
        prompted_users_by_exposure_non_toxic_content[n_e].append(user)
    else: # user in agents_not_prompted_to_be_toxic or does not have description
        non_prompted_users_by_exposure_non_toxic_content[n_e].append(user)

for user, n_e in count_chirper_exposition_toxic_content.items():
    if user in agents_prompted_to_be_toxic:
        prompted_users_by_exposure_toxic_content[n_e].append(user)
    else: # user in agents_not_prompted_to_be_toxic or does not have description
        non_prompted_users_by_exposure_toxic_content[n_e].append(user)

prob_prompted_toxic_adoption_given_exposure_non_toxic_content = {}
prob_non_prompted_toxic_adoption_given_exposure_non_toxic_content = {}

prob_prompted_toxic_adoption_given_exposure_toxic_content = {}
prob_non_prompted_toxic_adoption_given_exposure_toxic_content = {}

#prob_non_toxic_adoption_given_exposure_non_toxic_content = {}
#prob_non_toxic_adoption_given_exposure_toxic_content = {}

minimum_toxic_adoption = 0 # np.percentile(toxic_adoptions, 25) # 0 (?)
minimum_non_toxic_adoption = 0 # np.percentile(non_toxic_adoptions, 25) # 0 (?)

for N, users in prompted_users_by_exposure_non_toxic_content.items():
    toxic_adopted = sum(1 for u in users if count_chirper_toxic_adoption_non_toxic_content[u] > minimum_toxic_adoption)
    non_toxic_adopted = sum(1 for u in users if count_chirper_non_toxic_adoption_non_toxic_content[u] > minimum_non_toxic_adoption)
    
    total = len(users) 

    prob_prompted_toxic_adoption_given_exposure_non_toxic_content[N] = toxic_adopted / total if total > 0 else 0.0
    # prob_non_toxic_adoption_given_exposure_non_toxic_content[N] = non_toxic_adopted / total if total > 0 else 0.0

for N, users in non_prompted_users_by_exposure_non_toxic_content.items():
    toxic_adopted = sum(1 for u in users if count_chirper_toxic_adoption_non_toxic_content[u] > minimum_toxic_adoption)
    non_toxic_adopted = sum(1 for u in users if count_chirper_non_toxic_adoption_non_toxic_content[u] > minimum_non_toxic_adoption)
    
    total = len(users) 
    
    prob_non_prompted_toxic_adoption_given_exposure_non_toxic_content[N] = toxic_adopted / total if total > 0 else 0.0
    # prob_non_toxic_adoption_given_exposure_non_toxic_content[N] = non_toxic_adopted / total if total > 0 else 0.0

for N, users in prompted_users_by_exposure_toxic_content.items():
    toxic_adopted = sum(1 for u in users if count_chirper_toxic_adoption_toxic_content[u] > minimum_toxic_adoption)
    non_toxic_adopted = sum(1 for u in users if count_chirper_non_toxic_adoption_toxic_content[u] > minimum_non_toxic_adoption)
    
    total = len(users) 

    prob_prompted_toxic_adoption_given_exposure_toxic_content[N] = toxic_adopted / total if total > 0 else 0.0
    #prob_non_toxic_adoption_given_exposure_toxic_content[N] = non_toxic_adopted / total if total > 0 else 0.0

for N, users in non_prompted_users_by_exposure_toxic_content.items():
    toxic_adopted = sum(1 for u in users if count_chirper_toxic_adoption_toxic_content[u] > minimum_toxic_adoption)
    non_toxic_adopted = sum(1 for u in users if count_chirper_non_toxic_adoption_toxic_content[u] > minimum_non_toxic_adoption)
    
    total = len(users) 

    prob_non_prompted_toxic_adoption_given_exposure_toxic_content[N] = toxic_adopted / total if total > 0 else 0.0
    #prob_non_toxic_adoption_given_exposure_toxic_content[N] = non_toxic_adopted / total if total > 0 else 0.0

In [ ]:
prompted_toxic_N_vals_non_toxic_content = np.array(list(prob_prompted_toxic_adoption_given_exposure_non_toxic_content.keys()))
prompted_toxic_Pr_vals_non_toxic_content = np.array(list(prob_prompted_toxic_adoption_given_exposure_non_toxic_content.values()))

non_prompted_toxic_N_vals_non_toxic_content = np.array(list(prob_non_prompted_toxic_adoption_given_exposure_non_toxic_content.keys()))
non_prompted_toxic_Pr_vals_non_toxic_content = np.array(list(prob_non_prompted_toxic_adoption_given_exposure_non_toxic_content.values()))

#non_toxic_N_vals_non_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_non_toxic_content.keys()))
#non_toxic_Pr_vals_non_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_non_toxic_content.values()))

prompted_toxic_N_vals_toxic_content = np.array(list(prob_prompted_toxic_adoption_given_exposure_toxic_content.keys()))
prompted_toxic_Pr_vals_toxic_content = np.array(list(prob_prompted_toxic_adoption_given_exposure_toxic_content.values()))

non_prompted_toxic_N_vals_toxic_content = np.array(list(prob_non_prompted_toxic_adoption_given_exposure_toxic_content.keys()))
non_prompted_toxic_Pr_vals_toxic_content = np.array(list(prob_non_prompted_toxic_adoption_given_exposure_toxic_content.values()))

#non_toxic_N_vals_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_toxic_content.keys()))
#non_toxic_Pr_vals_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_toxic_content.values()))

In [ ]:
prompted_toxic_x_fit_non_toxic, prompted_toxic_y_fit_non_toxic = get_x_y_fit(prompted_toxic_N_vals_non_toxic_content, prompted_toxic_Pr_vals_non_toxic_content)
non_prompted_toxic_x_fit_non_toxic, non_prompted_toxic_y_fit_non_toxic = get_x_y_fit(non_prompted_toxic_N_vals_non_toxic_content, non_prompted_toxic_Pr_vals_non_toxic_content)

#non_toxic_x_fit_non_toxic, non_toxic_y_fit_non_toxic = get_x_y_fit(non_toxic_N_vals_non_toxic_content, non_toxic_Pr_vals_non_toxic_content)

prompted_toxic_x_fit_toxic, prompted_toxic_y_fit_toxic = get_x_y_fit(prompted_toxic_N_vals_toxic_content, prompted_toxic_Pr_vals_toxic_content)
non_prompted_toxic_x_fit_toxic, non_prompted_toxic_y_fit_toxic = get_x_y_fit(non_prompted_toxic_N_vals_toxic_content, non_prompted_toxic_Pr_vals_toxic_content)

#non_toxic_x_fit_toxic, non_toxic_y_fit_toxic = get_x_y_fit(non_toxic_N_vals_toxic_content, non_toxic_Pr_vals_toxic_content)

In [ ]:
treshold = 150

restricted_prompted_toxic_N_vals_non_toxic_content = prompted_toxic_N_vals_non_toxic_content[prompted_toxic_N_vals_non_toxic_content <= treshold]
restricted_prompted_toxic_Pr_vals_non_toxic_content = prompted_toxic_Pr_vals_non_toxic_content[prompted_toxic_N_vals_non_toxic_content <= treshold]

restricted_non_prompted_toxic_N_vals_non_toxic_content = non_prompted_toxic_N_vals_non_toxic_content[non_prompted_toxic_N_vals_non_toxic_content <= treshold]
restricted_non_prompted_toxic_Pr_vals_non_toxic_content = non_prompted_toxic_Pr_vals_non_toxic_content[non_prompted_toxic_N_vals_non_toxic_content <= treshold]

#restricted_non_toxic_N_vals_non_toxic_content = non_toxic_N_vals_non_toxic_content[non_toxic_N_vals_non_toxic_content <= treshold]
#restricted_non_toxic_Pr_vals_non_toxic_content = non_toxic_Pr_vals_non_toxic_content[non_toxic_N_vals_non_toxic_content <= treshold]

restricted_prompted_toxic_N_vals_toxic_content = prompted_toxic_N_vals_toxic_content[prompted_toxic_N_vals_toxic_content <= treshold]
restricted_prompted_toxic_Pr_vals_toxic_content = prompted_toxic_Pr_vals_toxic_content[prompted_toxic_N_vals_toxic_content <= treshold]

restricted_non_prompted_toxic_N_vals_toxic_content = non_prompted_toxic_N_vals_toxic_content[non_prompted_toxic_N_vals_toxic_content <= treshold]
restricted_non_prompted_toxic_Pr_vals_toxic_content = non_prompted_toxic_Pr_vals_toxic_content[non_prompted_toxic_N_vals_toxic_content <= treshold]

#restricted_non_toxic_N_vals_toxic_content = non_toxic_N_vals_toxic_content[non_toxic_N_vals_toxic_content <= treshold]
#restricted_non_toxic_Pr_vals_toxic_content = non_toxic_Pr_vals_toxic_content[non_toxic_N_vals_toxic_content <= treshold]

In [ ]:
restricted_prompted_toxic_x_fit_non_toxic, restricted_prompted_toxic_y_fit_non_toxic = get_x_y_fit(restricted_prompted_toxic_N_vals_non_toxic_content, restricted_prompted_toxic_Pr_vals_non_toxic_content)
restricted_non_prompted_toxic_x_fit_non_toxic, restricted_non_prompted_toxic_y_fit_non_toxic = get_x_y_fit(restricted_non_prompted_toxic_N_vals_non_toxic_content, restricted_non_prompted_toxic_Pr_vals_non_toxic_content)

#restricted_non_toxic_x_fit_non_toxic, restricted_non_toxic_y_fit_non_toxic = get_x_y_fit(restricted_non_toxic_N_vals_non_toxic_content, restricted_non_toxic_Pr_vals_non_toxic_content)

restricted_prompted_toxic_x_fit_toxic, restricted_prompted_toxic_y_fit_toxic = get_x_y_fit(restricted_prompted_toxic_N_vals_toxic_content, restricted_prompted_toxic_Pr_vals_toxic_content)
restricted_non_prompted_toxic_x_fit_toxic, restricted_non_prompted_toxic_y_fit_toxic = get_x_y_fit(restricted_non_prompted_toxic_N_vals_toxic_content, restricted_non_prompted_toxic_Pr_vals_toxic_content)

#restricted_non_toxic_x_fit_toxic, restricted_non_toxic_y_fit_toxic = get_x_y_fit(restricted_non_toxic_N_vals_toxic_content, restricted_non_toxic_Pr_vals_toxic_content)

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.set_style("whitegrid")
# scatter

toxic_color = sns.color_palette()[1]
non_toxic_color = sns.color_palette()[0]

plt.scatter(restricted_prompted_toxic_N_vals_non_toxic_content, restricted_prompted_toxic_Pr_vals_non_toxic_content, 
            alpha=0.2, label="Prompted Non-toxic", marker="^", c=sns.color_palette()[0])
plt.scatter(restricted_non_prompted_toxic_N_vals_non_toxic_content, restricted_non_prompted_toxic_Pr_vals_non_toxic_content, 
            alpha=0.2, label="Non-Prompted Non-Toxic", marker="H", c=sns.color_palette()[0])

from scipy.stats import mannwhitneyu
U1, p = mannwhitneyu(restricted_prompted_toxic_Pr_vals_non_toxic_content, restricted_non_prompted_toxic_Pr_vals_non_toxic_content)
print("Non-Toxic stimulus prompted/non-prompted", U1, p)

plt.scatter(restricted_prompted_toxic_N_vals_toxic_content, restricted_prompted_toxic_Pr_vals_toxic_content, 
            alpha=0.2, label="Prompted Toxic", marker="^", c=sns.color_palette()[1])
plt.scatter(restricted_non_prompted_toxic_N_vals_toxic_content, restricted_non_prompted_toxic_Pr_vals_toxic_content, 
            alpha=0.2, label="Non-Prompted Toxic", marker="H", c=sns.color_palette()[1])

U1, p = mannwhitneyu(restricted_prompted_toxic_Pr_vals_toxic_content, restricted_non_prompted_toxic_Pr_vals_toxic_content)

print("Toxic stimulus prompted/non-prompted", U1, p)


#print(mk.original_test(restricted_prompted_toxic_Pr_vals_non_toxic_content))
#print(mk.original_test(restricted_toxic_Pr_vals_toxic_content))

#from scipy.stats import mannwhitneyu
#U1, p = mannwhitneyu(restricted_toxic_Pr_vals_non_toxic_content, restricted_toxic_Pr_vals_toxic_content)
#print(U1, p)

# regression curve
plt.plot(restricted_prompted_toxic_x_fit_non_toxic, restricted_prompted_toxic_y_fit_non_toxic, linewidth=2.5, c=sns.color_palette()[0],
         marker="^",
        markevery=50,
         markersize=9,
         markerfacecolor="white",        # or match the line color
         markeredgecolor=sns.color_palette()[0],
         markeredgewidth=1.5,
         alpha=1,
         zorder=5)
plt.plot(restricted_non_prompted_toxic_x_fit_non_toxic, restricted_non_prompted_toxic_y_fit_non_toxic, linewidth=2.5, c=sns.color_palette()[0],
         marker="H",
        markevery=50,
         markersize=9,
         markerfacecolor="white",        # or match the line color
         markeredgecolor=sns.color_palette()[0],
         markeredgewidth=1.5,
         alpha=1,
         zorder=5)
#plt.plot(restricted_toxic_x_fit_non_toxic, restricted_toxic_y_fit_non_toxic, linewidth=2.5)

plt.plot(restricted_prompted_toxic_x_fit_toxic, restricted_prompted_toxic_y_fit_toxic, linewidth=2.5, 
         c=sns.color_palette()[1], marker="^",
         markevery=50,
         markersize=9,
         markerfacecolor="white",        # or match the line color
         markeredgecolor=sns.color_palette()[1],
         markeredgewidth=1.5,
         alpha=1,
         zorder=5)
plt.plot(restricted_non_prompted_toxic_x_fit_toxic, restricted_non_prompted_toxic_y_fit_toxic, linewidth=2.5, 
         c=sns.color_palette()[1], marker="H",
         markevery=50,
         markersize=9,
         markerfacecolor="white",        # or match the line color
         markeredgecolor=sns.color_palette()[1],
         markeredgewidth=1.5,
         alpha=1,
         zorder=10)
#plt.plot(restricted_toxic_x_fit_toxic, restricted_toxic_y_fit_toxic, linewidth=2.5)

# labels
plt.xlabel("$n_{\mathcal{S}}$", fontsize=16)
plt.ylabel(r"$\textrm{P}(\mathcal{R}_T|n_{\mathcal{S}})$", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

#plt.xlim(0, N_vals.max() + 10)
#plt.ylim(0, max(Pr_vals) * 1.1)
plt.legend(title="Stimulus", fontsize=12, title_fontsize=14, ncols=2)

handles, labels = ax.get_legend_handles_labels()

# Create a new figure for the legend
#fig_legend = plt.figure(figsize=(4, 1))   # adjust size as needed

#legend = fig_legend.legend(handles, labels, ncol=2, frameon=True, title="Agent/Stimulus")

#print(legend)
ax.legend().set_visible(False)
ax.get_legend().remove()

# Remove axes
#fig_legend.gca().set_axis_off()

plt.grid(linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(figs_dir, 'probability_of_prompted_non_prompted_agents_toxic_adoption_by_post_toxicity.png'), dpi=200)
plt.savefig(os.path.join(figs_dir, 'probability_of_prompted_non_prompted_agents_toxic_adoption_by_post_toxicity.pdf'), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig_legend, ax_legend = plt.subplots(figsize=(4, 2))

triangle = mlines.Line2D([], [], color="gray", marker="^", linestyle="None",
                         markersize=9, label="Prompted",
                         markerfacecolor="white", markeredgewidth=1.5)
hexagon  = mlines.Line2D([], [], color="gray", marker="h", linestyle="None",
                         markersize=9, label="Non-prompted",
                         markerfacecolor="white", markeredgewidth=1.5)

toxic_stimulus    = mpatches.Patch(color=sns.color_palette()[1], label="Toxic")
nontoxic_stimulus = mpatches.Patch(color=sns.color_palette()[0], label="Non-toxic")

ax_legend.legend(
    handles=[
        mlines.Line2D([], [], linestyle="none", label=" Agent "),
        triangle, hexagon,
        #mlines.Line2D([], [], linestyle="none", label=" "),
        mlines.Line2D([], [], linestyle="none", label=" Stimulus "),
        toxic_stimulus, nontoxic_stimulus,
    ],
    fontsize=17,
    handletextpad=0.5,
    framealpha=0.9,
    title="",
    title_fontsize=10,
    #loc="center",   
    ncol=2# center it in the blank figure
)

ax_legend.axis("off")   # hide the empty axes

plt.tight_layout()
plt.savefig(os.path.join(figs_dir, 'probability_of_prompted_non_prompted_agents_toxic_adoption_by_post_toxicity_legend.pdf'), dpi=200, bbox_inches="tight")


plt.show()

In [ ]:
data = []

for p in toxic_restricted_Pr_vals:
    data.append({
        "adoption_type": "Toxic", 
        "prob_adoption": p,
        "group": "All",
        "prompted": "All"
})

for p in non_toxic_restricted_Pr_vals:
    data.append({
        "adoption_type": "Non-Toxic",
        "prob_adoption": p,
        "group": "All",
        "prompted": "All"
})

for p in restricted_toxic_Pr_vals_non_toxic_content:
    data.append({
        "adoption_type": "Toxic",
        "prob_adoption": p,
        "group": "Non-Toxic", # exposure,
        "prompted": "All"
})

for p in restricted_non_toxic_Pr_vals_non_toxic_content:
    data.append({
        "adoption_type": "Non-Toxic",
        "prob_adoption": p,
        "group": "Non-Toxic",
        "prompted": "All"
})

for p in restricted_toxic_Pr_vals_toxic_content:
    data.append({
        "adoption_type": "Toxic",
        "prob_adoption": p,
        "group": "Toxic",
        "prompted": "All"
})
    
for p in restricted_non_toxic_Pr_vals_toxic_content:
    data.append({
        "adoption_type": "Non-toxic",
        "prob_adoption": p,
        "group": "Toxic",
        "prompted": "All"
})

for p in restricted_prompted_toxic_Pr_vals_toxic_content:
    data.append({
        "adoption_type": "Toxic",
        "prob_adoption": p,
        "group": "Toxic",
        "prompted": "Prompted"
})

for p in restricted_non_prompted_toxic_Pr_vals_toxic_content:
    data.append({
        "adoption_type": "Toxic",
        "prob_adoption": p,
        "group": "Toxic",
        "prompted": "Non-Prompted"
})


for p in restricted_prompted_toxic_Pr_vals_non_toxic_content:
    data.append({
        "adoption_type": "Toxic",
        "prob_adoption": p,
        "group": "Non-Toxic",
        "prompted": "Prompted"
})

for p in restricted_non_prompted_toxic_Pr_vals_non_toxic_content:
    data.append({
        "adoption_type": "Toxic",
        "prob_adoption": p,
        "group": "Non-Toxic",
        "prompted": "Non-Prompted"
})

df = pd.DataFrame(data)

In [ ]:
df = df.rename(columns={"adoption_type":"Adoption"})

In [ ]:
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
#mpl.rcParams['text.latex.preamble'] = [r'\usepackage{amsmath}'] #for \text command

In [ ]:
import ptitprince as pt

In [ ]:
# same thing with a single command: now x **must** be the categorical value

restricted_toxic_df = df[(df["Adoption"] == "Toxic") & (df["prompted"] == "All")]
dx = "group"
dy = "prob_adoption"
ort = "h"


U1, p = mannwhitneyu(restricted_toxic_df[restricted_toxic_df["group"] == "All"]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[restricted_toxic_df["group"] == "Toxic"]["prob_adoption"].values.tolist())
print("All with Toxic", U1, p)

U1, p = mannwhitneyu(restricted_toxic_df[restricted_toxic_df["group"] == "All"]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[restricted_toxic_df["group"] == "Non-Toxic"]["prob_adoption"].values.tolist())
print("All with Non-Toxic", U1, p)

U1, p = mannwhitneyu(restricted_toxic_df[restricted_toxic_df["group"] == "Toxic"]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[restricted_toxic_df["group"] == "Non-Toxic"]["prob_adoption"].values.tolist())
print("Toxic with Non-Toxic", U1, p)

groups = restricted_toxic_df["group"].unique()
#pal = "Set2"
sigma = 0.1
f, ax = plt.subplots(figsize=(6, 4))
colors = ['lightgray'] + sns.color_palette()[:2]

pt.RainCloud(
    x=dx, y=dy, data=restricted_toxic_df, hue="group", palette=colors, bw=sigma, width_viol=0.7, ax=ax, orient=ort, 
    box_showfliers = False, box_showmeans = True, #pointplot=True
    
    
)
ax.set_yticks(range(len(groups)))

groups[0] = "Both"
ax.set_yticklabels(groups, fontsize=15)
ax.set_ylabel("Stimulus", fontsize=18)
ax.set_xlabel(r"$\textrm{P}(\mathcal{R}_T|n_{\mathcal{S}})$ at $n_{\mathcal{S}} \le " + f"{treshold}$", fontsize=18)
plt.xticks(fontsize=14)

# plt.xlim(0.5, 1)

ax.grid(linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(os.path.join(figs_dir, "probability_of_toxic_adoption_by_toxicity_boxplot.png"), dpi=200)
plt.savefig(os.path.join(figs_dir, "probability_of_toxic_adoption_by_toxicity_boxplot.pdf"), dpi=200, bbox_inches="tight")

plt.show()

In [ ]:
restricted_toxic_df[restricted_toxic_df["group"] == "Toxic"]

In [ ]:
# same thing with a single command: now x **must** be the categorical value

restricted_toxic_df = df[(df["Adoption"] == "Toxic") & (df["group"] != "All")]

dx = "prompted"
dy = "prob_adoption"
ort = "h"

print("TOXIC STIMULI")
U1, p = mannwhitneyu(restricted_toxic_df[(restricted_toxic_df["group"] == "Toxic") & (restricted_toxic_df["prompted"] == "All")]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[(restricted_toxic_df["group"] == "Toxic") & (restricted_toxic_df["prompted"] == "Prompted")]["prob_adoption"].values.tolist())
print("All with Prompted", U1, p)

U1, p = mannwhitneyu(restricted_toxic_df[(restricted_toxic_df["group"] == "Toxic") & (restricted_toxic_df["prompted"] == "All")]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[(restricted_toxic_df["group"] == "Toxic") & (restricted_toxic_df["prompted"] == "Non-Prompted")]["prob_adoption"].values.tolist())
print("All with Non-Prompted", U1, p)

U1, p = mannwhitneyu(restricted_toxic_df[(restricted_toxic_df["group"] == "Toxic") & (restricted_toxic_df["prompted"] == "Prompted")]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[(restricted_toxic_df["group"] == "Toxic") & (restricted_toxic_df["prompted"] == "Non-Prompted")]["prob_adoption"].values.tolist())
print("Prompted with Non-Prompted", U1, p)


print("NON-TOXIC STIMULI")
U1, p = mannwhitneyu(restricted_toxic_df[(restricted_toxic_df["group"] == "Non-Toxic") & (restricted_toxic_df["prompted"] == "All")]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[(restricted_toxic_df["group"] == "Non-Toxic") & (restricted_toxic_df["prompted"] == "Prompted")]["prob_adoption"].values.tolist())
print("All with Prompted", U1, p)

U1, p = mannwhitneyu(restricted_toxic_df[(restricted_toxic_df["group"] == "Non-Toxic") & (restricted_toxic_df["prompted"] == "All")]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[(restricted_toxic_df["group"] == "Non-Toxic") & (restricted_toxic_df["prompted"] == "Non-Prompted")]["prob_adoption"].values.tolist())
print("All with Non-Prompted", U1, p)

U1, p = mannwhitneyu(restricted_toxic_df[(restricted_toxic_df["group"] == "Non-Toxic") & (restricted_toxic_df["prompted"] == "Prompted")]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[(restricted_toxic_df["group"] == "Non-Toxic") & (restricted_toxic_df["prompted"] == "Non-Prompted")]["prob_adoption"].values.tolist())
print("Prompted with Non-Prompted", U1, p)

groups = restricted_toxic_df[dx].unique()
#pal = "Set2"
sigma = 0.1

f, ax = plt.subplots(figsize=(6, 4))
colors = sns.color_palette()[:2]

pt.RainCloud(
    x=dx, y=dy, data=restricted_toxic_df, hue="group", palette=colors, bw=sigma, width_viol=0.7, ax=ax, orient=ort, 
    box_showfliers = False, box_showmeans = True, pointplot=True
    
    
)

for line in ax.get_lines():
    # pointplot connector lines tend to have >2 points
    if len(line.get_xdata()) > 2:
        line.set_markeredgecolor("black")
        line.set_markeredgewidth(1.5) 
        line.set_color(line.get_markerfacecolor())
        
ax.set_yticks(range(len(groups)))

#groups[0] = "All"
#ax.legend().set_title("Stimulus")
ax.set_yticklabels(groups, fontsize=15)
ax.set_ylabel("Agent", fontsize=18)
ax.set_xlabel(r"$\textrm{P}(\mathcal{R}_T|n_{\mathcal{S}})$ at $n_{\mathcal{S}} \le " + f"{treshold}$", fontsize=18)
plt.xticks(fontsize=14)

# plt.xlim(0.5, 1)

ax.grid(linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

handles, labels = ax.get_legend_handles_labels()

# Create a new figure for the legend
fig_legend = plt.figure(figsize=(4, 1))   # adjust size as needed

new_labels = ["Non-Toxic", "Toxic"]
legend = fig_legend.legend(handles[:2], new_labels, ncol=2, frameon=True, title="Stimulus")

ax.legend().set_visible(False)

# Remove axes
fig_legend.gca().set_axis_off()

# Save legend
fig_legend.savefig(os.path.join(figs_dir, "prompted_non_prompted_toxic_adoption_boxplot_legend.pdf"), dpi=300, bbox_inches="tight")

f.tight_layout()
f.savefig(os.path.join(figs_dir, "probability_of_prompted_non_prompted_toxic_adoption_boxplot.png"), dpi=200)
f.savefig(os.path.join(figs_dir, "probability_of_prompted_non_prompted_toxic_adoption_boxplot.pdf"), dpi=200, bbox_inches="tight")

plt.show()

In [ ]:
dx = "group"
dy = "prob_adoption"
ort = "h"
dhue = "Adoption"
groups = df["group"].unique()
#pal = "Set2"
sigma = 0.1
f, ax = plt.subplots(figsize=(10, 4))
#colors = #['lightgray'] + 
colors = sns.color_palette()[:2][::-1]


from matplotlib.collections import PolyCollection, PathCollection

ax = pt.RainCloud(
    x=dx,
    y=dy,
    hue=dhue,
    data=df,
    palette=colors,
    bw=sigma,
    #width_viol=1,
    ax=ax,
    orient=ort,
    alpha=0.65,
    dodge=True,
    width_box=0.4,
    box_showfliers = False, box_showmeans = True,
)


for coll in list(ax.collections):      # copy list to avoid mutating while iterating
    if isinstance(coll, PolyCollection):
        coll.remove()

    if isinstance(coll, PathCollection):
        coll.set_alpha(0.6) 

ax.set_yticks(range(len(groups)))
ax.set_yticklabels(groups, fontsize=15)
ax.set_ylabel("Original Post", fontsize=18)
# ax.set_xlabel(f"$Pr(A|n_e)$ at $n_e \le {treshold}$", fontsize=18)
ax.set_xlabel(f"$Pr(A|n_e)$ at $n_e \le {treshold}$", fontsize=18)

plt.xticks(fontsize=14)

plt.xlim(left=0.2)
ax.grid(linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(os.path.join("probability_of_toxic_non_toxic_adoption_by_toxicity_boxplot.png"), dpi=200)
plt.show()

In [ ]:
"""
Addressing the reviewer's comment on temporal construction of the prediction task.

(1) LANDMARK DESIGN: features from an agent's first K events, outcome in the
    NEXT M events. All predictor information strictly precedes the outcome,
    and every agent has exactly M opportunities -- which also removes the
    1-(1-p)^n artifact raised in an earlier comment.
(2) CHRONOLOGICAL train/test split, contrasted with random cross-validation.
(3) SURVIVAL: discrete-time hazard of the FIRST toxic response.

Requires: ev (agent, t_ms, event_idx, stim_toxic, resp_toxic, stim_tox_score),
          chirpers_df
"""

import numpy as np, pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

ev = ev.sort_values(["agent", "t_ms"]).reset_index(drop=True)
creator_of = chirpers_df.set_index("id")["user"].to_dict()

# ===========================================================================
# 1. LANDMARK DESIGN
# ===========================================================================
K, M = 20, 20          # feature window / outcome window, in events

n_ev = ev.groupby("agent")["event_idx"].max() + 1
elig = n_ev[n_ev >= K + M].index
print(f"agents with >= {K+M} events: {len(elig):,} of {n_ev.size:,}")

e = ev[ev.agent.isin(elig)]
fw = e[e.event_idx < K]                                   # feature window
ow = e[(e.event_idx >= K) & (e.event_idx < K + M)]        # outcome window

# At-risk set: agents that had NOT yet produced a toxic response by the index
# time. Otherwise the outcome is not "adoption" but "continued toxicity".
never_yet = fw.groupby("agent")["resp_toxic"].max()
at_risk = never_yet[never_yet == 0].index
print(f"at risk at index time (no toxic response in first {K} events): {len(at_risk):,}")

fwa = fw[fw.agent.isin(at_risk)]
X = fwa.groupby("agent").agg(
    n_tox_stim=("stim_toxic", "sum"),
    mean_stim_tox=("stim_tox_score", "mean"),
    max_stim_tox=("stim_tox_score", "max"),
    span_days=("t_ms", lambda s: (s.max() - s.min()) / 86_400_000),
    t0=("t_ms", "max"),
)
X["share_tox_stim"] = X.n_tox_stim / K
X["events_per_day"] = K / X.span_days.clip(lower=1e-6)

y = ow[ow.agent.isin(at_risk)].groupby("agent")["resp_toxic"].max()
X, y = X.align(y, join="inner", axis=0)
print(f"outcome prevalence: {y.mean():.4f}  (n = {len(y):,})")

# Every agent contributes exactly M opportunities, so the outcome cannot rise
# mechanically with exposure volume.

FEATS = ["n_tox_stim", "share_tox_stim", "mean_stim_tox", "max_stim_tox",
         "span_days", "events_per_day"]
BASELINE = ["span_days", "events_per_day"]      # activity only, no toxicity

# ===========================================================================
# 2. CHRONOLOGICAL SPLIT vs RANDOM CV
# ===========================================================================
def pipe():
    # SMOTE inside the pipeline: refit on training data only, never applied
    # to held-out data. imblearn's Pipeline (not sklearn's) skips resampling
    # at predict time.
    return Pipeline([("sc", StandardScaler()),
                     ("sm", SMOTE(random_state=0)),
                     ("lr", LogisticRegression(max_iter=2000))])

cut = X.t0.median()
tr, te = X.t0 <= cut, X.t0 > cut
print(f"\nchronological split at {pd.to_datetime(cut, unit='ms', utc=True)}: "
      f"train {tr.sum():,} / test {te.sum():,}")

print("\n--- chronological split (held-out later period) ---")
for name, cols in [("full", FEATS), ("activity only", BASELINE)]:
    m = pipe().fit(X.loc[tr, cols], y[tr])
    p = m.predict_proba(X.loc[te, cols])[:, 1]
    print(f"  {name:14s} AUC={roc_auc_score(y[te], p):.3f}  "
          f"PR-AUC={average_precision_score(y[te], p):.3f}")

print("\n--- random stratified 5-fold CV (for comparison only) ---")
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name, cols in [("full", FEATS), ("activity only", BASELINE)]:
    p = cross_val_predict(pipe(), X[cols], y, cv=cv, method="predict_proba")[:, 1]
    print(f"  {name:14s} AUC={roc_auc_score(y, p):.3f}  "
          f"PR-AUC={average_precision_score(y, p):.3f}")

# Report PR-AUC alongside AUC: with a rare outcome, AUC flatters a model that
# a precision-recall curve shows to be weak. The gap between the chronological
# and random estimates is itself worth reporting.

# Coefficients, for interpretability (fit on the training period only)
m = pipe().fit(X.loc[tr, FEATS], y[tr])
print("\nstandardised coefficients (training period):")
print(pd.Series(m.named_steps["lr"].coef_[0], index=FEATS).sort_values())

# ===========================================================================
# 3. SURVIVAL: discrete-time hazard of the FIRST toxic response
# ===========================================================================
# Each agent contributes events until its first toxic response, then exits.
# Agents that never adopt are censored at their last event.
ft = ev[ev.resp_toxic == 1].groupby("agent")["event_idx"].min()
ev["_ft"] = ev.agent.map(ft)
surv = ev[ev._ft.isna() | (ev.event_idx <= ev._ft)].copy()
surv["y"] = surv.resp_toxic

print(f"\n--- discrete-time hazard ---")
print(f"agent-event observations: {len(surv):,}   events (adoptions): "
      f"{int(surv.y.sum()):,}   censored agents: {surv._ft.isna().groupby(surv.agent).first().sum():,}")

surv["log_cum_prior"] = np.log1p(surv.groupby("agent")["stim_toxic"].cumsum()
                                - surv.stim_toxic)
surv["log_t"] = np.log1p(surv.event_idx)                 # baseline hazard
surv["month"] = surv.t.dt.tz_localize(None).dt.to_period("M").astype(str)

# No agent fixed effects here: adopters contribute a single event, so within-agent
# identification is unavailable and incidental-parameter bias would be severe.
# Standard practice for discrete-time hazards is a pooled model with clustered
# standard errors, with the baseline hazard carried by log_t.
D = pd.get_dummies(surv["month"], prefix="m", drop_first=True).astype(float)
Xs = sm.add_constant(pd.concat(
    [surv[["stim_toxic", "log_cum_prior", "log_t"]].astype(float).reset_index(drop=True),
     D.reset_index(drop=True)], axis=1))
cl = pd.factorize(surv.agent.map(lambda a: creator_of.get(a, f"NA_{a}")))[0]

mh = sm.Logit(surv["y"].astype(float).reset_index(drop=True), Xs).fit(
    disp=False, cov_type="cluster", cov_kwds={"groups": cl})
COLS = ["stim_toxic", "log_cum_prior", "log_t"]
print(pd.DataFrame({"HR_approx": np.exp(mh.params[COLS]), "se": mh.bse[COLS],
                    "p": mh.pvalues[COLS]}))

# log_cum_prior > 0 means accumulated PRIOR exposure raises the hazard of a
# first toxic response -- a strictly prospective statement, since the covariate
# is measured before each event and the agent exits at adoption.

In [ ]:
first_toxic_adoption_timestamp = {}

grouped_toxic_adoption_df = toxic_adoption_df.groupby("chirper")
for chirper, group in grouped_toxic_adoption_df:
    first_timestamp = group["exposure_date"].min()
    first_toxic_adoption_timestamp[chirper] = first_timestamp

In [ ]:
toxic_exposures_per_follower_before_its_first_toxic_adoption = {}

grouped_toxic_exposure_df = toxic_exposure_df.groupby("chirper")
for chirper, group in grouped_toxic_exposure_df:
    if chirper not in first_toxic_adoption_timestamp:
        before_group = group
    else:
        before_group = group[group["exposure_date"] < first_toxic_adoption_timestamp[chirper]]
    toxic_exposures_per_follower_before_its_first_toxic_adoption[chirper] = len(group)

In [ ]:
will_adopt = [1 if k in first_toxic_adoption_timestamp else 0 for k in toxic_exposures_per_follower_before_its_first_toxic_adoption]

In [ ]:
logistic_regression_df = pd.DataFrame({"n_exposures": list(toxic_exposures_per_follower_before_its_first_toxic_adoption.values()), 
                          "adopt": will_adopt})#, "n_adoptions": [toxic_adoptions_per_follower[k] if k in toxic_adoptions_per_follower else 0 for k in exposures_per_follower]})

In [ ]:
X, y = logistic_regression_df["n_exposures"].to_numpy(), logistic_regression_df["adopt"].to_numpy() 

In [ ]:
values, counts = np.unique(y, return_counts=True)

In [ ]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)

balanced_X, balanced_y = sm.fit_resample(X.reshape(-1, 1), y.reshape(-1, 1))

In [ ]:
values, counts = np.unique(balanced_y, return_counts=True)

In [ ]:
values, counts/np.sum(counts)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    balanced_X, balanced_y, test_size=0.20, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(max_depth=2, random_state=0)
clf.fit(X_train, y_train)

In [ ]:
random_forest_prediction = clf.predict(X_test)

In [ ]:
from sklearn.metrics import f1_score

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
def compute_cross_validation(clf, X, y, metrics):
    
    clf_scores = {}
    for m in metrics:
        m_scores = cross_val_score(clf, X, y, cv=kf, scoring=m)
        m_scores_mean, m_scores_std = round(m_scores.mean(), 4), m_scores.std()
        print(f"{m} scores for each fold: {m_scores}")
        print(f"Mean {m} across folds: {m_scores_mean}, Std: {m_scores_std}")
    
        clf_scores[m] = [m_scores_mean, m_scores_std]

    return clf_scores

In [ ]:
classification_metrics = ["accuracy", "recall", "precision", "f1"]

In [ ]:
log_reg_clf = LogisticRegression(random_state=42, class_weight='balanced')

log_reg_scores = compute_cross_validation(log_reg_clf, balanced_X, balanced_y, classification_metrics)

In [ ]:
# Initialize the Random Forest Classifier
rf_clf = RandomForestClassifier(random_state=42)#, class_weight='balanced')

# Define the cross-validation strategy (e.g., 5-fold)

# Perform cross-validation and get scores
rf_scores = compute_cross_validation(clf, balanced_X, balanced_y, classification_metrics)

In [ ]:
import xgboost as xgb

In [ ]:
# Initialize and train the XGBoost classifier
xgb_clf = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss')

xgb_scores = compute_cross_validation(xgb_clf, balanced_X, balanced_y, classification_metrics)


In [ ]:
from sklearn.neural_network import MLPClassifier

TUNE = True
if TUNE:
    max_n = 5
    best_median_scores = -1
    
    for i in range(1, max_n):
        for j in range(1, max_n):
            mlp_clf = MLPClassifier(alpha=1e-5,
                                hidden_layer_sizes=(i, j), 
                                    random_state=42)
            mlp_scores = compute_cross_validation(mlp_clf, balanced_X, balanced_y, classification_metrics)
    
            median_scores = np.median(list(mlp_scores.values()))
            if median_scores > best_median_scores:
                best_median_scores = median_scores
                best_i = i
                best_j = j

In [ ]:
best_i = 1
best_j = 2

In [ ]:
mlp_clf = MLPClassifier(alpha=1e-5,
                            hidden_layer_sizes=(best_i, best_j), 
                                random_state=42)
mlp_scores = compute_cross_validation(mlp_clf, balanced_X, balanced_y, classification_metrics)

In [ ]:
to_write = '''
\\begin{table}[!ht]
    \centering
    
    \caption{Binary classification performance for predicting chirper toxic responses based solely on the number of toxic stimuli encountered.}
    \label{tab:classification}
    \\resizebox{\linewidth}{!}{
    \\begin{tabular}{ccccc}
        \\toprule
        Classifier & Accuracy & Recall & Precision & F1-Weighted \\\\
        
        \midrule
'''

classifiers = ["Logistic Regression", "Random Forest", "XGBoost", "MultiLayerPerceptron"]
classifier_scores = [log_reg_scores, rf_scores, xgb_scores, mlp_scores]

for i, clf_scores in enumerate(classifier_scores):
    to_write += classifiers[i]
    
    for metric, score in clf_scores.items():
        to_write +=  " & $" + str(round(score[0], 3)) + " \pm " + str(round(score[1], 3)) + "$ \n"
    to_write += " \\\\ \n"
        
to_write += '''\\bottomrule
    \end{tabular}
    }
\end{table}
'''

In [ ]:
print(to_write)

# Grouping by Post toxicity (medium/high)

In [ ]:
import json

In [ ]:
(toxic_exposure_df == exposure_df[exposure_df["post_toxicity"] >= toxicity_threshold]).all()

In [ ]:
# high_toxicity_threshold = np.percentile(toxicity_values, 95)
high_toxicity_threshold = 0.75

In [ ]:
CREATE = False

other_paths = ["count_chirper_exposition_medium_toxic_content_dict.json",
     "count_chirper_exposition_high_toxic_content_dict.json",
     "count_chirper_toxic_adoption_medium_toxic_content_dict.json",
     "count_chirper_toxic_adoption_high_toxic_content_dict.json",
     "count_chirper_non_toxic_adoption_medium_toxic_content_dict.json",
     "count_chirper_non_toxic_adoption_high_toxic_content_dict.json"]
if CREATE:

    grouped_toxic_exposure_df = toxic_exposure_df.groupby("chirper")
    
    # high_toxicity_threshold = np.percentile(toxicity_values, 95)
    high_toxicity_threshold = 0.75
    
    count_chirper_exposition_medium_toxic_content = defaultdict()
    count_chirper_exposition_high_toxic_content = defaultdict()
    
    count_chirper_toxic_adoption_medium_toxic_content = defaultdict()
    count_chirper_toxic_adoption_high_toxic_content = defaultdict()
    
    count_chirper_non_toxic_adoption_medium_toxic_content = defaultdict()
    count_chirper_non_toxic_adoption_high_toxic_content = defaultdict()
    
    for chirper, group in tqdm(grouped_toxic_exposure_df, total=len(grouped_toxic_exposure_df)):
    
        chirper_exposed_medium_toxic_content = group[group["post_toxicity"] < high_toxicity_threshold]["chirp"].unique()
        chirper_exposed_high_toxic_content = group[group["post_toxicity"] >= high_toxicity_threshold]["chirp"].unique()
    
        # Toxic adoption
        chirper_toxic_adoption_df = toxic_adoption_df[toxic_adoption_df["chirper"] == chirper]
        chirper_toxic_adopted_medium_toxic_content = chirper_toxic_adoption_df[chirper_toxic_adoption_df["chirp"].isin(chirper_exposed_medium_toxic_content)]["chirp"].values.tolist()
        chirper_toxic_adopted_high_toxic_content = chirper_toxic_adoption_df[chirper_toxic_adoption_df["chirp"].isin(chirper_exposed_high_toxic_content)]["chirp"].values.tolist()
    
        # Non-toxic adoption
        chirper_non_toxic_adoption_df = non_toxic_adoption_df[non_toxic_adoption_df["chirper"] == chirper]
        chirper_non_toxic_adopted_medium_toxic_content = chirper_non_toxic_adoption_df[chirper_non_toxic_adoption_df["chirp"].isin(chirper_exposed_medium_toxic_content)]["chirp"].values.tolist()
        chirper_non_toxic_adopted_high_toxic_content = chirper_non_toxic_adoption_df[chirper_non_toxic_adoption_df["chirp"].isin(chirper_exposed_high_toxic_content)]["chirp"].values.tolist()
    
        count_chirper_exposition_medium_toxic_content[chirper] = len(chirper_exposed_medium_toxic_content)
        count_chirper_exposition_high_toxic_content[chirper] = len(chirper_exposed_high_toxic_content)
    
        count_chirper_toxic_adoption_medium_toxic_content[chirper] = len(chirper_toxic_adopted_medium_toxic_content)
        count_chirper_toxic_adoption_high_toxic_content[chirper] = len(chirper_toxic_adopted_high_toxic_content)
    
        count_chirper_non_toxic_adoption_medium_toxic_content[chirper] = len(chirper_non_toxic_adopted_medium_toxic_content)
        count_chirper_non_toxic_adoption_high_toxic_content[chirper] = len(chirper_non_toxic_adopted_high_toxic_content)


    other_counters = [count_chirper_exposition_medium_toxic_content, 
                    count_chirper_exposition_high_toxic_content,
                    count_chirper_toxic_adoption_medium_toxic_content,
                    count_chirper_toxic_adoption_high_toxic_content,
                    count_chirper_non_toxic_adoption_medium_toxic_content,
                    count_chirper_non_toxic_adoption_high_toxic_content]
        
    for i in range(len(other_paths)):
        with open(os.path.join(root_dir, "processed", other_paths[i]), "w") as f:
            json.dump(other_counters[i], f)
else:

    with open(os.path.join(root_dir, "processed", other_paths[0]), "r") as f:
        count_chirper_exposition_medium_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", other_paths[1]), "r") as f:
        count_chirper_exposition_high_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", other_paths[2]), "r") as f:
        count_chirper_toxic_adoption_medium_toxic_content = json.load(f)
        
    with open(os.path.join(root_dir, "processed", other_paths[3]), "r") as f:
        count_chirper_toxic_adoption_high_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", other_paths[4]), "r") as f:
        count_chirper_non_toxic_adoption_medium_toxic_content = json.load(f)

    with open(os.path.join(root_dir, "processed", other_paths[5]), "r") as f:
        count_chirper_non_toxic_adoption_high_toxic_content = json.load(f)

In [ ]:
users_by_exposure_medium_toxic_content = defaultdict(list)
users_by_exposure_high_toxic_content = defaultdict(list)

for user, n_e in count_chirper_exposition_medium_toxic_content.items():
    users_by_exposure_medium_toxic_content[n_e].append(user)

for user, n_e in count_chirper_exposition_high_toxic_content.items():
    users_by_exposure_high_toxic_content[n_e].append(user)

# 2. Compute P(A* | n_e = N)
prob_toxic_adoption_given_exposure_medium_toxic_content = {}
prob_toxic_adoption_given_exposure_high_toxic_content = {}

# 3. Compute P(A | n_e = N)
prob_non_toxic_adoption_given_exposure_medium_toxic_content = {}
prob_non_toxic_adoption_given_exposure_high_toxic_content = {}

minimum_toxic_adoption = np.percentile(toxic_adoptions, 25) # 0 (?)
minimum_non_toxic_adoption = np.percentile(non_toxic_adoptions, 25) # 0 (?)

for N, users in users_by_exposure_medium_toxic_content.items():
    # users with exposure N to non-toxic content who adopted
    toxic_adopted = sum(1 for u in users if count_chirper_toxic_adoption_medium_toxic_content[u] > minimum_toxic_adoption)
    non_toxic_adopted = sum(1 for u in users if count_chirper_non_toxic_adoption_medium_toxic_content[u] > minimum_non_toxic_adoption)
    
    total = len(users)

    prob_toxic_adoption_given_exposure_medium_toxic_content[N] = toxic_adopted / total if total > 0 else 0.0
    prob_non_toxic_adoption_given_exposure_medium_toxic_content[N] = non_toxic_adopted / total if total > 0 else 0.0

for N, users in users_by_exposure_high_toxic_content.items():
    # users with exposure N to toxic content who adopted
    toxic_adopted = sum(1 for u in users if count_chirper_toxic_adoption_high_toxic_content[u] > minimum_toxic_adoption)
    non_toxic_adopted = sum(1 for u in users if count_chirper_non_toxic_adoption_high_toxic_content[u] > minimum_non_toxic_adoption)
    
    total = len(users)

    prob_toxic_adoption_given_exposure_high_toxic_content[N] = toxic_adopted / total if total > 0 else 0.0
    prob_non_toxic_adoption_given_exposure_high_toxic_content[N] = non_toxic_adopted / total if total > 0 else 0.0

In [ ]:
toxic_N_vals_medium_toxic_content = np.array(list(prob_toxic_adoption_given_exposure_medium_toxic_content.keys()))
toxic_Pr_vals_medium_toxic_content = np.array(list(prob_toxic_adoption_given_exposure_medium_toxic_content.values()))

non_toxic_N_vals_medium_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_medium_toxic_content.keys()))
non_toxic_Pr_vals_medium_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_medium_toxic_content.values()))

toxic_N_vals_high_toxic_content = np.array(list(prob_toxic_adoption_given_exposure_high_toxic_content.keys()))
toxic_Pr_vals_high_toxic_content = np.array(list(prob_toxic_adoption_given_exposure_high_toxic_content.values()))

non_toxic_N_vals_high_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_high_toxic_content.keys()))
non_toxic_Pr_vals_high_toxic_content = np.array(list(prob_non_toxic_adoption_given_exposure_high_toxic_content.values()))

In [ ]:
from scipy.optimize import curve_fit

def get_x_y_fit(N_vals, Pr_vals):

    def log_func(x, a, b):
        return a * np.log(x) + b
    
    # Avoid log(0) → only fit points where N > 0
    fit_mask = N_vals > 0
    
    popt, _ = curve_fit(log_func, N_vals[fit_mask], Pr_vals[fit_mask])
    a, b = popt
    
    # smooth curve for plotting
    x_fit = np.linspace(1, N_vals.max(), 500)
    y_fit = np.clip(log_func(x_fit, a, b), 0, 1)

    return x_fit, y_fit

In [ ]:
toxic_x_fit_medium_toxic, toxic_y_fit_medium_toxic = get_x_y_fit(toxic_N_vals_medium_toxic_content, toxic_Pr_vals_medium_toxic_content)

non_toxic_x_fit_medium_toxic, non_toxic_y_fit_medium_toxic = get_x_y_fit(non_toxic_N_vals_medium_toxic_content, non_toxic_Pr_vals_medium_toxic_content)

toxic_x_fit_high_toxic, toxic_y_fit_high_toxic = get_x_y_fit(toxic_N_vals_high_toxic_content, toxic_Pr_vals_high_toxic_content)

non_toxic_x_fit_high_toxic, non_toxic_y_fit_high_toxic = get_x_y_fit(non_toxic_N_vals_high_toxic_content, non_toxic_Pr_vals_high_toxic_content)

In [ ]:
plt.figure(figsize=(5,4))

# scatter
plt.scatter(toxic_N_vals_toxic_content, toxic_Pr_vals_toxic_content, alpha=0.7, label="All")
plt.scatter(toxic_N_vals_medium_toxic_content, toxic_Pr_vals_medium_toxic_content, alpha=0.7, label="Medium-Toxic")
plt.scatter(toxic_N_vals_high_toxic_content, toxic_Pr_vals_high_toxic_content, alpha=0.7, label="High-Toxic")

# regression curve
plt.plot(toxic_x_fit_medium_toxic, toxic_y_fit_medium_toxic, linewidth=2.5)
plt.plot(toxic_x_fit_high_toxic, toxic_y_fit_high_toxic, linewidth=2.5)

# labels
plt.xlabel("Number of Exposures $n_e$", fontsize=12)
plt.ylabel("Probability of Adoption", fontsize=12)

#plt.xlim(0, N_vals.max() + 10)
#plt.ylim(0, max(Pr_vals) * 1.1)
plt.legend(title="Original Post")
plt.tight_layout()
plt.show()

In [ ]:
treshold = 50

toxic_restricted_medium_high_N_vals = toxic_N_vals_toxic_content[toxic_N_vals_toxic_content <= treshold]
toxic_restricted_medium_high_Pr_vals = toxic_Pr_vals_toxic_content[toxic_N_vals_toxic_content <= treshold]

restricted_toxic_N_vals_medium_toxic_content = toxic_N_vals_medium_toxic_content[toxic_N_vals_medium_toxic_content <= treshold]
restricted_toxic_Pr_vals_medium_toxic_content = toxic_Pr_vals_medium_toxic_content[toxic_N_vals_medium_toxic_content <= treshold]

non_toxic_restricted_medium_high_N_vals = non_toxic_N_vals_toxic_content[toxic_N_vals_toxic_content <= treshold]
non_toxic_restricted_medium_high_Pr_vals = non_toxic_Pr_vals_toxic_content[toxic_N_vals_toxic_content <= treshold]

restricted_non_toxic_N_vals_medium_toxic_content = non_toxic_N_vals_medium_toxic_content[non_toxic_N_vals_medium_toxic_content <= treshold]
restricted_non_toxic_Pr_vals_medium_toxic_content = non_toxic_Pr_vals_medium_toxic_content[non_toxic_N_vals_medium_toxic_content <= treshold]

restricted_toxic_N_vals_high_toxic_content = toxic_N_vals_high_toxic_content[toxic_N_vals_high_toxic_content <= treshold]
restricted_toxic_Pr_vals_high_toxic_content = toxic_Pr_vals_high_toxic_content[toxic_N_vals_high_toxic_content <= treshold]

restricted_non_toxic_N_vals_high_toxic_content = non_toxic_N_vals_high_toxic_content[non_toxic_N_vals_high_toxic_content <= treshold]
restricted_non_toxic_Pr_vals_high_toxic_content = non_toxic_Pr_vals_high_toxic_content[non_toxic_N_vals_high_toxic_content <= treshold]

In [ ]:
restricted_toxic_x_fit_medium_toxic, restricted_toxic_y_fit_medium_toxic = get_x_y_fit(restricted_toxic_N_vals_medium_toxic_content, restricted_toxic_Pr_vals_medium_toxic_content)

restricted_non_toxic_x_fit_medium_toxic, restricted_non_toxic_y_fit_medium_toxic = get_x_y_fit(restricted_non_toxic_N_vals_medium_toxic_content, restricted_non_toxic_Pr_vals_medium_toxic_content)

restricted_toxic_x_fit_high_toxic, restricted_toxic_y_fit_high_toxic = get_x_y_fit(restricted_toxic_N_vals_high_toxic_content, restricted_toxic_Pr_vals_high_toxic_content)

restricted_non_toxic_x_fit_high_toxic, restricted_non_toxic_y_fit_high_toxic = get_x_y_fit(restricted_non_toxic_N_vals_high_toxic_content, restricted_non_toxic_Pr_vals_high_toxic_content)

In [ ]:
plt.figure(figsize=(5,4))
sns.set_style("whitegrid")
# scatter

#plt.scatter(toxic_restricted_medium_high_N_vals, toxic_restricted_medium_high_Pr_vals, alpha=0.7, label="All")
plt.scatter(restricted_toxic_N_vals_medium_toxic_content, restricted_toxic_Pr_vals_medium_toxic_content, alpha=0.7, label="Medium")
plt.scatter(restricted_toxic_N_vals_high_toxic_content, restricted_toxic_Pr_vals_high_toxic_content, alpha=0.7, label="High")

# regression curve
plt.plot(restricted_toxic_x_fit_medium_toxic, restricted_toxic_y_fit_medium_toxic, linewidth=2.5)
plt.plot(restricted_toxic_x_fit_high_toxic, restricted_toxic_y_fit_high_toxic, linewidth=2.5)

print(mk.original_test(restricted_toxic_Pr_vals_medium_toxic_content))
print(mk.original_test(restricted_toxic_Pr_vals_high_toxic_content))

from scipy.stats import mannwhitneyu
U1, p = mannwhitneyu(restricted_toxic_Pr_vals_medium_toxic_content, restricted_toxic_Pr_vals_high_toxic_content)
print(U1, p)

# labels
plt.xlabel("$n_{\mathcal{S}}$", fontsize=16)
plt.ylabel(r"$\textrm{P}(\mathcal{R}_T|n_\mathcal{S})$", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

#plt.xlim(0, N_vals.max() + 10)
#plt.ylim(0, max(Pr_vals) * 1.1)
plt.legend(title="Stimulus Toxicity", fontsize=12, title_fontsize=14, ncols=2)
plt.grid(linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(figs_dir, 'probability_of_toxic_adoption_by_post_toxicity_(medium-high).pdf'), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(5,4))
sns.set_style("whitegrid")
# scatter
plt.scatter(restricted_non_toxic_N_vals_medium_toxic_content, restricted_non_toxic_Pr_vals_medium_toxic_content, alpha=0.7, label="Medium")
plt.scatter(restricted_non_toxic_N_vals_high_toxic_content, restricted_non_toxic_Pr_vals_high_toxic_content, alpha=0.7, label="High")

# regression curve
plt.plot(restricted_non_toxic_x_fit_medium_toxic, restricted_non_toxic_y_fit_medium_toxic, linewidth=2.5)
plt.plot(restricted_non_toxic_x_fit_high_toxic, restricted_non_toxic_y_fit_high_toxic, linewidth=2.5)

# labels
plt.xlabel("$n_e$", fontsize=16)
plt.ylabel("$Pr(\\tilde{A}|n_e)$", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

#plt.xlim(0, N_vals.max() + 10)
#plt.ylim(0, max(Pr_vals) * 1.1)
plt.legend(title="Original Post Toxicity", fontsize=12, title_fontsize=14, ncols=2)
plt.grid(linestyle='--')
plt.tight_layout()
# plt.savefig(os.path.join(figs_dir, 'probability_of_non_toxic_adoption_by_post_toxicity_(medium-high).png'), dpi=200)
plt.show()

In [ ]:
min(toxic_restricted_medium_high_Pr_vals)

In [ ]:
min(restricted_toxic_Pr_vals_medium_toxic_content)

In [ ]:
data = []

for p in toxic_restricted_medium_high_Pr_vals:
    data.append({
        "adoption_type": "Toxic", 
        "prob_adoption": p,
        "group": "All"
})

for p in non_toxic_restricted_medium_high_Pr_vals:
    data.append({
        "adoption_type": "Non-Toxic",
        "prob_adoption": p,
        "group": "All"
})

for p in restricted_toxic_Pr_vals_medium_toxic_content:
    data.append({
        "adoption_type": "Toxic",
        "prob_adoption": p,
        "group": "Medium"
})

for p in restricted_non_toxic_Pr_vals_medium_toxic_content:
    data.append({
        "adoption_type": "Non-Toxic",
        "prob_adoption": p,
        "group": "Medium"
})

for p in restricted_toxic_Pr_vals_high_toxic_content:
    data.append({
        "adoption_type": "Toxic",
        "prob_adoption": p,
        "group": "High"
})
    
for p in restricted_non_toxic_Pr_vals_high_toxic_content:
    data.append({
        "adoption_type": "Non-Toxic",
        "prob_adoption": p,
        "group": "High"
})

df = pd.DataFrame(data)

In [ ]:
# same thing with a single command: now x **must** be the categorical value

restricted_toxic_df = df[df["Adoption"] == "Toxic"]
dx = "group"
dy = "prob_adoption"
ort = "h"

U1, p = mannwhitneyu(restricted_toxic_df[restricted_toxic_df["group"] == "All"]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[restricted_toxic_df["group"] == "Medium"]["prob_adoption"].values.tolist())
print("All with Medium", U1, p)

U1, p = mannwhitneyu(restricted_toxic_df[restricted_toxic_df["group"] == "All"]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[restricted_toxic_df["group"] == "High"]["prob_adoption"].values.tolist())
print("All with High", U1, p)

U1, p = mannwhitneyu(restricted_toxic_df[restricted_toxic_df["group"] == "Medium"]["prob_adoption"].values.tolist(), 
                     restricted_toxic_df[restricted_toxic_df["group"] == "High"]["prob_adoption"].values.tolist())
print("Medium with High", U1, p)

groups = restricted_toxic_df["group"].unique()
#pal = "Set2"
sigma = 0.1
f, ax = plt.subplots(figsize=(6, 4))
colors = ['lightgray'] + sns.color_palette()[:2]

pt.RainCloud(
    x=dx, y=dy, data=restricted_toxic_df, hue="group", palette=colors, bw=sigma, width_viol=0.7, ax=ax, orient=ort, 
    box_showfliers = False, box_showmeans = True,
    
    
)
ax.set_yticks(range(len(groups)))
ax.set_yticklabels(["Bo"] + list(groups[1:]), fontsize=15)
ax.set_ylabel("Stimulus Toxicity", fontsize=18)
ax.set_xlabel(r"$\textrm{P}(R_T|n_S)$ at $n_S \le " + f"{treshold}$", fontsize=18)
plt.xticks(fontsize=14)

ax.grid(linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(os.path.join(figs_dir, "probability_of_toxic_adoption_by_toxicity_boxplot_(medium-high).pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
restricted_toxic_df = df[df["Adoption"] == "Non-Toxic"]
dx = "group"
dy = "prob_adoption"
ort = "h"

groups = restricted_toxic_df["group"].unique()
#pal = "Set2"
sigma = 0.1
f, ax = plt.subplots(figsize=(6, 4))
colors = ['lightgray'] + sns.color_palette()[:2]

pt.RainCloud(
    x=dx, y=dy, data=restricted_toxic_df, hue="group", palette=colors, bw=sigma, width_viol=0.7, ax=ax, orient=ort, 
    box_showfliers = False, box_showmeans = True,
    
    
)
ax.set_yticks(range(len(groups)))
ax.set_yticklabels(groups, fontsize=15)
ax.set_ylabel("Original Post Toxicity", fontsize=18)
ax.set_xlabel(r"$Pr(\tilde{A}|n_e)$ at $n_e \le " + f"{treshold}$", fontsize=18)
plt.xticks(fontsize=14)

ax.grid(linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
# plt.savefig("probability_of_non_toxic_adoption_by_toxicity_boxplot_(medium-high).png", dpi=200)
plt.show()

In [ ]:
dx = "group"
dy = "prob_adoption"
ort = "h"
dhue = "Adoption"
groups = df["group"].unique()
#pal = "Set2"
sigma = 0.1
f, ax = plt.subplots(figsize=(10, 4))
#colors = #['lightgray'] + 
colors = sns.color_palette()[:2][::-1]

ax = pt.RainCloud(
    x=dx,
    y=dy,
    hue=dhue,
    data=df,
    palette=colors,
    bw=sigma,
    width_viol=1,
    ax=ax,
    orient=ort,
    alpha=0.65,
    dodge=True,
    box_showfliers = False, box_showmeans = True,
)

ax.set_yticks(range(len(groups)))
ax.set_yticklabels(groups, fontsize=15)
ax.set_ylabel("Original Post Toxicity", fontsize=18)
# ax.set_xlabel(f"$Pr(A|n_e)$ at $n_e \le {treshold}$", fontsize=18)
ax.set_xlabel(f"$Pr(A|n_e)$ at $n_e \le {treshold}$", fontsize=18)

plt.xticks(fontsize=14)


ax.grid(linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
# plt.savefig("probability_of_toxic_non_toxic_adoption_by_toxicity_boxplot_(medium-high).png", dpi=200)
plt.show()

# IAR/SAR on Toxic Adoptions/Exposure

In [ ]:
iar = []
sar = []

prompted_iar = []
prompted_sar = []

non_prompted_iar = []
non_prompted_sar = []

iar_dict = {}
sar_dict = {}

found_prompted = False
found_non_prompted = False

for k in exposures_per_follower:

    # r is the number of (unique) toxic responses to toxic stimuli
    r = unique_count_chirper_toxic_adoption_toxic_content[k]

    # s is the number of toxic stimuli
    s = count_chirper_exposition_toxic_content[k] 

    # s could be the number of toxic responses
    #s = unique_count_chirper_toxic_adoption_non_toxic_content[k] + unique_count_chirper_toxic_adoption_toxic_content[k]

    if r == s == 0:
        continue

    # r_spont is the number of (unique) toxic responses to non-toxic stimuli
    r_spont = unique_count_chirper_toxic_adoption_non_toxic_content[k]

    # s_tox is the number of toxic responses
    s_tox = unique_count_chirper_toxic_adoption_non_toxic_content[k] + unique_count_chirper_toxic_adoption_toxic_content[k]

    # s_tox could be the number of non-toxic stimuli
    #s_tox = count_chirper_exposition_non_toxic_content[k]
        
    if r_spont == s_tox == 0:
        continue

    iar_score = r/s
    sar_score = r_spont/s_tox
    
    iar.append(iar_score)

    iar_dict[k] = iar_score

    sar.append(sar_score)

    sar_dict[k] = sar_score

    if k in agents_prompted_to_be_toxic and iar_score == 0:
        prompted_iar.append(iar_score)
        prompted_sar.append(sar_score)
        if not found_prompted:
            print("PROMPTED")
            print("number of (unique) toxic responses to toxic stimuli", unique_count_chirper_toxic_adoption_toxic_content[k])
            print("number of toxic responses", unique_count_chirper_toxic_adoption_non_toxic_content[k] + unique_count_chirper_toxic_adoption_toxic_content[k])
            print("number of (unique) toxic responses to non-toxic stimuli", unique_count_chirper_toxic_adoption_non_toxic_content[k])
            print("number of toxic stimuli", count_chirper_exposition_toxic_content[k])
            print("IAR", iar_score)
            print("SAR", sar_score)
            print(k)
        
            prompted_k = k
            print("****")
            found_prompted = True

    else:
        non_prompted_iar.append(iar_score)
        non_prompted_sar.append(sar_score)
        
        if not found_non_prompted and iar_score == 0:
            print("NON-PROMPTED")
            print("number of (unique) toxic responses to toxic stimuli", unique_count_chirper_toxic_adoption_toxic_content[k])
            print("number of toxic responses", unique_count_chirper_toxic_adoption_non_toxic_content[k] + unique_count_chirper_toxic_adoption_toxic_content[k])
            print("number of (unique) toxic responses to non-toxic stimuli", unique_count_chirper_toxic_adoption_non_toxic_content[k])
            print("number of toxic stimuli", count_chirper_exposition_toxic_content[k])
            print("IAR", iar_score)
            print("SAR", sar_score)
            print(k)
            non_prompted_k = k
            print("****")
            found_non_prompted = True

    # if found_prompted and found_non_prompted:
    #    break

In [ ]:
unique_count_chirper_toxic_adoption_non_toxic_content[non_prompted_k] + unique_count_chirper_toxic_adoption_toxic_content[non_prompted_k]

In [ ]:
print("***")

In [ ]:
unique_count_chirper_toxic_adoption_toxic_content[prompted_k]/count_chirper_exposition_toxic_content[prompted_k]

In [ ]:
unique_count_chirper_toxic_adoption_non_toxic_content[prompted_k]/count_chirper_exposition_non_toxic_content[prompted_k]

In [ ]:
0.25/0.1

In [ ]:
8/80

In [ ]:
iar_sar_df = pd.DataFrame({"score": iar + sar, "type": ["irr"]*len(iar) + ["srr"]*len(sar)})

In [ ]:
prompted_non_prompted_iar_sar_df = pd.DataFrame({"score": prompted_iar + prompted_sar + non_prompted_iar + non_prompted_sar, 
                                                 "type": ["irr"]*len(prompted_iar) + ["srr"]*len(prompted_sar) + ["irr"]*len(non_prompted_iar) + ["srr"]*len(non_prompted_sar),
                                                "agent": ["prompted"] * (len(prompted_iar) + len(prompted_sar)) + ["non-prompted"] * (len(non_prompted_iar) + len(non_prompted_sar))})

In [ ]:
fig, ax = plt.subplots()
# plt.boxplot([iar, sar], showfliers=True)

pt.RainCloud(y="score", data=prompted_non_prompted_iar_sar_df, hue="type", x="type", point_size=0, box_showmeans=True, width_viol=0.5)

colors = sns.color_palette()[:2]

for i, patch in enumerate(ax.patches):
    patch.set_edgecolor(colors[i%2])
    patch.set_linewidth(2)

rain_elements = {
    'whiskers': ax.lines,
    'caps': ax.lines,
    'medians': ax.lines,
    'means': ax.lines,
}

for element in rain_elements:
    for i, line in enumerate(rain_elements[element]):
        if i < len(rain_elements[element])//2 -1:
            line.set_color(colors[0])
        else:
            line.set_color(colors[1])
        line.set_linewidth(2)

plt.yticks(fontsize=16)
plt.xticks(fontsize=16)

ax.set_xticklabels(labels=["$\\textrm{ITRR}$", "$\\textrm{STRR}$"])

plt.ylabel("")#$\\textrm{Score}$")
plt.xlabel("")

plt.grid(linestyle='--')

plt.tight_layout()
plt.savefig(os.path.join(figs_dir, "iar_sar_distributions.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots()
# plt.boxplot([iar, sar], showfliers=True)

pt.RainCloud(y="score", data=prompted_non_prompted_iar_sar_df, hue="agent", x="type", point_size=0, box_showmeans=True, width_viol=0.5)

colors = sns.color_palette()[:2]

for i, patch in enumerate(ax.patches):
    patch.set_edgecolor(colors[i%2])
    patch.set_linewidth(2)

rain_elements = {
    'whiskers': ax.lines,
    'caps': ax.lines,
    'medians': ax.lines,
    'means': ax.lines,
}

for element in rain_elements:
    for i, line in enumerate(rain_elements[element]):
        if i < len(rain_elements[element])//2 -1:
            line.set_color(colors[0])
        else:
            line.set_color(colors[1])
        line.set_linewidth(2)

plt.yticks(fontsize=16)
plt.xticks(fontsize=16)

ax.set_xticklabels(labels=["$\\textrm{ITRR}$", "$\\textrm{STRR}$"])

plt.ylabel("")#$\\textrm{Score}$")
plt.xlabel("")

plt.grid(linestyle='--')

plt.tight_layout()
plt.savefig(os.path.join(figs_dir, "prompted_non-prompted_iar_sar_distributions.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots()
# plt.boxplot([iar, sar], showfliers=True)

sns.boxplot(y="score", data=prompted_non_prompted_iar_sar_df, hue="agent", x="type", showmeans=True, showfliers=False)

plt.yticks(fontsize=16)
plt.xticks(fontsize=16)

ax.set_xticklabels(labels=["$\\textrm{ITRR}$", "$\\textrm{STRR}$"])

plt.ylabel("")#$\\textrm{Score}$")
plt.xlabel("")

plt.grid(linestyle='--')

plt.tight_layout()
plt.savefig(os.path.join(figs_dir, "prompted_non-prompted_iar_sar_distributions.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
np.median(prompted_iar), np.median(non_prompted_iar)

In [ ]:
np.median(prompted_sar), np.median(non_prompted_sar)

In [ ]:
iar = np.array(iar)

In [ ]:
sar = np.array(sar)

In [ ]:
np.median(iar), np.median(sar) 

In [ ]:
np.mean(iar), np.mean(sar) 

In [ ]:
res, p_value = stats.spearmanr(iar, sar)

In [ ]:
res, p_value = stats.pearsonr(iar, sar)

In [ ]:
prompted_res, prompted_p_value = stats.spearmanr(prompted_iar, prompted_sar)
prompted_res, prompted_p_value 

In [ ]:
non_prompted_res, non_prompted_p_value = stats.spearmanr(non_prompted_iar, non_prompted_sar)
non_prompted_res, non_prompted_p_value 

In [ ]:
plt.figure(figsize=(5, 4))
plt.text(0.53, 0.85,
         rf"$\rho = {res:.4f}$",
         transform=plt.gca().transAxes,
         fontsize=20)
sns.regplot(x=iar, y=sar, line_kws={"color": "red"})

plt.xlim(-0.05, 1.05)
plt.ylim(-0.05, 1.05)

plt.xlabel("IAR")
plt.ylabel("SAR")

plt.grid(linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(figs_dir, "iar_sar_correlation.pdf"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
"""
Validation of ITRR / STRR, addressing the reviewer's comment.

Definitions as published (Eq. 2-3), adapted from Luceri et al. (2025):
    ITRR = a / nT          (= IAR  = |E cap A| / |E|)
    STRR = b / (a + b)     (= SAR  = 1 - |E cap A| / |A|)

where
    a  = unique toxic responses to toxic stimuli       (RT-ST)
    b  = unique toxic responses to non-toxic stimuli   (RT-SN)
    nT = unique toxic stimuli
    nN = unique non-toxic stimuli

Inclusion, matching the published loop: an agent is retained iff it received
at least one toxic stimulus AND produced at least one toxic response.

The shared count is `a`: it raises ITRR and lowers STRR. The extreme case is
an agent with a single toxic response, for which the two metrics are perfectly
anti-correlated by construction.
"""

import numpy as np, pandas as pd
from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm

rows = [(k,
         unique_count_chirper_toxic_adoption_toxic_content[k],
         unique_count_chirper_toxic_adoption_non_toxic_content[k],
         count_chirper_exposition_toxic_content[k],
         count_chirper_exposition_non_toxic_content[k])
        for k in exposures_per_follower]
df = pd.DataFrame(rows, columns=["agent", "a", "b", "nT", "nN"])
df["prompted"] = df.agent.isin(agents_prompted_to_be_toxic)
df["ntox_resp"] = df.a + df.b

# ---------------------------------------------------------------------------
# 1. INCLUSION ACCOUNTING (requested explicitly)
# ---------------------------------------------------------------------------
keep = ~((df.a == 0) & (df.nT == 0)) & ~((df.b == 0) & (df.ntox_resp == 0))
print(f"chirpers considered:                      {len(df):,}")
print(f"  excluded: no toxic stimulus (nT == 0):  {(df.nT == 0).sum():,}")
print(f"  excluded: no toxic response (a+b == 0): {(df.ntox_resp == 0).sum():,}")
print(f"  retained:                               {keep.sum():,}")
print(f"  a > nT (deduplication incomplete):      {(df.a > df.nT).sum():,}")

d = df[keep].copy()
d["ITRR"] = d.a / d.nT
d["STRR"] = d.b / d.ntox_resp
print(f"\nITRR mean {d.ITRR.mean():.3f} median {d.ITRR.median():.3f} "
      f"| at 0: {100*(d.ITRR == 0).mean():.1f}%  at 1: {100*(d.ITRR == 1).mean():.1f}%")
print(f"STRR mean {d.STRR.mean():.3f} median {d.STRR.median():.3f} "
      f"| at 0: {100*(d.STRR == 0).mean():.1f}%  at 1: {100*(d.STRR == 1).mean():.1f}%")

obs_p, obs_s = pearsonr(d.ITRR, d.STRR)[0], spearmanr(d.ITRR, d.STRR)[0]
print(f"\nobserved  pearson {obs_p:+.4f}   spearman {obs_s:+.4f}   n = {len(d):,}")

# ---------------------------------------------------------------------------
# 2. HOW MUCH IS DRIVEN BY AGENTS WITH FEW TOXIC RESPONSES?
#    For an agent with exactly one toxic response, STRR is 0 or 1 and the two
#    metrics are perfectly anti-correlated by construction. If rho attenuates
#    sharply as the minimum count rises, the coefficient is dominated by these.
# ---------------------------------------------------------------------------
print("\n--- correlation by minimum number of toxic responses ---")
print(f"{'min a+b':>8} {'agents':>9} {'pearson':>9} {'spearman':>9}")
for k in [1, 2, 3, 5, 10, 20]:
    s = d[d.ntox_resp >= k]
    if len(s) > 30:
        print(f"{k:>8} {len(s):>9,} {pearsonr(s.ITRR, s.STRR)[0]:>+9.3f}"
              f" {spearmanr(s.ITRR, s.STRR)[0]:>+9.3f}")
print("\ndistribution of toxic responses per retained agent:")
print(d.ntox_resp.describe(percentiles=[.25, .5, .75, .9]).round(2).to_string())

# ---------------------------------------------------------------------------
# 3. PERMUTATION NULL
#    Preserves per agent: nT, nN (activity and stimulus categories) and a+b
#    (overall toxicity rate). Randomises only WHICH stimuli elicited the toxic
#    responses -- exactly the null the reviewer specifies.
# ---------------------------------------------------------------------------
def perm_null(s, R=1000, seed=0):
    rng = np.random.default_rng(seed)
    nT, nN = s.nT.to_numpy(), s.nN.to_numpy()
    tot = np.minimum(s.ntox_resp.to_numpy(), nT + nN)
    out = np.empty((R, 2))
    for r in range(R):
        a0 = rng.hypergeometric(np.maximum(nT, 0), np.maximum(nN, 0), tot)
        b0 = tot - a0
        m = (nT > 0) & (tot > 0)
        x, y = a0[m] / nT[m], b0[m] / tot[m]
        out[r] = [pearsonr(x, y)[0], spearmanr(x, y)[0]]
    return out

nul = perm_null(d)
print("\n--- permutation null (activity, categories, toxicity rate preserved) ---")
for j, (nm, obs) in enumerate([("pearson", obs_p), ("spearman", obs_s)]):
    lo, md, hi = np.percentile(nul[:, j], [2.5, 50, 97.5])
    z = (obs - nul[:, j].mean()) / nul[:, j].std()
    print(f"  {nm:9s} observed {obs:+.4f} | null median {md:+.4f} "
          f"[{lo:+.4f}, {hi:+.4f}] | z = {z:+.1f}")

# Same test restricted to agents with enough toxic responses that the corner
# structure cannot dominate.
for k in [5, 10]:
    s = d[d.ntox_resp >= k]
    if len(s) > 100:
        n2 = perm_null(s)
        o = spearmanr(s.ITRR, s.STRR)[0]
        lo, md, hi = np.percentile(n2[:, 1], [2.5, 50, 97.5])
        print(f"  a+b >= {k:<2} (n={len(s):,})  spearman observed {o:+.4f} | "
              f"null {md:+.4f} [{lo:+.4f}, {hi:+.4f}]")

# ---------------------------------------------------------------------------
# 4. STABILITY AFTER CONTROLS
# ---------------------------------------------------------------------------
d["log_activity"] = np.log1p(d.nT + d.nN)
d["tox_share"] = d.nT / (d.nT + d.nN)
d["log_ntox_resp"] = np.log1p(d.ntox_resp)
d["prompted_f"] = d.prompted.astype(float)

CTRL = ["log_activity", "tox_share", "log_ntox_resp", "prompted_f"]
X = sm.add_constant(d[["ITRR"] + CTRL])
print("\n--- STRR on ITRR, controlling for activity, composition, count, prompt ---")
print(sm.OLS(d.STRR, X).fit(cov_type="HC3").summary().tables[1])

Z = sm.add_constant(d[CTRL])
rx = sm.OLS(d.ITRR, Z).fit().resid
ry = sm.OLS(d.STRR, Z).fit().resid
print(f"partial correlation given controls: {pearsonr(rx, ry)[0]:+.4f}")

for g, name in [(d[d.prompted], "prompted"), (d[~d.prompted], "non-prompted")]:
    if len(g) > 30:
        print(f"{name:13s} n={len(g):>7,} spearman {spearmanr(g.ITRR, g.STRR)[0]:+.4f}")

# ---------------------------------------------------------------------------
# 5. SENSITIVITY: disjoint variant (no shared count)
#    STRR' = b / nN, the counterpart rate to ITRR = a / nT.
# ---------------------------------------------------------------------------
e = df[(df.nT > 0) & (df.nN > 0)].copy()
e["ITRR"], e["STRRp"] = e.a / e.nT, e.b / e.nN
print(f"\ndisjoint variant (n={len(e):,}): "
      f"pearson {pearsonr(e.ITRR, e.STRRp)[0]:+.4f}  "
      f"spearman {spearmanr(e.ITRR, e.STRRp)[0]:+.4f}")

In [ ]:
for m in [1, 2, 3, 5, 10, 20]:
    s = d[d.ntox_resp >= m]
    n = perm_null(s, R=1000)
    lo, md, hi = np.percentile(n[:, 1], [2.5, 50, 97.5])
    print(f"{m:>3} {len(s):>7,} {spearmanr(s.ITRR, s.STRR)[0]:+.3f} "
          f"{md:+.3f} [{lo:+.3f}, {hi:+.3f}]")